# 🔥 Tambor rotativo de pirólise em batelada: combustão, troca térmica, cinética e carvão (Colab)

Uma batelada inteira de **lixo marinho misto** (o resíduo pós-consumo que a UFRJ/IVIG processa, recolhido na Baía de Guanabara) num **tambor rotativo aquecido por fora**, do tamanho do equipamento do vídeo (classe Beston BLJ-3, Ø 1,4 m × 4,9 m, aletas internas). Tudo em Python, com **GPU** no Colab, e com **validação** contra a literatura aberta.

| Etapa | O que faz |
|---|---|
| 1 | **Cinética** de 9 componentes do lixo (PE, PP, PS, PET, PVC em duas etapas, borracha, celulósicos, água, areia) e TGA |
| 2 | **Batelada de 6–8 h**: fornalha (zonas de Hottel, WSGG), casco girando, contato parede-carga (penetração), radiação interna, secagem, fusão, pirólise, carvão, craqueamento no espaço livre, condensador e **gás próprio voltando ao queimador** — **girando × parado**, com e sem aletas, várias rotações |
| 3 | **DEM com mudança de fase** (GPU): cada pedaço de lixo gira com o tambor e as aletas, amolece, gruda, funde numa poça viscosa, craqueia e vira carvão, no relógio da batelada |
| 4 | **LES 3D da fornalha**: chamas dos queimadores (renderizadas pela emissão da fuligem), gases quentes em volta do tambor e fluxo de calor no casco |
| 5 | **Validação** (Suzuki, Faussone, Fulgencio-Medrano, Beston, UFRJ) com veredito por item |
| 6 | **Vídeo em HD**: o time-lapse da batelada com o **gráfico se atualizando ao vivo**, a **taxa de pirólise** de cada componente junto com a mudança de fase, parado × girando, DEM e chamas |

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Sem GPU use o preset `teste`.

Hipóteses, equações e referências: `docs/brief_tambor_batelada.md`, `docs/brief_mudanca_de_fase.md` e `docs/pesquisa/` no repositório.

In [ ]:
#@title Ambiente: GPU, Cantera e pacote `tamborkit`
import os, sys, subprocess, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
print("GPU:", gpu or "nenhuma (use o preset 'teste')")
try:
    import cantera
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cantera"], check=False)
os.makedirs("tamborkit", exist_ok=True)
os.makedirs("saida", exist_ok=True)

In [ ]:
%%writefile tamborkit/__init__.py

In [ ]:
%%writefile tamborkit/geometry.py
"""Geometria do tambor rotativo em batelada: casco cilíndrico horizontal, aletas internas e fornalha.

Convenções (iguais em todo o pacote):
  * eixo do tambor em x, de 0 (porta de carga, na frente) a L (fundo, saída dos vapores);
  * seção transversal no plano (y, z), z para cima, centro do tambor em y = z = 0;
  * rotação positiva em torno de +x: θ(t) = Ω t. O ponto mais baixo do casco anda para +y, então a carga
    sobe pelo lado y > 0 e rola de volta pela superfície do leito;
  * ângulos na seção: φ = atan2(z, y).

As aletas são placas radiais soldadas no casco (retas ou helicoidais), opcionalmente com uma aba na ponta
voltada para o sentido de giro (aleta em L), que forma a "concha" que levanta a carga.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np
import torch


@dataclass
class Drum:
    R: float = 1.4              # raio interno do casco [m]
    L: float = 6.6              # comprimento interno [m]
    wall: float = 0.016         # espessura do casco [m]
    rpm: float = 0.8            # rotação [1/min]
    n_lifters: int = 4          # número de aletas
    lifter_h: float = 0.18      # altura radial da aleta [m]
    lifter_t: float = 0.016     # espessura da chapa [m]
    lifter_tip: float = 0.0     # largura da aba na ponta (aleta em L) [m]; 0 = aleta reta
    lifter_pitch: float = 0.0   # passo da hélice [m]; 0 = aletas retas, paralelas ao eixo
    lifter_x: tuple = (0.15, 0.15)   # folga das aletas até a porta e até o fundo [m]

    @property
    def omega(self) -> float:
        return 2 * math.pi * self.rpm / 60.0

    @property
    def D(self) -> float:
        return 2 * self.R

    def froude(self, g: float = 9.81) -> float:
        return self.omega ** 2 * self.R / g

    def volume(self) -> float:
        return math.pi * self.R ** 2 * self.L

    def lifter_angles(self) -> np.ndarray:
        return 2 * math.pi * np.arange(self.n_lifters) / max(self.n_lifters, 1)

    def lifter_boxes(self):
        """Caixas de cada aleta no referencial da própria aleta (u radial para fora, w no sentido de giro):
        lista de (u0, u1, w0, w1). A placa entra um pouco no casco para não deixar fresta."""
        h, t = self.lifter_h, self.lifter_t
        boxes = [(self.R - h, self.R + 0.05, -0.5 * t, 0.5 * t)]
        if self.lifter_tip > 0:
            boxes.append((self.R - h, self.R - h + t, -0.5 * t, self.lifter_tip))
        return boxes


def fill_half_angle(fill: float) -> float:
    """Meio ângulo ε do segmento circular ocupado pela carga: fill = (ε − sen ε cos ε)/π."""
    lo, hi = 0.0, math.pi
    for _ in range(80):
        e = 0.5 * (lo + hi)
        if (e - math.sin(e) * math.cos(e)) / math.pi < fill:
            lo = e
        else:
            hi = e
    return 0.5 * (lo + hi)


def bed_depth(fill: float, R: float) -> float:
    return R * (1 - math.cos(fill_half_angle(fill)))


def lifter_contacts(drum: Drum, x: torch.Tensor, theta: float, L_period: float | None = None):
    """Contato das partículas (centros x [N,3]) com todas as caixas das aletas.

    Devolve (dist [N,B], n [N,B,3], p_c [N,B,3]): distância do centro à superfície, normal unitária da
    partícula para a aleta e ponto de contato na superfície, para cada uma das B caixas."""
    ang = torch.as_tensor(drum.lifter_angles(), dtype=x.dtype, device=x.device)
    y, z = x[:, 1:2], x[:, 2:3]
    phi = ang[None, :] + theta
    if drum.lifter_pitch > 0:
        phi = phi + 2 * math.pi * x[:, 0:1] / drum.lifter_pitch
    c, s = torch.cos(phi), torch.sin(phi)
    u = y * c + z * s            # coordenada radial no referencial da aleta
    w = -y * s + z * c           # coordenada no sentido de giro
    xs = x[:, 0:1].expand_as(u)
    x0, x1 = drum.lifter_x[0], drum.L - drum.lifter_x[1]
    out_d, out_n, out_p = [], [], []
    for (u0, u1, w0, w1) in drum.lifter_boxes():
        qu, qw = u.clamp(u0, u1), w.clamp(w0, w1)
        qx = xs.clamp(x0, x1) if L_period is None else xs
        du, dw, dx = qu - u, qw - w, qx - xs
        d = torch.sqrt(du * du + dw * dw + dx * dx).clamp(min=1e-9)
        # volta ao referencial do laboratório: e_u = (0, c, s), e_w = (0, −s, c)
        ny, nz = du * c - dw * s, du * s + dw * c
        n = torch.stack([dx, ny, nz], -1) / d[..., None]
        py, pz = qu * c - qw * s, qu * s + qw * c
        p = torch.stack([qx, py, pz], -1)
        out_d.append(d); out_n.append(n); out_p.append(p)
    return torch.cat(out_d, 1), torch.cat(out_n, 1), torch.cat(out_p, 1)

In [ ]:
%%writefile tamborkit/feed.py
"""Carga de resíduo pós-consumo misto (lixo marinho) e cinética de pirólise por pseudocomponentes.

Cada pseudocomponente da carga reage por uma lei cinética da literatura aberta e se divide em grupos de
produtos ("lumps"):

  G_HC  gás não condensável de hidrocarbonetos (CH4, C2–C4, H2)        PCI ~46 MJ/kg
  G_OX  gás oxigenado (CO, CO2, um pouco de CH4 e H2)                  PCI ~8 MJ/kg
  O     óleo leve (C5–C20)          W  cera (C21+)          A  aromáticos (estireno, benzeno)
  OX    oxigenados e sublimados do PET (ácido benzoico/tereftálico)
  TAR   alcatrão primário da celulose (reage no espaço livre: vira gás ou coque)
  H2O   água (umidade livre e água de pirólise)      HCl  cloreto de hidrogênio do PVC
  CHAR  carvão (fica no leito)                       ASH  cinzas, areia e sal (inerte)

Leis cinéticas (k = A exp(−E/RT)):
  rs2    cisão aleatória L = 2: dx/dt = k(1 − x), α = x² (poliolefinas e PS; Aboulkas 2010, A ajustado
         aos picos de DTG de consenso a 10 K/min, como no reator agitado)
  first  primeira ordem: dα/dt = k(1 − α) (PET, frações da borracha)
  p2     nucleação P2 com semente: s = √α, ds/dt = k (PVC, etapa I de desidrocloração)
  f3     terceira ordem: dα/dt = k(1 − α)³ (PVC, etapa II, resíduo poliênico)
  sc     Shafizadeh–Chin: três reações paralelas de 1ª ordem para gás, alcatrão e carvão (celulósicos)
Umidade: platô de ebulição a 100 °C (o calor que chega acima disso evapora a água).

Rendimentos por kg de componente reagido para um tambor em batelada a ~500 °C (síntese da pesquisa,
docs/brief_tambor_batelada.md): composições e números com a fonte ao lado. [H] = hipótese de projeto.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

R_GAS = 8.314462618
T0 = 298.15

LUMPS = ["G_HC", "G_OX", "O", "W", "A", "OX", "TAR", "H2O", "HCl"]       # saem do leito como vapor
SOLIDS = ["CHAR", "ASH"]
CONDENSABLE = ["O", "W", "A", "OX", "TAR"]
LHV = {"G_HC": 46.3e6, "G_OX": 8.0e6}          # caso A da pesquisa (46,3 MJ/kg) e mistura CO/CO2 [H]
AFR = {"G_HC": 15.43, "G_OX": 2.2}             # ar estequiométrico [kg/kg]


@dataclass(frozen=True)
class Component:
    key: str
    name: str
    law: str                       # rs2 | first | p2 | f3 | sc | water | inert
    A: float = 0.0                 # [1/s]
    E: float = 0.0                 # [J/mol]
    yields: dict = field(default_factory=dict)   # kg de produto por kg reagido (lumps, CHAR, ou componente)
    dH: float = 0.0                # calor de reação + vaporização dos produtos [J/kg reagido] (> 0 endotérmico)
    cp: float = 1800.0             # calor específico do sólido [J/(kg K)]
    cp_l: float = 0.0              # calor específico do fundido (0 = igual ao sólido)
    Tm: float = 0.0                # fusão [K] (0 = não funde)
    hm: float = 0.0                # calor latente de fusão [J/kg]
    rho_loose: float = 100.0       # massa específica aparente como carregado (peso relativo) [kg/m³]
    rho_dense: float = 900.0       # massa específica aparente depois de fundir/compactar [kg/m³]
    k_dense: float = 0.2           # condutividade do material denso [W/(m K)]
    sc: tuple = ()                 # Shafizadeh–Chin: ((A, E) gás, (A, E) alcatrão, (A, E) carvão)
    seed: float = 0.0              # semente da lei p2 (α inicial)
    color: tuple = (0.6, 0.6, 0.6)
    group: str = ""                # rótulo para gráficos
    source: str = ""

    def k(self, T):
        return self.A * np.exp(-self.E / (R_GAS * np.asarray(T, float)))


def _sc_rates(c: Component, T):
    return [a * np.exp(-e / (R_GAS * np.asarray(T, float))) for a, e in c.sc]


# ----------------------------------------------------------------- componentes
PO_SRC = "E: Aboulkas 2010; A ajustado ao pico de DTG de consenso a 10 K/min; rendimentos: pesquisa R2 [H]"
COMPONENTS = {
    "H2O": Component("H2O", "umidade", "water", yields={"H2O": 1.0}, dH=2.257e6, cp=4180.0,
                     rho_loose=1e9, rho_dense=1e9, k_dense=0.6, color=(0.35, 0.55, 0.85), group="água",
                     source="h_fg a 100 °C"),
    "LDPE": Component("LDPE", "PEBD (filmes, sacolas)", "rs2", 1.92e13, 215e3,
                      {"G_HC": 0.10, "O": 0.55, "W": 0.34, "CHAR": 0.01}, 0.92e6, 1900.0, 2600.0,
                      383.15, 130e3, 45.0, 740.0, 0.22, color=(0.92, 0.92, 0.88), group="poliolefinas",
                      source=PO_SRC),
    "HDPE": Component("HDPE", "PEAD (frascos, tampas)", "rs2", 5.2e14, 238e3,
                      {"G_HC": 0.10, "O": 0.55, "W": 0.34, "CHAR": 0.01}, 0.92e6, 1900.0, 2600.0,
                      403.15, 200e3, 70.0, 740.0, 0.25, color=(0.30, 0.55, 0.85), group="poliolefinas",
                      source=PO_SRC),
    "PP": Component("PP", "PP (tampas, redes, embalagens)", "rs2", 6.98e10, 179e3,
                    {"G_HC": 0.12, "O": 0.60, "W": 0.26, "CHAR": 0.02}, 1.0e6, 1900.0, 2600.0,
                    438.15, 90e3, 90.0, 730.0, 0.20, color=(0.95, 0.80, 0.25), group="poliolefinas",
                    source=PO_SRC + "; ΔH ~1,0 MJ/kg (faixa 0,6–1,3)"),
    "PS": Component("PS", "PS/EPS (isopor, copos)", "rs2", 4.1e11, 180e3,
                    {"A": 0.65, "O": 0.25, "G_HC": 0.05, "CHAR": 0.05}, 0.8e6, 1500.0, 2000.0,
                    373.15, 0.0, 25.0, 950.0, 0.15, color=(0.97, 0.97, 0.97), group="PS",
                    source="E meio da faixa 150–204 kJ/mol; estireno ~0,65 [S/H]"),
    "PET": Component("PET", "PET (garrafas, poliéster)", "first", 2.59e13, 210e3,
                     {"CHAR": 0.15, "OX": 0.40, "G_OX": 0.43, "H2O": 0.02}, 0.4e6, 1300.0, 1900.0,
                     528.15, 50e3, 40.0, 1150.0, 0.20, color=(0.55, 0.85, 0.75), group="PET",
                     source="Dubdub & Al-Yaari 2023; rendimentos [H] (PET forma sublimados que sujam o condensador)"),
    "PVC": Component("PVC", "PVC (tubos, cabos) – etapa I", "p2", 3.0e4, 75e3,
                     {"HCl": 0.583, "PVCr": 0.417}, 0.4e6, 1000.0, 1500.0, 0.0, 0.0, 300.0, 1300.0, 0.17,
                     seed=1e-4, color=(0.85, 0.45, 0.35), group="PVC",
                     source="desidrocloração P2 (75 kJ/mol); HCl 0,583 kg/kg; ΔH [H, não verificado]"),
    "PVCr": Component("PVCr", "resíduo poliênico do PVC – etapa II", "f3", 8.07e7, 140e3,
                      {"A": 0.55, "G_HC": 0.15, "CHAR": 0.25, "O": 0.05}, 0.4e6, 1200.0, 0.0, 0.0, 0.0,
                      400.0, 1200.0, 0.17, color=(0.45, 0.25, 0.20), group="PVC",
                      source="F3 (140 kJ/mol); produtos [H] limitados pelo carbono fixo 5–6 % do PVC"),
    "CELL": Component("CELL", "celulósicos (papel, madeira, coco, algas)", "sc",
                      yields={}, dH=0.3e6, cp=1500.0, rho_loose=150.0, rho_dense=450.0, k_dense=0.12,
                      sc=((1.44e4, 88.6e3), (4.12e6, 112.7e3), (7.38e5, 106.5e3)),
                      color=(0.70, 0.52, 0.30), group="celulósicos",
                      source="Shafizadeh–Chin (carvalho), via Navarrete 2024; ΔH [H] (faixa −0,3 a +0,5 MJ/kg)"),
    "RUBa": Component("RUBa", "borracha – fração leve", "first", 3.8e2 / 60, 33.1e3,
                      {"O": 0.667, "A": 0.083, "G_HC": 0.25}, 0.5e6, 1900.0, 0.0, 0.0, 0.0, 400.0, 1100.0, 0.20,
                      color=(0.15, 0.15, 0.15), group="borracha", source="Cherbański 2017 (3 componentes)"),
    "RUBb": Component("RUBb", "borracha – óleos e aditivos", "first", 3.1e9 / 60, 124.1e3,
                      {"O": 0.667, "A": 0.083, "G_HC": 0.25}, 0.5e6, 1900.0, 0.0, 0.0, 0.0, 400.0, 1100.0, 0.20,
                      color=(0.15, 0.15, 0.15), group="borracha", source="Cherbański 2017"),
    "RUBc": Component("RUBc", "borracha – elastômero", "first", 1.6e11 / 60, 161.6e3,
                      {"O": 0.667, "A": 0.083, "G_HC": 0.25}, 0.5e6, 1900.0, 0.0, 0.0, 0.0, 400.0, 1100.0, 0.20,
                      color=(0.15, 0.15, 0.15), group="borracha", source="Cherbański 2017"),
    "CB": Component("CB", "negro de fumo da borracha", "inert", cp=1100.0, rho_loose=400.0, rho_dense=500.0,
                    k_dense=0.15, color=(0.08, 0.08, 0.08), group="borracha", source="carvão da borracha 38–40 %"),
    "INERT": Component("INERT", "areia, conchas e sal", "inert", cp=850.0, rho_loose=1300.0, rho_dense=1500.0,
                       k_dense=0.35, color=(0.80, 0.72, 0.55), group="inertes",
                       source="agregado fino 10,6 % (rejeitos do Rio, Cruz 2024)"),
    "CHAR": Component("CHAR", "carvão formado", "inert", cp=1100.0, rho_loose=300.0, rho_dense=350.0,
                      k_dense=0.12, color=(0.10, 0.09, 0.08), group="carvão", source="—"),
}
# celulósicos: o "alcatrão" primário sai como TAR; o "gás" como G_OX + água [H, calibrado para dar, a 500 °C,
# carvão ~0,29, orgânico ~0,25, água ~0,22, gás ~0,24 kg/kg (pesquisa R2, faixas de Navarrete e Fulgencio)]
SC_PRODUCTS = ({"G_OX": 0.75, "H2O": 0.25}, {"TAR": 0.62, "H2O": 0.38}, {"CHAR": 1.0})

ORDER = list(COMPONENTS)
IDX = {k: i for i, k in enumerate(ORDER)}
REACTIVE = [k for k in ORDER if COMPONENTS[k].law not in ("inert", "water")]


@dataclass
class Charge:
    """Carga do tambor: massa úmida, umidade e composição seca (frações mássicas, normalizadas)."""
    name: str = "lixo marinho misto (caso base)"
    mass: float = 600.0             # kg úmidos por batelada
    moisture: float = 0.15          # fração mássica de água livre (base úmida)
    dry: dict = field(default_factory=lambda: {
        "LDPE": 25, "HDPE": 15, "PP": 20, "PS": 12, "PET": 6, "PVC": 2, "RUB": 3, "CELL": 8, "INERT": 9})
    fill: float = 0.62              # fração do tambor ocupada pela carga solta (≤ 2/3)
    source: str = ("composição seca de projeto [H] (pesquisa R2: PE 40, PP 20, PS 12, PET 6, PVC 2, borracha 3, "
                   "celulósicos 8, inertes 9 %); 0,5–0,8 t por batelada, enchimento ≤ 2/3")

    def masses(self) -> dict:
        d = {k: float(v) for k, v in self.dry.items()}
        tot = sum(d.values())
        dry_mass = self.mass * (1 - self.moisture)
        m = {k: 0.0 for k in ORDER}
        m["H2O"] = self.mass * self.moisture
        for k, v in d.items():
            mk = dry_mass * v / tot
            if k == "RUB":       # Cherbański: 60 % voláteis em 3 frações (z = 0,053/0,294/0,653), 40 % negro de fumo
                for kk, z in (("RUBa", 0.053), ("RUBb", 0.294), ("RUBc", 0.653)):
                    m[kk] += 0.60 * mk * z
                m["CB"] += 0.40 * mk
            else:
                m[k] += mk
        return m

    def describe(self) -> str:
        d = self.dry
        tot = sum(d.values())
        return (f"{self.mass:.0f} kg úmidos ({100 * self.moisture:.0f} % de água); seco: "
                + ", ".join(f"{k} {100 * v / tot:.0f}%" for k, v in d.items()))


CHARGES = {
    "base": Charge(),
    "plastico_triado": Charge("plástico triado (poliolefinas)", 600.0, 0.05,
                              {"LDPE": 45, "HDPE": 20, "PP": 30, "PS": 3, "INERT": 2}),
    "rejeito_rio": Charge("rejeito de cooperativa do Rio (Cruz 2024)", 600.0, 0.15,
                          {"LDPE": 9, "HDPE": 2, "PP": 12, "PS": 30, "PET": 13, "PVC": 4, "CELL": 14, "RUB": 2,
                           "INERT": 14}),
}


# ---------------------------------------------------------------- propriedades
def melt_fraction(c: Component, T):
    """Fração fundida (rampa suave de ±8 K em torno de Tm)."""
    if c.Tm <= 0:
        return np.zeros_like(np.asarray(T, float))
    return np.clip((np.asarray(T, float) - (c.Tm - 8.0)) / 16.0, 0.0, 1.0)


def apparent_cp(c: Component, T):
    """c_p aparente: sensível (sólido → fundido) + calor latente espalhado na rampa de fusão."""
    T = np.asarray(T, float)
    if c.Tm <= 0:
        return np.full_like(T, c.cp)
    f = melt_fraction(c, T)
    cpl = c.cp_l or c.cp
    lat = np.where((T > c.Tm - 8.0) & (T < c.Tm + 8.0), c.hm / 16.0, 0.0)
    return (1 - f) * c.cp + f * cpl + lat


def sensible_enthalpy(c: Component, T):
    """h(T) − h(T0) por integração analítica da c_p aparente [J/kg]."""
    T = np.asarray(T, float)
    if c.Tm <= 0:
        return c.cp * (T - T0)
    cpl = c.cp_l or c.cp
    a, b = c.Tm - 8.0, c.Tm + 8.0
    Ts = np.minimum(T, a)
    h = c.cp * (Ts - T0)
    Tm_ = np.clip(T, a, b)
    # na rampa: c_p varia linearmente de cp a cpl, mais o latente uniforme
    x = (Tm_ - a)
    h = h + c.cp * x + 0.5 * (cpl - c.cp) * x ** 2 / 16.0 + c.hm * x / 16.0
    h = h + cpl * np.maximum(T - b, 0.0)
    return h


# ------------------------------------------------------------------- cinética
class KineticState:
    """Estado químico de N volumes de leito (fatias ou camadas): massas iniciais e progresso por componente.

    m0[c]: massa de referência (inicial, ou acumulada para os intermediários PVCr e CHAR);
    p[c]:  variável de progresso da lei (x para rs2, √α para p2, α para first/f3/sc).
    A massa que resta é m = m0 (1 − α(p)); intermediários recebem massa por "m0 += Δ" com α reescalado.
    """

    def __init__(self, masses: dict, n: int = 1):
        self.n = n
        self.m0 = {k: np.full(n, float(masses.get(k, 0.0))) for k in ORDER}
        self.p = {k: np.zeros(n) for k in ORDER}
        for k in ORDER:
            if COMPONENTS[k].law == "p2":
                self.p[k][:] = math.sqrt(COMPONENTS[k].seed)

    def alpha(self, k):
        c, p = COMPONENTS[k], self.p[k]
        if c.law == "rs2":
            return p ** 2
        if c.law == "p2":
            return np.minimum(p ** 2, 1.0)
        if c.law in ("first", "f3", "sc", "water"):
            return p
        return np.zeros_like(p)

    def mass(self, k):
        return self.m0[k] * (1.0 - self.alpha(k))

    def masses(self) -> dict:
        return {k: self.mass(k) for k in ORDER}

    def total(self):
        return sum(self.mass(k) for k in ORDER)

    def add(self, k, dm):
        """Acrescenta massa a um componente intermediário/inerte mantendo a massa já convertida."""
        m_now = self.mass(k)
        new = m_now + dm
        if COMPONENTS[k].law == "inert":
            self.m0[k] = new
            return
        # reescala para α' com m0' = m0 + dm: m0' (1 − α') = new
        m0n = self.m0[k] + dm
        a = np.where(m0n > 0, 1.0 - new / np.maximum(m0n, 1e-30), 0.0)
        self.m0[k] = m0n
        c = COMPONENTS[k]
        if c.law == "f3" or c.law == "first":
            self.p[k] = np.clip(a, 0.0, 1.0)
        elif c.law == "rs2":
            self.p[k] = np.sqrt(np.clip(a, 0.0, 1.0))

    def react(self, T, dt):
        """Avança as reações por dt na temperatura T (array n) com integração exata por lei.

        Devolve (produtos {lump: kg}, calor de reação [J], taxas por componente {k: kg/s})."""
        T = np.asarray(T, float)
        prod = {k: np.zeros(self.n) for k in LUMPS}
        heat = np.zeros(self.n)
        rates = {}
        new_mass = {}
        for k in REACTIVE:
            c = COMPONENTS[k]
            m_before = self.mass(k)
            if not np.any(self.m0[k] > 0):
                continue
            p = self.p[k]
            if c.law == "rs2":
                self.p[k] = 1.0 - (1.0 - p) * np.exp(-c.k(T) * dt)
            elif c.law == "first":
                self.p[k] = 1.0 - (1.0 - p) * np.exp(-c.k(T) * dt)
            elif c.law == "p2":
                self.p[k] = np.minimum(p + c.k(T) * dt, 1.0)
            elif c.law == "f3":
                q = np.maximum(1.0 - p, 1e-12)
                self.p[k] = 1.0 - 1.0 / np.sqrt(q ** -2 + 2 * c.k(T) * dt)
            elif c.law == "sc":
                ks = _sc_rates(c, T)
                kt = ks[0] + ks[1] + ks[2]
                self.p[k] = 1.0 - (1.0 - p) * np.exp(-kt * dt)
            dm = np.maximum(m_before - self.mass(k), 0.0)
            rates[k] = dm / dt
            heat += dm * c.dH
            if c.law == "sc":
                for kk, frac in zip(ks, SC_PRODUCTS):
                    w = kk / np.maximum(kt, 1e-300)
                    for lump, y in frac.items():
                        new_mass[lump] = new_mass.get(lump, 0.0) + dm * w * y
            else:
                for lump, y in c.yields.items():
                    new_mass[lump] = new_mass.get(lump, 0.0) + dm * y
        for lump, dm in new_mass.items():
            if lump in prod:
                prod[lump] += dm
            else:   # intermediário ou sólido que fica no leito
                self.add(lump, dm)
        return prod, heat, rates

    # ----------------------------------------------------- propriedades do leito
    def heat_capacity(self, T):
        """Σ m c_p,aparente [J/K]."""
        return sum(self.mass(k) * apparent_cp(COMPONENTS[k], T) for k in ORDER)

    def enthalpy(self, T):
        return sum(self.mass(k) * sensible_enthalpy(COMPONENTS[k], T) for k in ORDER)

    def melt_state(self, T):
        """Fração em volume denso do leito que está fundida (0 = pedaços soltos, 1 = poça de fundido)."""
        vm, vt = 0.0, 0.0
        for k in ORDER:
            c = COMPONENTS[k]
            if k == "H2O":
                continue
            v = self.mass(k) / c.rho_dense
            vt = vt + v
            if c.Tm > 0:
                vm = vm + v * melt_fraction(c, T)
        return np.where(vt > 0, vm / np.maximum(vt, 1e-30), 0.0)

    def volume(self, T, loose_scale: float = 1.0):
        """Volume aparente do leito [m³]: pedaços soltos até fundirem; depois poça densa.

        Componentes que não fundem ficam soltos enquanto o leito não tiver fundido (melt_state)."""
        ms = self.melt_state(T)
        V = 0.0
        for k in ORDER:
            c = COMPONENTS[k]
            if k == "H2O":
                continue
            m = self.mass(k)
            if c.Tm > 0:
                f = melt_fraction(c, T)
                V = V + m * ((1 - f) / (c.rho_loose * loose_scale) + f / c.rho_dense)
            else:
                V = V + m * ((1 - ms) / (c.rho_loose * loose_scale) + ms / c.rho_dense)
        return V

    def state_fractions(self, T):
        """Frações (em volume denso) do leito: pedaços soltos, fundido e resíduo granular (carvão/cinzas)."""
        T = np.asarray(T, float)
        loose = melt = res = 0.0
        for k in ORDER:
            c = COMPONENTS[k]
            if k == "H2O":
                continue
            v = self.mass(k) / c.rho_dense
            if k in ("CHAR", "CB", "INERT", "PVCr"):
                res = res + v
            elif c.Tm > 0:
                f = melt_fraction(c, T)
                melt = melt + v * f
                loose = loose + v * (1 - f)
            else:
                loose = loose + v
        tot = np.maximum(loose + melt + res, 1e-30)
        return loose / tot, melt / tot, res / tot

    def conductivity(self, T, d_piece: float = 0.08, eps: float = 0.85):
        """Condutividade efetiva do leito [W/(m K)]: média em volume do material denso, ponderada pela
        compacidade, mais o termo de radiação nos poros entre os pedaços (Zehner–Schlünder simplificado)."""
        T = np.asarray(T, float)
        Vd, kd = 0.0, 0.0
        for k in ORDER:
            if k == "H2O":
                continue
            c = COMPONENTS[k]
            v = self.mass(k) / c.rho_dense
            Vd = Vd + v
            kd = kd + v * c.k_dense
        kd = np.where(Vd > 0, kd / np.maximum(Vd, 1e-30), 0.2)
        V = np.maximum(self.volume(T), 1e-30)
        solid = np.clip(Vd / V, 0.0, 1.0)              # fração de material denso no volume aparente
        k_gas = 0.026 + 5.0e-5 * (T - 300.0)
        k_rad = 4 * 5.670374e-8 * eps / (2 - eps) * d_piece * T ** 3 * (1 - solid)
        # mistura série/paralelo: k_s^φ k_g^(1−φ) (média geométrica) + radiação nos vazios
        return kd ** solid * k_gas ** (1 - solid) + k_rad


# ------------------------------------------------------------------- TGA
def tga(key_or_masses, beta_K_min: float, T_start: float = 300.0, T_end: float = 1000.0, dT: float = 0.25):
    """TGA a taxa constante para um componente (chave) ou uma mistura {chave: massa}.

    Devolve T [K], massa relativa restante (sólidos inclusive) e −dm/dT [1/K]."""
    masses = {key_or_masses: 1.0} if isinstance(key_or_masses, str) else dict(key_or_masses)
    st = KineticState(masses, 1)
    m_init = st.total()[0]
    beta = beta_K_min / 60.0
    Ts = np.arange(T_start, T_end + dT, dT)
    mr = np.empty_like(Ts)
    dt = dT / beta
    for i, T in enumerate(Ts):
        if "H2O" in masses and T > 373.15:       # TGA: a umidade sai ao passar de 100 °C (taxa rápida)
            st.p["H2O"] = np.minimum(st.p["H2O"] + 0.2 * dT, 1.0)
        st.react(np.array([T]), dt)
        mr[i] = st.total()[0] / m_init
    return Ts, mr, -np.gradient(mr, Ts)


def dtg_peak(key: str, beta_K_min: float) -> float:
    """Temperatura do pico de DTG [°C] (com refino parabólico)."""
    T, _, d = tga(key, beta_K_min)
    i = int(np.argmax(d))
    if 0 < i < len(T) - 1:
        y0, y1, y2 = d[i - 1], d[i], d[i + 1]
        den = y0 - 2 * y1 + y2
        off = 0.5 * (y0 - y2) / den if den != 0 else 0.0
        return float(T[i] + off * (T[1] - T[0]) - 273.15)
    return float(T[i] - 273.15)


def t_k1e3(key: str) -> float:
    """Temperatura em que k = 10⁻³ s⁻¹ [°C] (verificação da pesquisa)."""
    c = COMPONENTS[key]
    if c.law == "sc":
        A, E = c.sc[1]
    else:
        A, E = c.A, c.E
    return E / (R_GAS * math.log(A / 1e-3)) - 273.15


def ultimate_yields(charge: Charge, T: float = 773.15, hours: float = 10.0) -> dict:
    """Rendimentos finais em isoterma longa (sem craqueamento secundário): {lump/sólido: fração da carga}."""
    st = KineticState(charge.masses(), 1)
    tot0 = st.total()[0]
    out = {k: 0.0 for k in LUMPS}
    out["H2O"] += charge.mass * charge.moisture
    st.p["H2O"][:] = 1.0
    t, dt = 0.0, 5.0
    while t < hours * 3600:
        prod, _, _ = st.react(np.array([T]), dt)
        for k, v in prod.items():
            out[k] += float(v[0])
        t += dt
    out["CHAR"] = float(st.mass("CHAR")[0] + st.mass("CB")[0])
    out["ASH"] = float(st.mass("INERT")[0])
    out["resto"] = float(st.total()[0]) - out["CHAR"] - out["ASH"]
    return {k: v / tot0 for k, v in out.items()}

In [ ]:
%%writefile tamborkit/furnace.py
"""Fornalha em volta do tambor: modelo zonal de Hottel (gás bem misturado + refratário + casco).

A fornalha é a caixa de alvenaria/fibra cerâmica que envolve o tambor. Os queimadores (combustível
auxiliar no início, depois o gás não condensável da própria pirólise) queimam na câmara de combustão sob o
tambor; o gás quente sobe pelos dois corredores laterais (folga entre o casco e o refratário), junta-se no
topo e sai pela chaminé no fundo do tambor (x = L).

Em cada fatia axial há quatro zonas de gás — câmara (FB), lateral direita (SR), lateral esquerda (SL) e
topo (TP) — cada uma com um nó de refratário. Por zona, regime quase estacionário (o gás responde em
segundos; o casco e a carga, em minutos e horas):

  gás:        ṁ c_p (T_g − T_entra) = Q_liberado − Q_gás→casco − Q_gás→refratário
  refratário: Q_gás→refratário = Q_refratário→casco + U A_r (T_r − T_amb)
  casco (setor j): q_j = σ ε_g (1+ε_s)/2 (T_g⁴ − T_j⁴) + σ ε_rs (1 − ε_g)(T_r⁴ − T_j⁴) + h_c (T_g − T_j)

ε_g do gás de combustão pela soma ponderada de gases cinzas (WSGG, Smith, Shen & Friedman 1982, razão
p_H2O/p_CO2 = 1) com o comprimento médio de feixe L_m = 3,6 V/A; ε_rs = 1/(1/ε_r + 1/ε_s − 1).
A convecção é pequena (Re ~2400 no corredor, 2–6 W/m²K): o casco é aquecido sobretudo por radiação.
"""
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np

SIGMA = 5.670374419e-8
CP_FG = 1250.0          # c_p médio dos gases de combustão, 300–1500 K [J/(kg K)]

# WSGG de Smith et al. (1982), p_w/p_c = 1: κ_i [1/(atm m)] e b_ij (a_i = Σ_j b_ij T^(j−1))
_WSGG_K = np.array([0.4303, 7.055, 178.1])
_WSGG_B = np.array([[5.150e-1, -2.303e-4, 0.9779e-7, -1.494e-11],
                    [0.7749e-1, 3.399e-4, -2.297e-7, 3.770e-11],
                    [1.907e-1, -1.824e-4, 0.5608e-7, -0.5122e-11]])

FUELS = {
    # PCI [J/kg], ar estequiométrico [kg/kg], (mols CO2 + H2O, mols O2 estequiométrico) por mol de combustível
    "GLP": dict(LHV=46.35e6, AFR=15.6, nCH=7.0, nO2=5.0, name="GLP (propano)"),
    "diesel": dict(LHV=42.8e6, AFR=14.5, nCH=25.0, nO2=18.5, name="óleo diesel"),
    "G_HC": dict(LHV=46.3e6, AFR=15.43, nCH=3.0, nO2=2.2, name="gás de pirólise (hidrocarbonetos)"),
    "G_OX": dict(LHV=8.0e6, AFR=2.2, nCH=1.0, nO2=0.35, name="gás de pirólise (CO/CO₂)"),
}


def wsgg_emissivity(T, pL):
    """Emissividade total do gás (H2O + CO2), T [K], pL = (p_w + p_c)·L [atm m]."""
    T = np.clip(np.asarray(T, float), 600.0, 2400.0)
    Tp = np.stack([np.ones_like(T), T, T ** 2, T ** 3], -1)
    a = Tp @ _WSGG_B.T                                 # [..., 3]
    return (a * (1 - np.exp(-_WSGG_K * np.asarray(pL, float)[..., None]))).sum(-1)


def flue_pwc(lam: float, fuel: str = "GLP") -> float:
    """Pressão parcial p_H2O + p_CO2 [atm] nos gases de combustão com excesso de ar λ."""
    f = FUELS[fuel]
    n_prod = f["nCH"] + (lam - 1) * f["nO2"] + 3.76 * lam * f["nO2"]
    return f["nCH"] / n_prod


@dataclass
class Furnace:
    gap: float = 0.20               # folga casco–refratário nos corredores [m]
    fb_height: float = 0.55         # altura da câmara de combustão abaixo do casco [m]
    fb_width: float = 2.0           # largura da câmara [m]
    burners_x: tuple = (1.2, 3.7)   # posições axiais dos queimadores [m]
    flame_sigma: float = 0.7        # espalhamento axial da liberação de calor de cada chama [m]
    axial_mix: float = 0.5          # fração da liberação/vazão distribuída uniformemente (mistura na câmara)
    Q_max: float = 465e3            # capacidade dos queimadores [W] (2 × 200 000 kcal/h, BLJ-3, não verificado)
    Q_min: float = 15e3             # chama piloto [W]
    lam: float = 1.3                # excesso de ar
    aux_fuel: str = "GLP"
    T_air: float = 298.15
    T_amb: float = 298.15
    eps_r: float = 0.85             # refratário / fibra cerâmica
    eps_s: float = 0.85             # casco oxidado (lado de fora)
    ins_thick: float = 0.15         # isolamento [m]
    k_ins: float = 0.12             # fibra cerâmica a ~600 °C [W/(m K)]
    h_out: float = 10.0             # convecção + radiação na face externa [W/(m² K)]
    h_conv: float = 5.0             # convecção gás → casco/refratário [W/(m² K)]
    # setores (ângulo φ = atan2(z, y) no referencial do laboratório) de cada zona, em graus
    zones: tuple = (("FB", -150.0, -30.0), ("SR", -30.0, 75.0), ("TP", 75.0, 105.0), ("SL", 105.0, 210.0))

    @property
    def U_ins(self) -> float:
        return 1.0 / (self.ins_thick / self.k_ins + 1.0 / self.h_out)

    def zone_of(self, phi_deg: np.ndarray) -> np.ndarray:
        """Índice da zona (0 FB, 1 SR, 2 TP, 3 SL) de cada ângulo de laboratório."""
        p = (np.asarray(phi_deg, float) + 150.0) % 360.0 - 150.0
        out = np.zeros(p.shape, int)
        for i, (_, a, b) in enumerate(self.zones):
            out[(p >= a) & (p < b)] = i
        return out

    def release_weights(self, xc: np.ndarray) -> np.ndarray:
        """Distribuição axial da liberação de calor (e da vazão de gases) entre as fatias."""
        g = sum(np.exp(-0.5 * ((xc - xb) / self.flame_sigma) ** 2) for xb in self.burners_x)
        g = g / g.sum()
        u = np.full_like(xc, 1.0 / len(xc))
        return self.axial_mix * u + (1 - self.axial_mix) * g


class FurnaceZones:
    """Resolve as zonas de gás e refratário de todas as fatias, dado o casco e a queima."""

    def __init__(self, fur: Furnace, R_out: float, L: float, xc: np.ndarray, phi_deg: np.ndarray):
        self.f = fur
        self.R, self.L, self.xc = R_out, L, xc
        self.nx, self.nphi = len(xc), len(phi_deg)
        dx = L / self.nx
        self.dx = dx
        self.zone = fur.zone_of(phi_deg)                            # zona de cada setor (lab)
        self.A_sec = 2 * math.pi * R_out / self.nphi * dx           # área de um setor numa fatia [m²]
        self.w = fur.release_weights(xc)
        # geometria de cada zona por fatia: área de refratário e comprimento médio de feixe
        arcs = np.array([(b - a) for _, a, b in fur.zones]) * math.pi / 180.0
        A_shell_z = arcs * R_out * dx
        g, H, W = fur.gap, fur.fb_height, fur.fb_width
        A_ref, V = np.zeros(4), np.zeros(4)
        # câmara: piso + paredes laterais da caixa abaixo do casco
        A_ref[0] = (W + 2 * (H + 0.3 * R_out)) * dx
        V[0] = (W * (H + 0.5 * R_out) - 0.5 * arcs[0] * R_out ** 2 * 0.6) * dx
        for i in (1, 2, 3):            # corredores e topo: anel de folga g
            A_ref[i] = arcs[i] * (R_out + g) * dx
            V[i] = 0.5 * arcs[i] * ((R_out + g) ** 2 - R_out ** 2) * dx
        self.A_ref, self.A_shell_z = A_ref, A_shell_z
        self.Lm = 3.6 * V / (A_ref + A_shell_z)
        self.eps_rs = 1.0 / (1.0 / fur.eps_r + 1.0 / fur.eps_s - 1.0)
        self.Tg = np.full((self.nx, 4), 900.0)
        self.Tr = np.full((self.nx, 4), 800.0)

    def _zone_solve(self, Tin, mdot, Qrel, Ts_list, Ash_list, Ar, Lm, pwc, Tg0, Tr0, iters=30):
        """Newton 2×2 (T_g, T_r) vetorizado; Ts_list [nx, ns], Ash_list [nx, ns] (áreas por setor)."""
        f = self.f
        Tg, Tr = Tg0.copy(), Tr0.copy()
        es = (1 + f.eps_s) / 2
        er = (1 + f.eps_r) / 2
        U = f.U_ins
        for _ in range(iters):
            eg = wsgg_emissivity(Tg, pwc * Lm)
            Ts4 = Ts_list ** 4
            # fluxos para o casco
            qg = SIGMA * eg[:, None] * es * (Tg[:, None] ** 4 - Ts4) + f.h_conv * (Tg[:, None] - Ts_list)
            qr = SIGMA * self.eps_rs * (1 - eg[:, None]) * (Tr[:, None] ** 4 - Ts4)
            Qgs = (qg * Ash_list).sum(1)
            Qrs = (qr * Ash_list).sum(1)
            Qgr = Ar * (SIGMA * eg * er * (Tg ** 4 - Tr ** 4) + f.h_conv * (Tg - Tr))
            F1 = mdot * CP_FG * (Tg - Tin) - Qrel + Qgs + Qgr
            F2 = Qgr - Qrs - Ar * U * (Tr - f.T_amb)
            # jacobiano (ε_g congelado na iteração)
            dqg = (SIGMA * eg[:, None] * es * 4 * Tg[:, None] ** 3 + f.h_conv) * Ash_list
            dQgr_g = Ar * (SIGMA * eg * er * 4 * Tg ** 3 + f.h_conv)
            dQgr_r = -Ar * (SIGMA * eg * er * 4 * Tr ** 3 + f.h_conv)
            dQrs_r = (SIGMA * self.eps_rs * (1 - eg[:, None]) * 4 * Tr[:, None] ** 3 * Ash_list).sum(1)
            J11 = mdot * CP_FG + dqg.sum(1) + dQgr_g
            J12 = dQgr_r
            J21 = dQgr_g
            J22 = dQgr_r - dQrs_r - Ar * U
            det = J11 * J22 - J12 * J21
            dTg = (-F1 * J22 + F2 * J12) / det
            dTr = (-F2 * J11 + F1 * J21) / det
            Tg = np.clip(Tg + np.clip(dTg, -300, 300), 250.0, 2600.0)
            Tr = np.clip(Tr + np.clip(dTr, -300, 300), 250.0, 2600.0)
            if np.max(np.abs(dTg)) + np.max(np.abs(dTr)) < 0.01:
                break
        eg = wsgg_emissivity(Tg, pwc * Lm)
        q = (SIGMA * eg[:, None] * es * (Tg[:, None] ** 4 - Ts_list ** 4) + f.h_conv * (Tg[:, None] - Ts_list)
             + SIGMA * self.eps_rs * (1 - eg[:, None]) * (Tr[:, None] ** 4 - Ts_list ** 4))
        loss = Ar * U * (Tr - f.T_amb)
        return Tg, Tr, q, loss, eg

    def solve(self, Ts_lab: np.ndarray, Q_fire: float, m_fuel_flue: float, pwc: float, phi_deg=None):
        """Ts_lab [nx, nphi] temperatura do casco em cada setor de laboratório.

        Q_fire: liberação total [W]; m_fuel_flue: vazão total de gases de combustão [kg/s].
        phi_deg: ângulo de laboratório atual de cada setor (o casco gira; padrão = ângulos da construção).
        Devolve q [nx, nphi] (fluxo da fornalha para o casco, W/m²) e um dicionário de diagnóstico."""
        f = self.f
        zone = self.zone if phi_deg is None else f.zone_of(phi_deg)
        q = np.zeros_like(Ts_lab)
        mdot = np.maximum(m_fuel_flue, 1e-4) * self.w
        Qrel = Q_fire * self.w
        loss = np.zeros((self.nx, 4))
        eg_all = np.zeros((self.nx, 4))
        order = [0, 1, 3, 2]                    # câmara → laterais → topo
        T_side_out = np.zeros((self.nx, 2))
        for zi in order:
            sel = zone == zi
            Ts = Ts_lab[:, sel]
            Ash = np.full(Ts.shape, self.A_sec)
            if zi == 0:
                Tin, md, Qr = np.full(self.nx, f.T_air), mdot, Qrel
            elif zi in (1, 3):
                Tin, md, Qr = self.Tg[:, 0], 0.5 * mdot, np.zeros(self.nx)
            else:
                # topo: recebe os dois corredores da fatia + o fluxo axial vindo das fatias anteriores
                Tg_top = np.zeros(self.nx); Tr_top = np.zeros(self.nx); q_top = np.zeros_like(Ts)
                ls, eg_t = np.zeros(self.nx), np.zeros(self.nx)
                m_up, T_up = 0.0, f.T_air
                for k in range(self.nx):
                    m_side = 0.5 * mdot[k]
                    m_in = m_up + 2 * m_side
                    Tin_k = (m_up * T_up + m_side * (self.Tg[k, 1] + self.Tg[k, 3])) / max(m_in, 1e-9)
                    tg, tr, qq, lo, eg = self._zone_solve(np.array([Tin_k]), np.array([m_in]), np.zeros(1),
                                                         Ts[k:k + 1], Ash[k:k + 1], self.A_ref[zi],
                                                         self.Lm[zi], pwc, self.Tg[k:k + 1, zi], self.Tr[k:k + 1, zi])
                    Tg_top[k], Tr_top[k], q_top[k], ls[k], eg_t[k] = tg[0], tr[0], qq[0], lo[0], eg[0]
                    m_up, T_up = m_in, tg[0]
                self.Tg[:, zi], self.Tr[:, zi] = Tg_top, Tr_top
                q[:, sel] = q_top
                loss[:, zi], eg_all[:, zi] = ls, eg_t
                self.T_flue = T_up
                continue
            tg, tr, qq, lo, eg = self._zone_solve(Tin, md, Qr, Ts, Ash, self.A_ref[zi], self.Lm[zi], pwc,
                                                 self.Tg[:, zi], self.Tr[:, zi])
            self.Tg[:, zi], self.Tr[:, zi] = tg, tr
            q[:, sel] = qq
            loss[:, zi], eg_all[:, zi] = lo, eg
        Q_shell = float((q * self.A_sec).sum())
        Q_flue = float(m_fuel_flue * CP_FG * (self.T_flue - f.T_air))
        diag = dict(Q_shell=Q_shell, Q_loss=float(loss.sum()), Q_flue=Q_flue, T_flue=float(self.T_flue),
                    Tg=self.Tg.copy(), Tr=self.Tr.copy(), eps_g=eg_all,
                    closure=(Q_fire - Q_shell - float(loss.sum()) - Q_flue) / max(Q_fire, 1.0))
        return q, diag


def fuel_mix(Q_fire: float, ncg: dict, aux: str, lam: float):
    """Divide a demanda de queima entre o gás de pirólise disponível e o auxiliar.

    ncg: {"G_HC": kg/s, "G_OX": kg/s} disponível. Devolve (uso {combustível: kg/s}, ṁ dos gases de combustão,
    p_wc, potência do gás de pirólise usada, potência que sobra para o flare)."""
    Q_ncg_avail = sum(ncg.get(k, 0.0) * FUELS[k]["LHV"] for k in ("G_HC", "G_OX"))
    Q_from_ncg = min(Q_ncg_avail, Q_fire)
    frac = Q_from_ncg / Q_ncg_avail if Q_ncg_avail > 0 else 0.0
    use = {k: ncg.get(k, 0.0) * frac for k in ("G_HC", "G_OX")}
    use[aux] = max(Q_fire - Q_from_ncg, 0.0) / FUELS[aux]["LHV"]
    m_fg = sum(m * (1 + lam * FUELS[k]["AFR"]) for k, m in use.items())
    # p_wc ponderada pela vazão de gases de cada combustível
    w = {k: m * (1 + lam * FUELS[k]["AFR"]) for k, m in use.items()}
    tot = sum(w.values())
    pwc = sum(w[k] * flue_pwc(lam, k) for k in w) / tot if tot > 0 else flue_pwc(lam, aux)
    return use, m_fg, pwc, Q_from_ncg, Q_ncg_avail - Q_from_ncg

In [ ]:
%%writefile tamborkit/batch.py
"""Batelada completa do tambor rotativo: fornalha → casco girando → carga → vapores → condensador → queimador.

Escalas de tempo (pesquisa R5): contato ~ms ≪ chama ~s ≪ volta do tambor 75 s ~ residência do vapor 1–4 min
< inércia do casco 3–10 min ≪ batelada 6–8 h. O modelo avança no relógio lento (Δt ~2 s) e trata como
quase estacionários a fornalha (zonas de Hottel) e o espaço livre (escoamento pistonado dos vapores).

Carga (dois modos, mesma química e mesma fornalha):
  * "rotativo": a cada volta a carga renova o contato com a parede; em cada fatia axial o leito é bem
    misturado na seção (a mistura leva poucas voltas, ≪ batelada — verificado pelo DEM) e a parede coberta
    troca calor pelo modelo de penetração (Tscheng & Watkinson 1979; Schlünder & Mollekopf 1984):
        1/h_wb = 1/h_contato + 1/h_pen,   h_pen = 2 √(k ρ c / (π t_c)),   t_c = 2ε/Ω;
    as aletas entram como aletas térmicas (eficiência tanh(mL)/mL) e como cortina que intercepta a radiação.
  * "estatico": tambor parado; a carga é uma pilha que só conduz calor (camadas lagrangianas da parede até a
    superfície, área variando do arco molhado à corda), com a mancha quente do casco fixa sobre a chama.

Parede exposta ↔ superfície da carga: radiosidade entre os setores do casco (cordas inscritas) e a corda do
leito, fatores de forma pelas cordas cruzadas de Hottel. Espaço livre: vapores em escoamento pistonado até a
saída (x = L), convecção com parede e carga, craqueamento secundário (cera → óleo → gás + aromáticos + coque;
alcatrão → gás/coque). Condensador: óleo (O, W, A, OX, TAR), fase aquosa (H2O, HCl) e gás não condensável,
que volta ao queimador; o excedente vai para o flare.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field

import numpy as np

from . import feed as fd
from .furnace import Furnace, FurnaceZones, fuel_mix, FUELS, SIGMA
from .geometry import Drum, fill_half_angle

T0 = 298.15
# espaço livre: craqueamento secundário (pesquisa, brief §3.7; Shafizadeh–Chin para o alcatrão)
K_O = (1.0e16, 300e3)              # óleo → gás/aromáticos/coque
K_W_FACTOR = 3.0                   # k_cera = 3 k_óleo
O_CRACK = {"G_HC": 0.60, "A": 0.30, "COKE": 0.10}
K_TAR_GAS = (3.09e6, 93.3e3)
K_TAR_CHAR = (1.0e5, 107.5e3)
MW_LUMP = {"G_HC": 0.025, "G_OX": 0.033, "O": 0.15, "W": 0.40, "A": 0.10, "OX": 0.12, "TAR": 0.10,
           "H2O": 0.018, "HCl": 0.0365, "N2": 0.028}
CP_VAP = 2200.0
# grupos de componentes para o gráfico da taxa de pirólise (na ordem em que reagem)
GROUPS = {"água": ["H2O"], "PVC": ["PVC", "PVCr"], "celulósicos": ["CELL"], "PS": ["PS"], "PP": ["PP"],
          "PE": ["LDPE", "HDPE"], "PET": ["PET"], "borracha": ["RUBa", "RUBb", "RUBc"]}


@dataclass
class BedModel:
    """Parâmetros da troca de calor interna (valores de projeto [H] e faixas da literatura)."""
    h_contact: tuple = (30.0, 150.0, 70.0)     # parede→pedaços soltos / fundido / carvão [W/(m² K)] [H]
    beta_deg: tuple = (32.0, 4.0, 33.0)        # ângulo dinâmico: soltos (DEM: 32°) / fundido / carvão
    curtain: tuple = (0.25, 0.08, 0.30)        # área extra de cortina / área da superfície (com aletas) [H/DEM]
    d_pore: tuple = (0.02, 0.002, 0.004)       # tamanho de poro para a radiação no leito [m]
    eps_wall_in: float = 0.80                  # casco por dentro (com depósitos)
    eps_bed: float = 0.90
    h_fb: float = 6.0                          # convecção no espaço livre [W/(m² K)]
    D_ax: float = 2.0e-4                       # dispersão axial do leito girando [m²/s] [H]
    purge: float = 2.0e-4                      # N2 de purga/vedação [kg/s]
    eps_out: float = 0.85


@dataclass
class Control:
    setpoint_C: float = 500.0                  # "500 graus" (UFRJ): parede média do tambor
    variable: str = "parede"                   # "parede" (casco médio) ou "carga"
    T_shell_max_C: float = 650.0               # limite do aço do casco (Q345R ≤ 475 °C de serviço; alarme)
    Kp: float = 4000.0                         # W/K
    Ti: float = 900.0                          # s
    ramp_K_min: float = 2.5                    # rampa de partida do setpoint (protege o casco e a vedação) [H]
    ncg_recycle: bool = True


@dataclass
class BatchConfig:
    mode: str = "rotativo"
    nx: int = 7
    nphi: int = 36
    n_layers: int = 24
    dt: float = 2.0
    t_max_h: float = 10.0
    end_frac: float = 0.03                     # fim: produção < 3 % do pico (e carga > 400 °C)
    log_every: float = 30.0                    # s
    snap_every: float = 300.0                  # s
    wall: float = 0.012
    rho_s: float = 7850.0
    cp_s: float = 560.0
    k_s: float = 40.0
    bed: BedModel = field(default_factory=BedModel)
    control: Control = field(default_factory=Control)


# ------------------------------------------------------------------ geometria
def crossed_strings(P1, P2, Q1, Q2):
    """F(P→Q) entre segmentos 2D num recinto convexo (cordas cruzadas de Hottel). Arrays [..., 2]."""
    d = lambda a, b: np.linalg.norm(a - b, axis=-1)  # noqa: E731
    L = d(P1, P2)
    cross = d(P1, Q2) + d(P2, Q1)
    uncross = d(P1, Q1) + d(P2, Q2)
    return np.abs(cross - uncross) / (2 * np.maximum(L, 1e-12))


def arc_overlap(a0, a1, b0, b1):
    """Comprimento angular de [a0, a1] ∩ [b0, b1] (rad), com ângulos já desembrulhados."""
    out = 0.0
    for s in (-2 * math.pi, 0.0, 2 * math.pi):
        out = out + np.clip(np.minimum(a1, b1 + s) - np.maximum(a0, b0 + s), 0.0, None)
    return out


class BatchDrum:
    def __init__(self, drum: Drum, charge: fd.Charge, furnace: Furnace | None = None,
                 cfg: BatchConfig | None = None, rpm: float | None = None):
        self.drum, self.charge = drum, charge
        self.fur = furnace or Furnace()
        self.cfg = cfg or BatchConfig()
        c = self.cfg
        self.rotating = c.mode == "rotativo"
        self.rpm = (drum.rpm if rpm is None else rpm) if self.rotating else 0.0
        self.Omega = 2 * math.pi * self.rpm / 60.0
        R, L = drum.R, drum.L
        self.R, self.L = R, L
        self.dx = L / c.nx
        self.xc = (np.arange(c.nx) + 0.5) * self.dx
        self.dphi = 2 * math.pi / c.nphi
        self.phi0 = -math.pi + (np.arange(c.nphi) + 0.5) * self.dphi   # ângulo no referencial do tambor
        self.A_sec = R * self.dphi * self.dx
        self.R_out = R + c.wall
        self.A_out = self.R_out * self.dphi * self.dx
        self.zones = FurnaceZones(self.fur, self.R_out, L, self.xc, np.degrees(self.phi0))
        # casco
        self.Ts = np.full((c.nx, c.nphi), T0)
        self.m_s = c.rho_s * c.wall * self.A_sec
        # aletas como aletas térmicas
        if drum.n_lifters:
            m_fin = math.sqrt(2 * 60.0 / (c.k_s * drum.lifter_t))
            eta = math.tanh(m_fin * drum.lifter_h) / (m_fin * drum.lifter_h)
            self.fin = drum.n_lifters * 2 * drum.lifter_h * eta / (2 * math.pi * R)
        else:
            self.fin = 0.0
        if not self.rotating:
            self.fin *= 0.5        # parado: só as aletas que ficaram dentro da pilha ajudam
        # carga
        m = charge.masses()
        n = c.nx if self.rotating else c.nx * c.n_layers
        per = {k: v / n for k, v in m.items()}
        self.kin = fd.KineticState(per, n)
        st1 = fd.KineticState(m, 1)
        V_raw = float(st1.volume(np.array([T0]))[0])
        self.loose_scale = V_raw / (charge.fill * drum.volume())
        self.T = np.full(n, T0)
        self.Tg = np.full(c.nx, T0)
        self.m_charge0 = sum(m.values())
        self.m0_comp = dict(m)
        # contabilidade
        self.t, self.theta = 0.0, 0.0
        self.prod = {k: 0.0 for k in fd.LUMPS + ["COKE"]}
        self.cum = {k: 0.0 for k in ["oleo", "aquoso", "gas", "coque", "Q_fire", "Q_aux", "Q_ncg", "Q_flare",
                                     "Q_shell", "Q_flue", "Q_loss", "Q_in_drum", "Q_rxn", "Q_evap", "aux_kg",
                                     "Q_vap_out"]}
        self.Q_fire = 0.3 * self.fur.Q_max
        self.I_err = 0.0
        self.ncg_rate = {"G_HC": 0.0, "G_OX": 0.0}
        self.peak_rate = 0.0
        self.log, self.snaps = [], []
        self.done = False
        self.E_shell0 = self._shell_energy()
        self.H_bed0 = self._bed_enthalpy()

    # -------------------------------------------------------------- utilidades
    def _shell_energy(self):
        return float((self.m_s * self.cfg.cp_s * (self.Ts - T0)).sum())

    def _bed_enthalpy(self):
        return float(np.sum(self.kin.enthalpy(self.T)))

    def _slice_view(self, a):
        """Array por volume do leito → [nx, n_l] (estático) ou [nx, 1]."""
        return a.reshape(self.cfg.nx, -1)

    def bed_geometry(self):
        """Por fatia: enchimento F, meio ângulo ε, ângulo do leito β, centro do arco coberto φ_c."""
        c, b = self.cfg, self.cfg.bed
        V = self._slice_view(self.kin.volume(self.T, self.loose_scale)).sum(1)
        F = np.clip(V / (math.pi * self.R ** 2 * self.dx), 1e-4, 0.95)
        eps = np.array([fill_half_angle(f) for f in F])
        Tm = self._slice_view(self.T).mean(1)
        lo, me, re = (self._slice_view(a).mean(1) for a in self.kin.state_fractions(self.T))
        if self.rotating:
            beta = np.radians(lo * b.beta_deg[0] + me * b.beta_deg[1] + re * b.beta_deg[2])
        else:
            beta = np.zeros(c.nx)
        phic = -math.pi / 2 + beta
        return dict(V=V, F=F, eps=eps, beta=beta, phic=phic, frac=(lo, me, re), Tm=Tm)

    # -------------------------------------------------------- radiação interna
    def _radiation(self, phi_lab, geo, Tbs):
        """Radiosidade por fatia entre setores expostos do casco e a superfície da carga.

        Devolve q_rad [nx, nphi] (perda de cada setor por radiação, W/m² de setor) e Q_bed [nx] (W)."""
        c, b = self.cfg, self.cfg.bed
        nx, nphi = c.nx, c.nphi
        R = self.R
        q_out = np.zeros((nx, nphi))
        Qb = np.zeros(nx)
        for k in range(nx):
            pc = geo["phic"][k]
            mid = pc + (phi_lab - pc + math.pi) % (2 * math.pi) - math.pi      # janela [φc − π, φc + π)
            a0, a1 = mid - 0.5 * self.dphi, mid + 0.5 * self.dphi
            c0, c1 = pc - geo["eps"][k], pc + geo["eps"][k]
            # parte exposta de cada setor: o intervalo do setor menos o arco coberto (maior pedaço)
            lo = np.where((a0 >= c0) & (a0 < c1), c1, a0)
            hi = np.where((a1 > c0) & (a1 <= c1), c0, a1)
            cov_mid = (a0 < c0) & (a1 > c1)
            hi = np.where(cov_mid, c0, hi)
            exp_len = np.clip(hi - lo, 0.0, None)
            exp_len = np.where((a0 >= c0) & (a1 <= c1), 0.0, exp_len)
            P1 = np.stack([R * np.cos(lo), R * np.sin(lo)], -1)
            P2 = np.stack([R * np.cos(hi), R * np.sin(hi)], -1)
            B1 = np.array([R * math.cos(c0), R * math.sin(c0)])
            B2 = np.array([R * math.cos(c1), R * math.sin(c1)])
            Ps = np.concatenate([P1, B2[None]], 0)
            Pe = np.concatenate([P2, B1[None]], 0)
            n = nphi + 1
            Lseg = np.linalg.norm(Pe - Ps, axis=-1)
            Lseg[:nphi] = np.where(exp_len > 1e-6, Lseg[:nphi], 0.0)
            F = crossed_strings(Ps[:, None], Pe[:, None], Ps[None], Pe[None])
            np.fill_diagonal(F, 0.0)
            F[:, Lseg == 0] = 0.0
            F[Lseg == 0, :] = 0.0
            rs = F.sum(1, keepdims=True)
            F = np.where(rs > 0, F / np.maximum(rs, 1e-12), 0.0)
            eps = np.full(n, b.eps_wall_in); eps[-1] = b.eps_bed
            T = np.concatenate([self._shell_lab_T[k], [Tbs[k]]])
            Eb = SIGMA * T ** 4
            Amat = np.eye(n) - (1 - eps)[:, None] * F
            J = np.linalg.solve(Amat, eps * Eb)
            qnet = eps / (1 - eps) * (Eb - J)                 # W/m² da superfície (positivo = perde)
            Qnet = qnet * Lseg * self.dx
            q_out[k] = Qnet[:nphi] / self.A_sec
            Qb[k] = -Qnet[-1]
        return q_out, Qb

    # ------------------------------------------------------------- um passo
    def step(self):
        c, b, cn = self.cfg, self.cfg.bed, self.cfg.control
        dt = c.dt
        nx, nphi = c.nx, c.nphi
        geo = self.bed_geometry()
        phi_lab = self.phi0 + self.theta                                   # ângulo de laboratório dos setores
        phi_wrapped = (phi_lab + math.pi) % (2 * math.pi) - math.pi
        # cobertura de cada setor
        cov = np.zeros((nx, nphi))
        for k in range(nx):
            c0, c1 = geo["phic"][k] - geo["eps"][k], geo["phic"][k] + geo["eps"][k]
            cov[k] = arc_overlap(phi_wrapped - 0.5 * self.dphi, phi_wrapped + 0.5 * self.dphi, c0, c1) / self.dphi
        self._shell_lab_T = self.Ts
        # ---- queima e fornalha
        use, m_fg, pwc, Q_ncg_used, Q_flare = fuel_mix(self.Q_fire, self.ncg_rate if cn.ncg_recycle else {},
                                                       self.fur.aux_fuel, self.fur.lam)
        q_f, fdiag = self.zones.solve(self.Ts, self.Q_fire, m_fg, pwc, np.degrees(phi_wrapped))
        # ---- carga: propriedades e coeficientes
        lo, me, re = geo["frac"]
        Tb_sl = self._slice_view(self.T)
        T_bot = Tb_sl[:, 0]
        T_top = Tb_sl[:, -1]
        h_c = lo * b.h_contact[0] + me * b.h_contact[1] + re * b.h_contact[2]
        d_pore = lo * b.d_pore[0] + me * b.d_pore[1] + re * b.d_pore[2]
        curt = (lo * b.curtain[0] + me * b.curtain[1] + re * b.curtain[2]) if (self.rotating and self.drum.n_lifters) else 0.0
        if self.rotating:
            kb = self.kin.conductivity(self.T, d_piece=np.repeat(d_pore, len(self.T) // nx))
            mass = self.kin.total()
            Vb = self.kin.volume(self.T, self.loose_scale)
            rho_b = mass / np.maximum(Vb, 1e-9)
            cp_b = self.kin.heat_capacity(self.T) / np.maximum(mass, 1e-9)
            t_c = np.maximum(2 * geo["eps"] / max(self.Omega, 1e-9), 1.0)
            h_pen = 2 * np.sqrt(kb * rho_b * cp_b / (math.pi * t_c))
            h_wb = 1.0 / (1.0 / h_c + 1.0 / h_pen)
        else:
            # primeira camada: contato em série com meia espessura de condução
            k0 = self.kin.conductivity(self.T, d_piece=np.repeat(d_pore, c.n_layers))
            k0 = self._slice_view(k0)[:, 0]
            t0 = self._layer_geom(geo)["t"][:, 0]
            h_wb = 1.0 / (1.0 / h_c + 0.5 * t0 / np.maximum(k0, 1e-3))
        h_wb_eff = h_wb * (1 + self.fin)
        # ---- trocas internas por setor
        q_bed = cov * h_wb_eff[:, None] * (self.Ts - T_bot[:, None])        # W/m² de setor
        q_rad, Q_rad_bed = self._radiation(phi_wrapped, geo, T_top)
        Q_curt = curt * np.clip(Q_rad_bed, 0.0, None)                       # cortina intercepta mais radiação
        q_rad_c = q_rad.copy()
        pos = np.clip(q_rad, 0.0, None) * (1 - cov)
        share = pos / np.maximum(pos.sum(1, keepdims=True), 1e-12)
        q_rad_c += share * (Q_curt / self.A_sec)[:, None]
        exposed = 1 - cov
        q_fb = exposed * b.h_fb * (self.Ts - self.Tg[:, None])
        q_in = q_bed + q_rad_c + q_fb                                        # sai do casco para dentro
        # condução no aço (circunferencial e axial)
        ks, w = c.k_s, c.wall
        ds = self.R * self.dphi
        lap = (np.roll(self.Ts, 1, 1) + np.roll(self.Ts, -1, 1) - 2 * self.Ts) / ds ** 2
        Tpx = np.vstack([self.Ts[:1], self.Ts, self.Ts[-1:]])
        lap += (Tpx[2:] + Tpx[:-2] - 2 * self.Ts) / self.dx ** 2
        dTs = dt * (self.A_out * q_f - self.A_sec * q_in + ks * w * self.A_sec * lap) / (self.m_s * c.cp_s)
        # ---- energia que entra na carga
        Q_wall = (q_bed * self.A_sec).sum(1)
        A_surf = 2 * self.R * np.sin(geo["eps"]) * self.dx
        Q_conv_top = b.h_fb * A_surf * (self.Tg - T_top)
        Q_top = Q_rad_bed + Q_curt + Q_conv_top
        # ---- química
        prod, heat, rx_rates = self.kin.react(self.T, dt)
        if self.rotating:
            Q = Q_wall + Q_top
            # dispersão axial
            C = self.kin.heat_capacity(self.T)
            Tp = np.concatenate([self.T[:1], self.T, self.T[-1:]])
            Q_ax = b.D_ax * (C / (self.dx ** 2)) * (Tp[2:] + Tp[:-2] - 2 * self.T)
            Q = Q + Q_ax
            self.T = self.T + (Q * dt - heat) / np.maximum(C, 1e-6)
        else:
            # condução implícita entre camadas (camadas que fundem ficam finas: explícito seria instável)
            G = self._conductances(geo)                       # [nx, n_l − 1]
            C = self._slice_view(self.kin.heat_capacity(self.T))
            S = -self._slice_view(heat) / dt
            S[:, 0] += Q_wall
            S[:, -1] += Q_top
            self.T = self._implicit_layers(self._slice_view(self.T), C, G, S, dt).reshape(-1)
        # ---- umidade: platô a 100 °C
        mw = self.kin.mass("H2O")
        over = (self.T > 373.15) & (mw > 0)
        Cn = self.kin.heat_capacity(self.T)
        evap = np.where(over, np.minimum(mw, Cn * (self.T - 373.15) / fd.COMPONENTS["H2O"].dH), 0.0)
        self.T = np.where(over, self.T - evap * fd.COMPONENTS["H2O"].dH / np.maximum(Cn, 1e-6), self.T)
        m0w = self.kin.m0["H2O"]
        self.kin.p["H2O"] = np.where(m0w > 0, np.clip(1 - (mw - evap) / np.maximum(m0w, 1e-30), 0, 1), 1.0)
        prod["H2O"] = prod["H2O"] + evap
        # ---- casco
        self.Ts = self.Ts + dTs
        # ---- espaço livre: vapores da fatia k, pistonado até a saída
        gen = {k: self._slice_view(v).sum(1) / dt for k, v in prod.items()}    # kg/s por fatia
        gen_tot = sum(gen.values())
        flow = {k: 0.0 for k in fd.LUMPS + ["N2"]}
        flow["N2"] = b.purge
        Tg_in = T0
        A_exp = (exposed * self.A_sec).sum(1)
        Tw_exp = (exposed * self.A_sec * self.Ts).sum(1) / np.maximum(A_exp, 1e-9)
        Vfb = math.pi * self.R ** 2 * self.dx * (1 - geo["F"])
        coke = 0.0
        Q_gas_heat = 0.0
        for k in range(nx):
            m_in = sum(flow.values())
            hA_w, hA_b = b.h_fb * A_exp[k], b.h_fb * A_surf[k]
            mcp_in, mcp_gen = m_in * CP_VAP, gen_tot[k] * CP_VAP
            Tg = (mcp_in * Tg_in + mcp_gen * T_top[k] + hA_w * Tw_exp[k] + hA_b * T_top[k]) / \
                (mcp_in + mcp_gen + hA_w + hA_b)
            self.Tg[k] = Tg
            for lump in fd.LUMPS:
                flow[lump] = flow[lump] + gen[lump][k]
            # residência na fatia e craqueamento
            mtot = sum(flow.values())
            Mmix = mtot / max(sum(flow[s] / MW_LUMP[s] for s in flow), 1e-12)
            rho = 101325.0 * Mmix / (8.314 * Tg)
            tau = Vfb[k] * rho / max(mtot, 1e-9)
            kO = K_O[0] * math.exp(-K_O[1] / (8.314 * Tg))
            dO = flow["O"] * (1 - math.exp(-kO * tau))
            dW = flow["W"] * (1 - math.exp(-K_W_FACTOR * kO * tau))
            flow["W"] -= dW
            flow["O"] += dW - dO
            for p_, y in O_CRACK.items():
                if p_ == "COKE":
                    coke += dO * y
                else:
                    flow[p_] += dO * y
            k1 = K_TAR_GAS[0] * math.exp(-K_TAR_GAS[1] / (8.314 * Tg))
            k2 = K_TAR_CHAR[0] * math.exp(-K_TAR_CHAR[1] / (8.314 * Tg))
            dT_ = flow["TAR"] * (1 - math.exp(-(k1 + k2) * tau))
            flow["TAR"] -= dT_
            flow["G_OX"] += dT_ * k1 / (k1 + k2)
            coke += dT_ * k2 / (k1 + k2)
            Tg_in = Tg
        Q_gas_heat = (sum(flow.values()) - b.purge) * CP_VAP * (Tg_in - T0)
        self.T_vap_out = Tg_in
        # ---- condensador e reciclo do gás
        oil = sum(flow[k] for k in fd.CONDENSABLE)
        aq = flow["H2O"] + flow["HCl"]
        gas = flow["G_HC"] + flow["G_OX"]
        self.ncg_rate = {"G_HC": flow["G_HC"], "G_OX": flow["G_OX"]}
        for k in fd.LUMPS:
            self.prod[k] += flow[k] * dt
        self.prod["COKE"] += coke * dt
        self.kin.add("CHAR", np.full(self.kin.n, coke * dt / self.kin.n))   # o coque fica no tambor
        cu = self.cum
        cu["oleo"] += oil * dt; cu["aquoso"] += aq * dt; cu["gas"] += gas * dt; cu["coque"] += coke * dt
        cu["Q_fire"] += self.Q_fire * dt
        cu["Q_ncg"] += Q_ncg_used * dt; cu["Q_aux"] += (self.Q_fire - Q_ncg_used) * dt
        cu["Q_flare"] += Q_flare * dt
        cu["aux_kg"] += use.get(self.fur.aux_fuel, 0.0) * dt
        cu["Q_shell"] += fdiag["Q_shell"] * dt; cu["Q_flue"] += fdiag["Q_flue"] * dt
        cu["Q_loss"] += fdiag["Q_loss"] * dt
        cu["Q_in_drum"] += float((q_in * self.A_sec).sum()) * dt
        cu["Q_rxn"] += float(heat.sum()); cu["Q_evap"] += float(evap.sum()) * fd.COMPONENTS["H2O"].dH
        cu["Q_vap_out"] += Q_gas_heat * dt
        # ---- controle da queima (PI com anti-windup e limite do casco)
        T_ctrl = float(self.Ts.mean()) if cn.variable == "parede" else float(self.T.mean())
        T_sp = min(cn.setpoint_C + 273.15, T0 + 40.0 + cn.ramp_K_min * self.t / 60.0) if cn.ramp_K_min > 0 \
            else cn.setpoint_C + 273.15
        err = T_sp - T_ctrl
        Ts_max = float(self.Ts.max())
        if Ts_max > cn.T_shell_max_C + 273.15:
            err = min(err, (cn.T_shell_max_C + 273.15 - Ts_max) * 3.0)
        Q_try = cn.Kp * (err + self.I_err / cn.Ti)
        if self.fur.Q_min < Q_try < self.fur.Q_max or (Q_try >= self.fur.Q_max and err < 0) or \
                (Q_try <= self.fur.Q_min and err > 0):
            self.I_err += err * dt
        self.Q_fire = float(np.clip(cn.Kp * (err + self.I_err / cn.Ti), self.fur.Q_min, self.fur.Q_max))
        # ---- avanço do tempo
        self.t += dt
        self.theta += self.Omega * dt
        rate = oil + gas
        self.peak_rate = max(self.peak_rate, rate)
        self._rx = {k: float(np.sum(v)) for k, v in rx_rates.items()}
        self._rx["H2O"] = float(np.sum(evap)) / dt
        self._last = dict(geo=geo, q_f=q_f, q_in=q_in, cov=cov, fdiag=fdiag, h_wb=h_wb, rates=dict(oil=oil, aq=aq,
                          gas=gas, coke=coke), use=use, Q_ncg=Q_ncg_used, Q_flare=Q_flare, Q_rad_bed=Q_rad_bed,
                          Q_wall=Q_wall, Q_curt=Q_curt, phi_lab=phi_wrapped)
        if (self.t > 3600 and rate < c.end_frac * self.peak_rate and self.peak_rate > 0
                and float((self.kin.total() * self.T).sum() / max(self.kin.total().sum(), 1e-9)) > 673.15):
            self.done = True

    # ---------------------------------------------- pilha estática: camadas
    def _layer_geom(self, geo):
        """Espessura e áreas das interfaces das camadas (da parede para a superfície), por fatia."""
        c = self.cfg
        R = self.R
        eps = geo["eps"]
        S = 2 * eps * R                     # arco molhado
        W = 2 * R * np.sin(eps)             # superfície livre
        Aseg = R ** 2 * (eps - np.sin(eps) * np.cos(eps))
        Heq = 2 * Aseg / np.maximum(S + W, 1e-9)
        v = self._slice_view(self.kin.volume(self.T, self.loose_scale)) / self.dx       # área por camada
        v = v / np.maximum(v.sum(1, keepdims=True), 1e-12) * Aseg[:, None]
        acum = np.concatenate([np.zeros((c.nx, 1)), np.cumsum(v, 1)], 1)
        a = (W - S) / (2 * np.maximum(Heq, 1e-9))
        # S s + a s² = acum → s
        disc = np.sqrt(np.maximum(S[:, None] ** 2 + 4 * a[:, None] * acum, 0.0))
        s = np.where(np.abs(a[:, None]) > 1e-9, (-S[:, None] + disc) / (2 * a[:, None] + 1e-30),
                     acum / np.maximum(S[:, None], 1e-9))
        A_if = S[:, None] + 2 * a[:, None] * s
        return dict(t=np.diff(s, axis=1), s=s, A=A_if, S=S, W=W, Heq=Heq)

    @staticmethod
    def _implicit_layers(T, C, G, S, dt):
        """(C/dt)(T' − T) = G_{i−1}(T'_{i−1} − T'_i) + G_i(T'_{i+1} − T'_i) + S (Thomas vetorizado)."""
        nl = T.shape[1]
        a = np.zeros_like(T); b = C / dt; cc = np.zeros_like(T); d = C / dt * T + S
        b[:, :-1] += G; b[:, 1:] += G
        a[:, 1:] = -G; cc[:, :-1] = -G
        cp_, dp_ = np.zeros_like(T), np.zeros_like(T)
        cp_[:, 0] = cc[:, 0] / b[:, 0]; dp_[:, 0] = d[:, 0] / b[:, 0]
        for i in range(1, nl):
            den = b[:, i] - a[:, i] * cp_[:, i - 1]
            cp_[:, i] = cc[:, i] / den
            dp_[:, i] = (d[:, i] - a[:, i] * dp_[:, i - 1]) / den
        x = np.zeros_like(T)
        x[:, -1] = dp_[:, -1]
        for i in range(nl - 2, -1, -1):
            x[:, i] = dp_[:, i] - cp_[:, i] * x[:, i + 1]
        return x

    def _conductances(self, geo):
        c = self.cfg
        lg = self._layer_geom(geo)
        lo, me, re = geo["frac"]
        d_pore = lo * c.bed.d_pore[0] + me * c.bed.d_pore[1] + re * c.bed.d_pore[2]
        k = self._slice_view(self.kin.conductivity(self.T, d_piece=np.repeat(d_pore, c.n_layers)))
        t = np.maximum(lg["t"], 1e-5)
        kf = 2 * k[:, 1:] * k[:, :-1] / (k[:, 1:] + k[:, :-1])
        return lg["A"][:, 1:-1] * self.dx * kf / (0.5 * (t[:, 1:] + t[:, :-1]))

    def _conduction(self, geo):
        c = self.cfg
        lg = self._layer_geom(geo)
        lo, me, re = geo["frac"]
        d_pore = lo * c.bed.d_pore[0] + me * c.bed.d_pore[1] + re * c.bed.d_pore[2]
        k = self._slice_view(self.kin.conductivity(self.T, d_piece=np.repeat(d_pore, c.n_layers)))
        T = self._slice_view(self.T)
        t = np.maximum(lg["t"], 1e-5)
        kf = 2 * k[:, 1:] * k[:, :-1] / (k[:, 1:] + k[:, :-1])
        G = lg["A"][:, 1:-1] * self.dx * kf / (0.5 * (t[:, 1:] + t[:, :-1]))
        flux = G * (T[:, :-1] - T[:, 1:])          # da camada i para i+1
        Q = np.zeros_like(T)
        Q[:, :-1] -= flux
        Q[:, 1:] += flux
        return Q.reshape(-1)

    # ---------------------------------------------------------------- saída
    def record(self):
        g = self._last["geo"]
        fd_ = self._last["fdiag"]
        r = self._last["rates"]
        mass = self.kin.total()
        T_mw = float((mass * self.T).sum() / max(mass.sum(), 1e-9))
        st = dict(t_h=self.t / 3600, T_carga=T_mw - 273.15,
                  T_carga_min=float(self.T.min()) - 273.15, T_carga_max=float(self.T.max()) - 273.15,
                  T_casco=float(self.Ts.mean()) - 273.15, T_casco_max=float(self.Ts.max()) - 273.15,
                  T_vapor=self.T_vap_out - 273.15, T_fumos=fd_["T_flue"] - 273.15,
                  T_camara=float(fd_["Tg"][:, 0].mean()) - 273.15,
                  enchimento=float(g["F"].mean()), Q_fire_kW=self.Q_fire / 1e3,
                  Q_ncg_kW=self._last["Q_ncg"] / 1e3, Q_aux_kW=(self.Q_fire - self._last["Q_ncg"]) / 1e3,
                  Q_flare_kW=self._last["Q_flare"] / 1e3, Q_casco_kW=fd_["Q_shell"] / 1e3,
                  Q_parede_carga_kW=float(self._last["Q_wall"].sum()) / 1e3,
                  Q_rad_carga_kW=float(self._last["Q_rad_bed"].sum() + self._last["Q_curt"].sum()) / 1e3,
                  h_wb=float(np.mean(self._last["h_wb"])),
                  oleo_kg=self.cum["oleo"], aquoso_kg=self.cum["aquoso"], gas_kg=self.cum["gas"],
                  coque_kg=self.cum["coque"], carvao_kg=float(np.sum(self.kin.mass("CHAR") + self.kin.mass("CB"))),
                  cinzas_kg=float(np.sum(self.kin.mass("INERT"))),
                  carga_kg=float(np.sum(self.kin.total())), agua_kg=float(np.sum(self.kin.mass("H2O"))),
                  vazao_oleo_kg_h=r["oil"] * 3600, vazao_gas_kg_h=r["gas"] * 3600,
                  frac_fundido=float(np.mean(g["frac"][1])), aux_kg=self.cum["aux_kg"],
                  HCl_kg=self.prod["HCl"])
        # taxa de reação (kg/h de componente consumido) e conversão de cada grupo: o "DTG" da batelada
        rx = getattr(self, "_rx", {})
        for g, keys in GROUPS.items():
            st[f"taxa_{g}"] = 3600.0 * sum(rx.get(k, 0.0) for k in keys)
            m0 = sum(self.m0_comp[k] for k in keys)
            st[f"conv_{g}"] = 1.0 - sum(float(np.sum(self.kin.mass(k))) for k in keys) / m0 if m0 > 0 else 1.0
        st["taxa_total"] = sum(st[f"taxa_{g}"] for g in GROUPS if g != "água")
        for k in fd.ORDER:
            st[f"m_{k}"] = float(np.sum(self.kin.mass(k)))
        self.log.append(st)
        return st

    def snapshot(self):
        g = self._last["geo"]
        snap = dict(t=self.t, theta=self.theta, Ts=self.Ts.copy(), T=self.T.copy(), F=g["F"].copy(),
                    eps=g["eps"].copy(), beta=g["beta"].copy(), frac=[a.copy() for a in g["frac"]],
                    q_f=self._last["q_f"].copy(), Tg_fur=self._last["fdiag"]["Tg"].copy(),
                    Tr_fur=self._last["fdiag"]["Tr"].copy(), Q_fire=self.Q_fire, Tg=self.Tg.copy(),
                    char=self.kin.mass("CHAR") + self.kin.mass("CB"), total=self.kin.total(),
                    vol=self.kin.volume(self.T, self.loose_scale))
        self.snaps.append(snap)
        return snap

    def run(self, verbose: bool = True, callback=None):
        c = self.cfg
        n_log = max(int(round(c.log_every / c.dt)), 1)
        n_snap = max(int(round(c.snap_every / c.dt)), 1)
        i = 0
        t0 = time.time()
        while self.t < c.t_max_h * 3600 and not self.done:
            self.step()
            i += 1
            if i % n_log == 0 or i == 1:
                st = self.record()
                if callback:
                    callback(self, st)
            if i % n_snap == 0 or i == 1:
                self.snapshot()
            if verbose and i % int(round(1800 / c.dt)) == 0:
                st = self.log[-1]
                print(f"  {st['t_h']:4.1f} h  carga {st['T_carga']:5.0f} °C  casco {st['T_casco']:5.0f} °C "
                      f"(máx {st['T_casco_max']:4.0f})  queima {st['Q_fire_kW']:4.0f} kW "
                      f"(gás próprio {st['Q_ncg_kW']:4.0f})  óleo {st['oleo_kg']:5.0f} kg  "
                      f"enchimento {100 * st['enchimento']:4.1f} %  [{time.time() - t0:4.0f} s]", flush=True)
        self.record(); self.snapshot()
        return self.summary()

    def summary(self) -> dict:
        m0 = self.m_charge0
        cu = self.cum
        char = float(np.sum(self.kin.mass("CHAR") + self.kin.mass("CB")))
        ash = float(np.sum(self.kin.mass("INERT")))
        left = float(np.sum(self.kin.total())) - char - ash
        out = dict(duracao_h=self.t / 3600, oleo=cu["oleo"] / m0, aquoso=cu["aquoso"] / m0, gas=cu["gas"] / m0,
                   carvao=char / m0, cinzas=ash / m0, nao_convertido=left / m0, coque_kg=cu["coque"],
                   oleo_kg=cu["oleo"], gas_kg=cu["gas"], carvao_kg=char, cinzas_kg=ash)
        out["fecho_massa"] = (cu["oleo"] + cu["aquoso"] + cu["gas"] + float(np.sum(self.kin.total())) -
                              cu["coque"] - m0) / m0
        out["Q_queima_GJ"] = cu["Q_fire"] / 1e9
        out["Q_aux_GJ"] = cu["Q_aux"] / 1e9
        out["aux_kg"] = cu["aux_kg"]
        out["aux_GJ_por_t"] = cu["Q_aux"] / 1e9 / (m0 / 1000)
        out["Q_gas_proprio_GJ"] = cu["Q_ncg"] / 1e9
        out["Q_flare_GJ"] = cu["Q_flare"] / 1e9
        out["eficiencia_fornalha"] = cu["Q_shell"] / max(cu["Q_fire"], 1)
        # balanço de energia da fornalha e do tambor
        out["fecho_fornalha"] = (cu["Q_fire"] - cu["Q_shell"] - cu["Q_flue"] - cu["Q_loss"]) / max(cu["Q_fire"], 1)
        dE_shell = self._shell_energy() - self.E_shell0
        out["fecho_casco"] = (cu["Q_shell"] - cu["Q_in_drum"] - dE_shell) / max(cu["Q_shell"], 1)
        T_end = self.T
        return out

In [ ]:
%%writefile tamborkit/dem.py
"""Carga do tambor por elementos discretos (DEM) na GPU.

Esferas macias (mola-amortecedor linear na normal; mola tangencial com histórico e limite de Coulomb,
Cundall & Strack 1979; resistência ao rolamento de torque constante, modelo A de Ai et al. 2011) num tambor
que gira com aletas. As paredes (casco, tampas e aletas) entram como superfícies analíticas que giram com o
tambor; a velocidade da parede no ponto de contato é Ω x̂ × p.

O objetivo não é seguir 6–8 h de batelada (impossível em DEM), e sim:
  * mostrar o movimento real da carga (leito rolante, cortinas das aletas) para o vídeo;
  * medir as grandezas que o modelo térmico da batelada precisa: ângulo dinâmico, arco de parede coberto,
    tempo de contato com a parede (modelo de penetração), fração em queda livre nas cortinas, espessura
    da camada ativa, taxa de mistura (índice de Lacey) e torque/potência de acionamento.

Rigidez reduzida: k_n é escolhido para uma sobreposição de ~1,5 % do diâmetro no fundo do leito, prática
usual para leitos grandes (Lommen et al. 2014); o regime transversal depende de Fr, do atrito e do
enchimento, não de k_n, enquanto a sobreposição for pequena.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field

import numpy as np
import torch

from .geometry import Drum, bed_depth, lifter_contacts

G = 9.81


@dataclass
class DEMConfig:
    fill: float = 0.30           # fração do volume interno ocupada pela carga (volume aparente)
    d_mean: float = 0.09         # diâmetro médio dos pedaços [m]
    d_spread: float = 0.30       # diâmetros uniformes em d_mean·(1 ± d_spread)
    rho_bulk: float = 300.0      # massa específica aparente do leito [kg/m³]
    phi_pack: float = 0.58       # fração de sólidos do empacotamento aleatório de esferas
    e_n: float = 0.30            # coeficiente de restituição (pedaços de plástico e madeira: baixo)
    mu: float = 0.55             # atrito de deslizamento partícula–partícula
    mu_wall: float = 0.45        # atrito partícula–aço
    mu_r: float = 0.20           # resistência ao rolamento (adimensional, braço = μ_r·r)
    overlap: float = 0.010       # sobreposição alvo no fundo do leito / d (define k_n)
    kt_kn: float = 2.0 / 7.0     # razão das rigidezes tangencial/normal
    cohesion: float = 0.0        # força coesiva máxima / peso médio da partícula (fundido pegajoso)
    slab: float | None = None    # comprimento de uma fatia periódica em x [m]; None = tambor inteiro
    skin: float = 0.30           # pele da lista de vizinhos (fração de d_max)
    k_nb: int = 20               # vizinhos guardados por partícula
    dt_frac: float = 1.0 / 18.0  # Δt = t_contato·dt_frac
    check_every: int = 10        # passos entre verificações de reconstrução da lista
    device: str = "auto"
    seed: int = 0
    compile: bool = False        # torch.compile do passo (GPU); cai para o modo normal se falhar
    contact_time: float | None = None   # se dado: rigidez por par k = m_eff (π/t_c)²/(1 − β²) — duração do
    #                                     contato fixa, independente do tamanho (pedaços que encolhem/fundem)


@dataclass
class DEMStats:
    t: float = 0.0
    revs: float = 0.0
    beta_deg: float = float("nan")         # ângulo dinâmico do leito
    covered_frac: float = float("nan")     # fração do perímetro do casco coberta pela carga
    wall_contact_s: float = float("nan")   # duração média de um contato partícula–casco
    passive_contact_s: float = float("nan")  # tempo de passagem do leito sobre a parede: 2ε/Ω
    airborne_frac: float = float("nan")    # fração em queda livre (cortinas)
    active_layer_m: float = float("nan")   # espessura da camada ativa
    lacey: float = float("nan")            # índice de mistura (0 segregado, 1 aleatório)
    torque_Nm: float = float("nan")        # torque de acionamento médio
    power_W: float = float("nan")
    max_overlap: float = float("nan")      # maior sobreposição / d

    def as_dict(self):
        return dict(self.__dict__)


class DrumDEM:
    def __init__(self, drum: Drum, cfg: DEMConfig, labels_frac: dict | None = None, init: dict | None = None):
        self.drum, self.cfg = drum, cfg
        dev = cfg.device
        if dev == "auto":
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = torch.device(dev)
        self.ft = torch.float32
        gen = np.random.default_rng(cfg.seed)
        self.periodic = cfg.slab is not None
        self.Lx = cfg.slab if self.periodic else drum.L

        # ---- partículas: tamanhos, massas e posições iniciais numa rede acima do leito final
        t = lambda a: torch.as_tensor(np.asarray(a), dtype=self.ft, device=self.dev)  # noqa: E731
        d_lo, d_hi = cfg.d_mean * (1 - cfg.d_spread), cfg.d_mean * (1 + cfg.d_spread)
        Vmean = math.pi / 6 * (d_hi ** 4 - d_lo ** 4) / (4 * (d_hi - d_lo)) if d_hi > d_lo else math.pi / 6 * cfg.d_mean ** 3
        A = math.pi * drum.R ** 2
        if init is not None:          # partículas dadas (DEM com mudança de fase): posições, diâmetros, densidades
            pts, d = np.asarray(init["x"], float), np.asarray(init["d"], float)
            N = len(pts)
            d_hi = float(d.max())
            self.rho_p = t(init["rho"])
            grow0 = init.get("grow0", 0.5)
            active = np.asarray(init.get("active", np.ones(N, bool)))
        else:
            N = int(round(cfg.phi_pack * cfg.fill * A * self.Lx / Vmean))
            pts = self._lattice(N, gen)
            N = len(pts)
            d = gen.uniform(d_lo, d_hi, N)
            self.rho_p = cfg.rho_bulk / cfg.phi_pack
            grow0 = min(1.0, cfg.d_mean / d_hi)
            active = np.ones(N, bool)
        self.d_max = d_hi
        self.N = N
        self.x = t(pts)
        self.v = torch.zeros_like(self.x)
        self.w = torch.zeros_like(self.x)
        self.r_final = t(0.5 * d)
        self.grow0 = grow0
        self.r = self.r_final * self.grow0
        self.active = t(active.astype(np.float32))
        self.stick = torch.zeros(N, dtype=self.ft, device=self.dev)   # pegajosidade (0–1) de cada partícula
        self._set_mass()
        # rótulos de material (cores) e de mistura (lado inicial)
        if init is not None:
            self.label_names = list(init["names"])
            self.label = torch.as_tensor(np.asarray(init["label"]), device=self.dev, dtype=torch.long)
        elif labels_frac:
            keys = list(labels_frac)
            p = np.array([labels_frac[k] for k in keys], float); p /= p.sum()
            self.label_names = keys
            self.label = torch.as_tensor(gen.choice(len(keys), N, p=p), device=self.dev)
        else:
            self.label_names = ["carga"]
            self.label = torch.zeros(N, dtype=torch.long, device=self.dev)

        # ---- contato: rigidez para a sobreposição alvo no fundo do leito (ou duração de contato fixa)
        lne = math.log(max(cfg.e_n, 1e-4))
        self.beta_n = -lne / math.sqrt(math.pi ** 2 + lne ** 2)
        self.tc_mode = cfg.contact_time is not None
        if self.tc_mode:
            t_c = cfg.contact_time
            self.w2 = (math.pi / t_c) ** 2 / (1 - self.beta_n ** 2)
            self.kn = self.kt = None
            self.f_coh = 0.0
        else:
            H = bed_depth(cfg.fill, drum.R)
            self.kn = cfg.rho_bulk * G * H * cfg.d_mean / cfg.overlap
            self.kt = cfg.kt_kn * self.kn
            m_mean = float(np.mean(np.asarray(self.rho_p.cpu() if torch.is_tensor(self.rho_p) else self.rho_p))) * Vmean
            t_c = math.pi * math.sqrt(0.5 * d_lo ** 3 / cfg.d_mean ** 3 * m_mean / self.kn)
            self.f_coh = cfg.cohesion * m_mean * G
        self.t_contact = t_c
        self.dt = cfg.dt_frac * t_c
        self.t, self.theta = 0.0, 0.0
        self.omega = 0.0          # começa parado (assentamento)
        self.damping = 0.0        # amortecimento global (só no assentamento)
        self.steps = 0
        # ---- listas de vizinhos e históricos tangenciais
        self.cell = self.d_max * (1 + cfg.skin)
        self.lo = t([0.0, -drum.R - self.cell, -drum.R - self.cell])
        hi = np.array([self.Lx, drum.R + self.cell, drum.R + self.cell])
        self.dims = np.maximum(np.ceil((hi - self.lo.cpu().numpy()) / self.cell).astype(int), 1)
        csize = np.full(3, self.cell)
        if self.periodic:   # células inteiras ao longo da fatia periódica (sem célula parcial na emenda)
            self.dims[0] = int(self.Lx // self.cell)
            if self.dims[0] < 3:
                raise ValueError("fatia periódica curta demais para a lista de vizinhos (mín. 3 células)")
            csize[0] = self.Lx / self.dims[0]
        self.csize = t(csize)
        self.nwall = 1 + (0 if self.periodic else 2) + drum.n_lifters * len(drum.lifter_boxes())
        self.xi = torch.zeros(N, cfg.k_nb, 3, device=self.dev)
        self.xi_w = torch.zeros(N, self.nwall, 3, device=self.dev)
        self._rebuild()
        # ---- estatísticas
        self.reset_stats()
        self._step_fn = self._step
        if cfg.compile and self.dev.type == "cuda":
            try:
                self._step_fn = torch.compile(self._step, dynamic=False)
            except Exception:   # noqa: BLE001
                self._step_fn = self._step

    # ------------------------------------------------------------ montagem
    def _set_mass(self):
        V = 4.0 / 3.0 * math.pi * self.r ** 3
        self.m = self.rho_p * V
        self.I = 0.4 * self.m * self.r ** 2

    def _lattice(self, N, gen):
        """Rede cúbica dentro do tambor, longe das aletas, preenchida de baixo para cima."""
        dr, a = self.drum, self.cfg.d_mean * 1.03     # rede no diâmetro médio: os raios crescem no assentamento
        nx = max(int(self.Lx / a), 1)
        xs = (np.arange(nx) + 0.5) * self.Lx / nx
        ys = np.arange(-dr.R + a, dr.R - a / 2, a)
        X, Y, Z = np.meshgrid(xs, ys, ys, indexing="ij")
        P = np.stack([X.ravel(), Y.ravel(), Z.ravel()], 1)
        P = P[np.hypot(P[:, 1], P[:, 2]) < dr.R - 0.55 * a]
        if dr.n_lifters:
            dist, _, _ = lifter_contacts(dr, torch.as_tensor(P, dtype=torch.float64), 0.0,
                                         self.Lx if self.periodic else None)
            P = P[(dist.min(1).values > 0.6 * a).numpy()]
        P = P[np.argsort(P[:, 2] + 1e-3 * gen.random(len(P)), kind="stable")][:N]
        P += gen.uniform(-0.03, 0.03, P.shape) * a
        return P

    # ------------------------------------------------------------ vizinhança
    def _wrap(self, dx):
        if self.periodic:
            dx = dx.clone()
            dx[..., 0] -= self.Lx * torch.round(dx[..., 0] / self.Lx)
        return dx

    def _rebuild(self):
        x, c, dims = self.x, self.cell, self.dims
        idx = ((x - self.lo) / self.csize).floor().long()
        for a in range(3):
            idx[:, a] = idx[:, a].clamp(0, int(dims[a]) - 1)
        cid = (idx[:, 0] * int(dims[1]) + idx[:, 1]) * int(dims[2]) + idx[:, 2]
        order = torch.argsort(cid)
        cs = cid[order]
        start = torch.searchsorted(cs, cs)
        rank = torch.arange(self.N, device=self.dev) - start
        Kc = int(rank.max().item()) + 1
        ncell = int(np.prod(dims))
        table = torch.full((ncell, Kc), -1, dtype=torch.long, device=self.dev)
        table[cs, rank] = order
        off = torch.stack(torch.meshgrid(*[torch.arange(-1, 2, device=self.dev)] * 3, indexing="ij"), -1).reshape(-1, 3)
        nid = idx[:, None, :] + off[None]
        ok = torch.ones(nid.shape[:2], dtype=torch.bool, device=self.dev)
        if self.periodic:
            nid[..., 0] = torch.remainder(nid[..., 0], int(dims[0]))
        for a in range(3):
            ok &= (nid[..., a] >= 0) & (nid[..., a] < int(dims[a]))
        ncid = (nid[..., 0].clamp(0, int(dims[0]) - 1) * int(dims[1]) + nid[..., 1].clamp(0, int(dims[1]) - 1)) \
            * int(dims[2]) + nid[..., 2].clamp(0, int(dims[2]) - 1)
        cand = table[ncid]                                   # [N, 27, Kc]
        cand = torch.where(ok[..., None], cand, torch.full_like(cand, -1)).reshape(self.N, -1)
        j = cand.clamp(min=0)
        dx = self._wrap(x[j] - x[:, None])
        dist = dx.norm(dim=-1)
        ar = torch.arange(self.N, device=self.dev)[:, None]
        valid = (cand >= 0) & (cand != ar) & (dist < self.r[:, None] + self.r[j] + self.cfg.skin * self.d_max)
        valid &= (self.active[:, None] > 0) & (self.active[j] > 0)
        key = torch.where(valid, dist, torch.full_like(dist, float("inf")))
        K = self.cfg.k_nb
        nmax = int(valid.sum(1).max().item())
        if nmax > K:      # cresce a lista (e o histórico) em vez de perder contatos
            grow = nmax + 4 - K
            K = self.cfg.k_nb = nmax + 4
            if hasattr(self, "nb"):
                self.nb = torch.cat([self.nb, torch.full((self.N, grow), -1, dtype=self.nb.dtype, device=self.dev)], 1)
            self.xi = torch.cat([self.xi, torch.zeros(self.N, grow, 3, device=self.dev)], 1)
        val, pos = torch.topk(key, K, dim=1, largest=False)
        nb = torch.gather(cand, 1, pos)
        nb = torch.where(torch.isfinite(val), nb, torch.full_like(nb, -1))
        # transfere o histórico tangencial dos pares que continuam na lista
        if hasattr(self, "nb"):
            same = nb[:, :, None] == self.nb[:, None, :]
            same &= nb[:, :, None] >= 0
            has = same.any(-1)
            src = same.float().argmax(-1)
            xi = torch.gather(self.xi, 1, src[..., None].expand(-1, -1, 3))
            self.xi = torch.where(has[..., None], xi, torch.zeros_like(xi))
        self.nb = nb
        self.x_ref = x.clone()

    # ------------------------------------------------------------ contato
    def _law(self, delta, n, u, m_eff, xi, kn, kt, mu, rad, dt, f_coh):
        """Força na partícula i (normal + tangencial), torque tangencial e novo histórico.

        delta: sobreposição; n: normal unitária de i para o outro corpo; u: velocidade do ponto de contato
        de i relativa ao outro corpo; rad: braço (raio de i até o ponto de contato)."""
        touch = delta > 0
        un = (u * n).sum(-1)
        gn = 2 * self.beta_n * torch.sqrt(m_eff * kn)
        fn = kn * delta + gn * un
        fn = torch.where(touch, torch.maximum(fn, -f_coh * torch.ones_like(fn)), torch.zeros_like(fn))
        ut = u - un[..., None] * n
        xi = xi - (xi * n).sum(-1, keepdim=True) * n + ut * dt
        gt = 2 * self.beta_n * torch.sqrt(m_eff * kt)
        ktv = kt[..., None] if torch.is_tensor(kt) else kt
        ft = -ktv * xi - gt[..., None] * ut
        fmax = mu * (fn + f_coh).clamp(min=0)
        ftn = ft.norm(dim=-1).clamp(min=1e-12)
        slip = ftn > fmax
        ft = torch.where(slip[..., None], ft * (fmax / ftn)[..., None], ft)
        xi = torch.where(slip[..., None], -ft / ktv, xi)
        xi = torch.where(touch[..., None], xi, torch.zeros_like(xi))
        ft = torch.where(touch[..., None], ft, torch.zeros_like(ft))
        F = -fn[..., None] * n + ft
        Mt = torch.cross(rad[..., None] * n, ft, dim=-1)
        return F, Mt, fn, xi

    def _walls(self, x, theta):
        """Corpos de parede: casco, tampas (se não periódico) e caixas das aletas → (dist, n, p_c)."""
        dr = self.drum
        rho = torch.hypot(x[:, 1], x[:, 2]).clamp(min=1e-9)
        ey, ez = x[:, 1] / rho, x[:, 2] / rho
        zero = torch.zeros_like(rho)
        dists = [dr.R - rho]
        ns = [torch.stack([zero, ey, ez], -1)]
        ps = [torch.stack([x[:, 0], dr.R * ey, dr.R * ez], -1)]
        if not self.periodic:
            dists += [x[:, 0], dr.L - x[:, 0]]
            ns += [torch.stack([-torch.ones_like(rho), zero, zero], -1), torch.stack([torch.ones_like(rho), zero, zero], -1)]
            ps += [torch.stack([zero, x[:, 1], x[:, 2]], -1), torch.stack([zero + dr.L, x[:, 1], x[:, 2]], -1)]
        dist = torch.stack(dists, 1)
        n = torch.stack(ns, 1)
        p = torch.stack(ps, 1)
        if dr.n_lifters:
            dl, nl, pl = lifter_contacts(dr, x, theta, self.Lx if self.periodic else None)
            dist, n, p = torch.cat([dist, dl], 1), torch.cat([n, nl], 1), torch.cat([p, pl], 1)
        return dist, n, p

    def _step(self, x, v, w, xi, xi_w, nb, theta, omega, damping, dt):
        r, m, I = self.r, self.m, self.I
        valid = nb >= 0
        j = nb.clamp(min=0)
        dx = x[j] - x[:, None]
        if self.periodic:
            dx = torch.cat([dx[..., :1] - self.Lx * torch.round(dx[..., :1] / self.Lx), dx[..., 1:]], -1)
        dist = dx.norm(dim=-1).clamp(min=1e-9)
        n = dx / dist[..., None]
        delta = torch.where(valid, r[:, None] + r[j] - dist, torch.full_like(dist, -1.0))
        ai = (r[:, None] - 0.5 * delta.clamp(min=0))
        aj = (r[j] - 0.5 * delta.clamp(min=0))
        u = (v[:, None] + torch.cross(w[:, None].expand_as(n), ai[..., None] * n, dim=-1)) \
            - (v[j] + torch.cross(w[j], -aj[..., None] * n, dim=-1))
        m_eff = m[:, None] * m[j] / (m[:, None] + m[j])
        if self.tc_mode:
            kn_p = m_eff * self.w2
            kt_p = self.cfg.kt_kn * kn_p
            fc_p = self.cfg.cohesion * G * m_eff * torch.minimum(self.stick[:, None], self.stick[j])
        else:
            kn_p, kt_p, fc_p = self.kn, self.kt, self.f_coh
        Fp, Mp, fnp, xi = self._law(delta, n, u, m_eff, xi, kn_p, kt_p, self.cfg.mu, ai, dt, fc_p)
        r_eff = r[:, None] * r[j] / (r[:, None] + r[j])
        wrel = w[:, None] - w[j]
        # paredes
        dw_, nw, pw = self._walls(x, theta)
        delw = torch.where(self.active[:, None] > 0, r[:, None] - dw_, torch.full_like(dw_, -1.0))
        aw = r[:, None] - 0.5 * delw.clamp(min=0)
        vwall = torch.stack([torch.zeros_like(pw[..., 0]), -omega * pw[..., 2], omega * pw[..., 1]], -1)
        uw = v[:, None] + torch.cross(w[:, None].expand_as(nw), aw[..., None] * nw, dim=-1) - vwall
        mw_ = m[:, None].expand_as(delw)
        if self.tc_mode:
            kn_w = mw_ * self.w2
            kt_w = self.cfg.kt_kn * kn_w
            fc_w = 0.5 * self.cfg.cohesion * G * mw_ * self.stick[:, None]
        else:
            kn_w, kt_w, fc_w = self.kn, self.kt, self.f_coh
        Fw, Mw, fnw, xi_w = self._law(delw, nw, uw, mw_, xi_w, kn_w, kt_w, self.cfg.mu_wall, aw, dt, fc_w)
        # resistência ao rolamento (torque constante, regularizado perto de ω_rel = 0)
        mr = self.cfg.mu_r

        def roll(wr, fn, re):
            mag = mr * re * fn.clamp(min=0)
            wn = wr.norm(dim=-1)
            w_c = (mag * dt / I[:, None]).clamp(min=1e-6)
            return -mag[..., None] * wr / torch.maximum(wn, w_c)[..., None]
        w_wall = torch.stack([w[:, 0] - omega, w[:, 1], w[:, 2]], -1)
        Mr = roll(wrel, fnp, r_eff).sum(1) + roll(w_wall[:, None].expand_as(nw), fnw, r[:, None].expand_as(fnw)).sum(1)
        F = Fp.sum(1) + Fw.sum(1)
        F = F + torch.stack([torch.zeros_like(m), torch.zeros_like(m), -m * G * self.active], -1)
        M = Mp.sum(1) + Mw.sum(1) + Mr
        F = F - damping * m[:, None] * v
        M = M - damping * I[:, None] * w
        v = v + F / m[:, None] * dt
        w = w + M / I[:, None] * dt
        x = x + v * dt
        if self.periodic:
            x = torch.cat([torch.remainder(x[:, :1], self.Lx), x[:, 1:]], -1)
        # torque das partículas sobre o tambor (em torno de x): parede recebe −Fw
        tq = (pw[..., 1] * Fw[..., 2] - pw[..., 2] * Fw[..., 1]).sum()   # torque que o motor precisa dar
        ncont = (delta > 0).sum(1) + (delw > 0).sum(1)
        ovl = torch.maximum(delta.max(), delw.max())
        near_shell = delw[:, 0] > -0.1 * 2 * r     # junto ao casco (folga < 0,1 d): contato térmico
        return x, v, w, xi, xi_w, tq, ncont, near_shell, ovl

    # ------------------------------------------------------------ marcha
    def reset_stats(self):
        self._acc = dict(n=0, torque=0.0, air=0.0, covered=0.0, ovl=0.0, ep_sum=0.0, ep_n=0)
        self._contact_t0 = torch.full((self.N,), -1.0, device=self.dev)
        self._free = torch.zeros(self.N, device=self.dev)

    def step(self, nsteps: int = 1):
        dt = self.dt
        for _ in range(nsteps):
            sc = lambda a: torch.tensor(a, dtype=self.ft, device=self.dev)  # noqa: E731
            (self.x, self.v, self.w, self.xi, self.xi_w, tq, ncont, shell, ovl) = self._step_fn(
                self.x, self.v, self.w, self.xi, self.xi_w, self.nb, sc(self.theta), sc(self.omega),
                sc(self.damping), dt)
            self.t += dt
            self.theta += self.omega * dt
            self.steps += 1
            self._post_step(dt)
            # estatísticas (sem sincronizar com a CPU)
            a = self._acc
            a["n"] += 1
            a["torque"] = a["torque"] + tq
            self._free = torch.where(ncont == 0, self._free + dt, torch.zeros_like(self._free))
            a["air"] = a["air"] + ((self._free > 4 * self.t_contact).float() * self.active).sum() / self.active.sum().clamp(min=1)
            a["ovl"] = torch.maximum(torch.as_tensor(a["ovl"], device=self.dev), ovl)
            started = shell & (self._contact_t0 < 0)
            ended = (~shell) & (self._contact_t0 >= 0)
            dur = (self.t - self._contact_t0)
            a["ep_sum"] = a["ep_sum"] + torch.where(ended, dur, torch.zeros_like(dur)).sum()
            a["ep_n"] = a["ep_n"] + ended.sum()
            self._contact_t0 = torch.where(started, torch.full_like(dur, self.t),
                                           torch.where(ended, torch.full_like(dur, -1.0), self._contact_t0))
            if self.steps % self.cfg.check_every == 0:
                disp = self._wrap(self.x - self.x_ref).norm(dim=-1).max()
                if float(disp) > 0.5 * self.cfg.skin * self.d_max:
                    self._rebuild()

    def _post_step(self, dt):
        """Gancho para forças externas aplicadas após a integração (poça de fundido no DEM com mudança de fase)."""
        return None

    def settle(self, seconds: float = 2.0, damping: float = 4.0):
        """Assentamento com o tambor parado e amortecimento global; os raios crescem até o valor final
        nos primeiros 60 % (a rede inicial usa o diâmetro médio, sem sobreposição)."""
        om, self.omega, self.damping = self.omega, 0.0, damping
        for i in range(10):
            f = self.grow0 + (1 - self.grow0) * min(1.0, (i + 1) / 6)
            self.r = self.r_final * f
            self._set_mass()
            self.run(seconds / 10)
        self.omega, self.damping = om, 0.0
        self.v.zero_(); self.w.zero_()
        self.reset_stats()

    def spin_up(self, rpm: float | None = None):
        self.omega = self.drum.omega if rpm is None else 2 * math.pi * rpm / 60

    def run(self, seconds: float, frame_dt: float | None = None, on_frame=None, verbose: bool = False):
        n = max(int(round(seconds / self.dt)), 1)
        every = max(int(round(frame_dt / self.dt)), 1) if frame_dt else n + 1
        t0, done = time.time(), 0
        while done < n:
            k = min(every - (self.steps % every) if frame_dt else n - done, n - done)
            self.step(k)
            done += k
            if frame_dt and self.steps % every == 0 and on_frame is not None:
                on_frame(self)
            if verbose and (done == n or done % max(n // 10, 1) < k):
                print(f"  DEM t = {self.t:7.2f} s  ({done}/{n} passos, {time.time() - t0:5.1f} s de relógio)", flush=True)

    # ------------------------------------------------------------ diagnóstico
    def frame(self) -> dict:
        """Estado leve para renderização (float16 na CPU)."""
        return dict(t=self.t, theta=self.theta, x=self.x.half().cpu().numpy(),
                    speed=self.v.norm(dim=-1).half().cpu().numpy(), r=self.r.half().cpu().numpy(),
                    active=self.active.cpu().numpy() > 0)

    def surface_angle(self):
        """Ângulo dinâmico: reta ajustada ao topo do leito no miolo da corda (partículas fora das aletas)."""
        x = self.x.cpu().numpy().astype(float)
        dr = self.drum
        rho = np.hypot(x[:, 1], x[:, 2])
        inner = rho < dr.R - dr.lifter_h - self.d_max if dr.n_lifters else rho < dr.R
        inner &= self.active.cpu().numpy() > 0              # partículas reservadas (inativas) não contam
        y, z = x[inner, 1], x[inner, 2]
        edges = np.linspace(-dr.R, dr.R, 41)
        k = np.digitize(y, edges)
        yc, zt = [], []
        for b in range(1, len(edges)):
            zz = z[k == b]
            if len(zz) >= 5:
                yc.append(0.5 * (edges[b - 1] + edges[b])); zt.append(np.sort(zz)[-3:].mean())
        yc, zt = np.array(yc), np.array(zt)
        if len(yc) < 6:
            return float("nan"), None
        lo, hi = np.quantile(yc, [0.25, 0.75])
        sel = (yc >= lo) & (yc <= hi)
        b, a = np.polyfit(yc[sel], zt[sel], 1)
        return float(np.degrees(np.arctan(b))), (a, b)

    def active_layer(self, line):
        """Espessura da camada ativa: profundidade, abaixo da superfície, onde a velocidade ao longo da
        encosta troca de sinal (desce na camada ativa, sobe com a parede no leito passivo)."""
        if line is None:
            return float("nan")
        a, b = line
        x = self.x.cpu().numpy().astype(float)
        v = self.v.cpu().numpy().astype(float)
        t = np.array([1.0, b]) / math.hypot(1.0, b)        # direção da encosta (sobe para +y se b > 0)
        nrm = np.array([-t[1], t[0]])                        # normal para cima
        yz = x[:, 1:]
        depth = (np.array([0.0, a]) - yz) @ nrm              # >0 abaixo da superfície
        s = yz @ t
        mid = np.abs(s) < 0.25 * self.drum.R
        vs = v[:, 1:] @ t
        bins = np.arange(0, 0.8 * self.drum.R, 0.5 * self.cfg.d_mean)
        prof = []
        for lo, hi in zip(bins[:-1], bins[1:]):
            m = mid & (depth >= lo) & (depth < hi)
            prof.append(np.mean(vs[m]) if m.sum() > 3 else np.nan)
        prof = np.array(prof)
        down = -np.sign(b) if b != 0 else -1.0                # a carga desce na direção −sinal(b)
        flow = prof * down
        for i in range(1, len(flow)):
            if np.isfinite(flow[i]) and np.isfinite(flow[i - 1]) and flow[i - 1] > 0 >= flow[i]:
                return float(bins[i - 1] + (bins[i] - bins[i - 1]) * flow[i - 1] / (flow[i - 1] - flow[i]))
        return float("nan")

    def lacey(self, labels: np.ndarray, ncell: int = 12) -> float:
        """Índice de Lacey na seção transversal (amostras numa grade y–z)."""
        x = self.x.cpu().numpy()
        p = labels.mean()
        R = self.drum.R
        iy = np.clip(((x[:, 1] + R) / (2 * R) * ncell).astype(int), 0, ncell - 1)
        iz = np.clip(((x[:, 2] + R) / (2 * R) * ncell).astype(int), 0, ncell - 1)
        key = iy * ncell + iz
        cnt = np.bincount(key, minlength=ncell * ncell)
        s1 = np.bincount(key, weights=labels, minlength=ncell * ncell)
        ok = cnt >= 8
        if ok.sum() < 3 or p in (0.0, 1.0):
            return float("nan")
        frac = s1[ok] / cnt[ok]
        S2 = np.average((frac - p) ** 2, weights=cnt[ok])
        S0 = p * (1 - p)
        Sr = S0 / cnt[ok].mean()
        return float(np.clip((S0 - S2) / (S0 - Sr), 0.0, 1.0))

    def stats(self) -> DEMStats:
        a = self._acc
        n = max(a["n"], 1)
        st = DEMStats(t=self.t, revs=self.theta / (2 * math.pi))
        beta, line = self.surface_angle()
        st.beta_deg = beta
        st.active_layer_m = self.active_layer(line)
        st.torque_Nm = float(a["torque"]) / n if a["n"] else float("nan")
        st.power_W = st.torque_Nm * self.omega
        st.airborne_frac = float(a["air"]) / n if a["n"] else float("nan")
        st.max_overlap = float(a["ovl"]) / self.cfg.d_mean
        st.wall_contact_s = float(a["ep_sum"]) / max(int(a["ep_n"]), 1) if int(a["ep_n"]) else float("nan")
        # arco coberto: setores de 10° com ao menos uma partícula encostada no casco (folga < 2 cm)
        x = self.x.cpu().numpy()
        r = self.r.cpu().numpy()
        rho = np.hypot(x[:, 1], x[:, 2])
        near = (rho > self.drum.R - r - 0.02) & (self.active.cpu().numpy() > 0)
        if near.any():
            phi = np.arctan2(x[near, 2], x[near, 1])
            h, _ = np.histogram(phi, bins=36, range=(-math.pi, math.pi))
            st.covered_frac = float((h >= 1).mean())
            if self.omega > 0:
                st.passive_contact_s = st.covered_frac * 2 * math.pi / self.omega
        return st

    def kinetic_energy(self) -> float:
        return float(0.5 * (self.m * (self.v ** 2).sum(-1)).sum() + 0.5 * (self.I * (self.w ** 2).sum(-1)).sum())

    def describe(self) -> str:
        d = self.drum
        return (f"DEM: {self.N} pedaços de {self.cfg.d_mean * 1000:.0f} mm (±{self.cfg.d_spread * 100:.0f} %), "
                f"tambor {d.D:.2f} m × {self.Lx:.2f} m{' (fatia periódica)' if self.periodic else ''}, "
                f"{d.n_lifters} aletas, enchimento {self.cfg.fill * 100:.0f} %, k_n = {self.kn:.3g} N/m, "
                f"Δt = {self.dt * 1e3:.3f} ms, Fr = {d.froude():.2e}")

In [ ]:
%%writefile tamborkit/phasedem.py
"""DEM com mudança de fase do lixo ao longo de toda a batelada (acoplado em uma via ao modelo da batelada).

Cada partícula é um pedaço de um material do lixo, com a massa real que ele representa na fatia simulada:
filme de PEBD, frasco de PEAD, tampa/rede de PP, isopor (PS expandido), garrafa PET, tubo de PVC, papel/
madeira, borracha e areia/conchas. O modelo da batelada continua mandando no calor e na química (temperatura da
carga e conversão de cada componente ao longo do tempo); o DEM mostra o que isso faz com os pedaços:

  * secagem: a água sai (massa) sem mudar a forma;
  * plásticos que fundem (PEBD ~110 °C, PEAD ~130, PS amolece ~100–150, PP ~165, PET ~255 °C): o volume do
    pedaço colapsa do aparente ("solto") para o denso conforme a fração fundida f_m(T); na janela de fusão o
    pedaço fica pegajoso (coesão + atrito maiores: pontes de fundido); com f_m ≥ f_merge ele coalesce na
    POÇA DE FUNDIDO (sai do DEM e entra no inventário da poça, por componente);
  * celulósicos, borracha e PVC não fundem: perdem massa pela cinética e encolhem até virar carvão (escurecem);
  * a poça: volume pela massa de cada polímero ainda não craqueado / densidade do fundido; viscosidade
    η(T, conversão) que despenca com a cisão das cadeias; nos pedaços submersos atuam empuxo (ρ_fundido V g) e
    arrasto de Stokes (relaxação exata da velocidade para a do fundido, estável com η alto);
  * o carvão que se forma no fundido (PS, PET, PVC, poliolefinas) aparece como grânulos novos na poça.

Relógio: o movimento de cada volta é o real (mesmo regime de rolamento), mas a batelada de horas é comprimida
em poucas voltas — o DEM não tenta reproduzir as ~290 voltas de 6 h; a temperatura e a conversão vêm do modelo
validado da batelada (a carga girando é quase isotérmica na seção). Uma pequena dispersão de temperatura entre
pedaços (σ ≈ 8 K: graus e formas diferentes do mesmo material) espalha a fusão de cada classe.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np
import torch

from . import feed as fd
from .dem import DEMConfig, DrumDEM, G
from .geometry import Drum, fill_half_angle, lifter_contacts

# rótulo no DEM, componentes da carga, diâmetro típico do pedaço [m], cor sRGB (cores aproximadas dos materiais)
MATERIALS = [
    ("filme", ["LDPE"], 0.09, (228, 228, 218)),
    ("rigido_pe", ["HDPE"], 0.08, (70, 130, 200)),
    ("rigido_pp", ["PP"], 0.07, (232, 186, 64)),
    ("isopor", ["PS"], 0.10, (246, 246, 246)),
    ("pet", ["PET"], 0.08, (125, 200, 175)),
    ("pvc", ["PVC"], 0.06, (150, 152, 160)),
    ("organico", ["CELL"], 0.07, (150, 105, 60)),
    ("borracha", ["RUBa", "RUBb", "RUBc", "CB"], 0.06, (42, 42, 42)),
    ("areia", ["INERT"], 0.03, (205, 185, 140)),
]
CHAR_RGB = (26, 22, 20)
MELT_RGB = (196, 128, 52)           # fundido de poliolefina âmbar; escurece com o craqueamento
NAMES = [m[0] for m in MATERIALS] + ["carvao"]
MELTERS = {"filme": "LDPE", "rigido_pe": "HDPE", "rigido_pp": "PP", "isopor": "PS", "pet": "PET"}


@dataclass
class PhaseConfig:
    slab: float = 0.5                # comprimento da fatia periódica [m]
    rpm: float = 3.0                 # rotação simulada (regime de rolamento; o real é 0,8 rpm)
    n_rev: float = 5.0               # voltas do DEM para a batelada inteira (relógio comprimido)
    pre_rev: float = 1.0             # voltas frias antes de ligar o relógio (segmento "dentro do tambor");
                                     # a validação do DEM (ângulo, arco coberto) usa a 2ª metade
    contact_time: float = 6.0e-3     # duração do contato [s] (rigidez proporcional à massa)
    d_spread: float = 0.25
    phi_pack: float = 0.58
    rho_inert: float = 2300.0        # pedaços de concha/vidro/pedra [kg/m³]
    sigma_T: float = 8.0             # dispersão de temperatura entre pedaços [K]
    f_merge: float = 0.85            # fração fundida a partir da qual o pedaço entra na poça
    cohesion: float = 1.5            # força coesiva máxima / peso efetivo, para pedaços totalmente pegajosos
    flow_frac: float = 0.6           # velocidade do fundido = flow_frac × rotação de corpo rígido
    beta_max_deg: float = 30.0       # inclinação máxima da superfície da poça (fundido muito viscoso, λ < 2)
    char_d: float = 0.03             # diâmetro dos grânulos de carvão que nascem na poça [m]
    char_rho: float = 420.0          # massa específica aparente dos grânulos [kg/m³]
    shrink_char: dict = field(default_factory=lambda: {"organico": 0.45, "borracha": 0.55, "pvc": 0.70})
    reserve: float = 0.25            # partículas reservadas para os grânulos (fração das ativas)
    upd_dt: float = 0.05             # intervalo entre atualizações de estado [s de DEM]
    frames: int = 1200               # quadros gravados ao longo da batelada
    pre_frames: int = 120            # quadros gravados no trecho frio
    seed: int = 1
    device: str = "auto"


def melt_viscosity(T_K, conv):
    """Viscosidade do fundido de poliolefinas [Pa s] durante a pirólise.

    η₀(T) de PE/PP fundido (~10³ Pa s a 200 °C, energia de ativação de fluxo ~30 kJ/mol) e queda com a cisão
    das cadeias: Mn/Mn₀ = 1/(1 + DPn₀ · 0,068 · √α) (cisão aleatória, mesma relação do reator agitado) e
    η ∝ Mn^3,4 acima do emaranhamento. Abaixo dele, o ramo das ceras: ~18 mPa s a 200 °C (cera de PE com
    Mn ~1500, ficha POLYWAX), energia de ativação ~22 kJ/mol → ~3 mPa s a 420 °C, o valor central do fundido
    já craqueado (0,4–9 mPa s para α = 0,05–0,8). O colapso da viscosidade vem antes da perda de massa
    (α ~ 1 %), como no fundido de PE degradado (Radhakrishnan; NIST TN 1493)."""
    T = np.asarray(T_K, float)
    eta0 = 1.0e3 * np.exp(30e3 / 8.314 * (1.0 / np.maximum(T, 300.0) - 1.0 / 473.15))
    dpn0 = 30.0 / 0.028
    ratio = 1.0 / (1.0 + dpn0 * 0.068 * np.sqrt(np.clip(conv, 0.0, 1.0)))
    eta = eta0 * ratio ** 3.4
    wax = 1.8e-2 * np.exp(22e3 / 8.314 * (1.0 / np.maximum(T, 300.0) - 1.0 / 473.15))
    return np.maximum(eta, wax)


class PhaseDEM(DrumDEM):
    """DEM da carga com mudança de fase dirigida pelo log da batelada (BatchDrum.log)."""

    def __init__(self, drum: Drum, charge: fd.Charge, batch_log: list, pc: PhaseConfig | None = None):
        pc = pc or PhaseConfig()
        self.pc = pc
        self.log = batch_log
        gen = np.random.default_rng(pc.seed)
        self.gen = gen
        m_all = charge.masses()
        frac_slab = pc.slab / drum.L
        st1 = fd.KineticState(m_all, 1)
        V_raw = float(st1.volume(np.array([fd.T0]))[0])
        self.loose_scale = V_raw / (charge.fill * drum.volume())
        dry_tot = sum(v for k, v in m_all.items() if k != "H2O")
        self.moist = m_all["H2O"] / max(dry_tot, 1e-9)
        # ---- pedaços por material: massa real na fatia, densidade aparente do pedaço
        d_list, rho_list, lab_list, m0_list = [], [], [], []
        self.piece_rho, self.dense_rho = {}, {}
        for li, (name, comps, dmean, _) in enumerate(MATERIALS):
            M = sum(m_all[c] for c in comps) * frac_slab
            if M <= 0:
                continue
            if name == "areia":
                rho_piece = pc.rho_inert
                rho_dense = pc.rho_inert
            else:
                c0 = fd.COMPONENTS[comps[0]]
                rho_piece = c0.rho_loose * self.loose_scale / pc.phi_pack
                rho_dense = c0.rho_dense
            self.piece_rho[name] = rho_piece
            self.dense_rho[name] = rho_dense
            V1 = math.pi / 6 * dmean ** 3 * (1 + pc.d_spread ** 2)
            n = max(int(round(M / (rho_piece * V1))), 1)
            d = dmean * gen.uniform(1 - pc.d_spread, 1 + pc.d_spread, n)
            mp = rho_piece * math.pi / 6 * d ** 3
            mp *= M / mp.sum()                         # massa exata da fatia
            d_list.append(d); rho_list.append(mp / (math.pi / 6 * d ** 3)); lab_list += [li] * n
            m0_list.append(mp)
        d = np.concatenate(d_list); rho = np.concatenate(rho_list); m0 = np.concatenate(m0_list)
        lab = np.array(lab_list)
        n_act = len(d)
        n_res = int(pc.reserve * n_act)
        d = np.concatenate([d, np.full(n_res, pc.char_d)])
        rho = np.concatenate([rho, np.full(n_res, pc.char_rho)])
        lab = np.concatenate([lab, np.full(n_res, len(MATERIALS))])
        m0 = np.concatenate([m0, np.zeros(n_res)])
        active = np.r_[np.ones(n_act, bool), np.zeros(n_res, bool)]
        x0 = self._random_fill(drum, pc.slab, d, gen)
        cfg = DEMConfig(fill=charge.fill, d_mean=float(np.median(d[:n_act])), d_spread=pc.d_spread,
                        rho_bulk=float(np.sum(m0) / (charge.fill * math.pi * drum.R ** 2 * pc.slab)),
                        phi_pack=pc.phi_pack, slab=pc.slab, contact_time=pc.contact_time, cohesion=pc.cohesion,
                        device=pc.device, seed=pc.seed)
        super().__init__(drum, cfg, init=dict(x=x0, d=d, rho=rho, label=lab, names=NAMES, active=active, grow0=0.5))
        self.n_act0 = n_act
        self.m0p = torch.as_tensor(m0, dtype=self.ft, device=self.dev)          # massa seca inicial do pedaço
        self.r0 = self.r_final.clone()
        self.dT = torch.as_tensor(gen.normal(0.0, pc.sigma_T, self.N), dtype=self.ft, device=self.dev)
        self.f_m = torch.zeros(self.N, dtype=self.ft, device=self.dev)
        self.charf = torch.zeros(self.N, dtype=self.ft, device=self.dev)      # fração da massa que é carvão
        self.Tp = torch.full((self.N,), fd.T0, dtype=self.ft, device=self.dev)
        self.merged = torch.zeros(self.N, dtype=torch.bool, device=self.dev)
        # inventário da poça: massa "inicial equivalente" de cada polímero (m0_eq·u(t) é o que resta)
        self.pool_m0 = {c: 0.0 for c in MELTERS.values()}
        self.char_spawned = 0.0
        self.pool = dict(area=0.0, eps=0.0, beta=0.0, T=fd.T0, mu=1e3, rho=750.0, conv=0.0, lam=float("inf"))
        self.omega_real = drum.omega                 # a poça representa o fundido do tambor real
        self.t_b = 0.0
        self.records, self.pre_records = [], []
        self.lt = np.array([r["t_h"] for r in batch_log]) * 3600.0
        self.m0_comp = {k: float(sum(m_all[k] for _ in [0])) for k in fd.ORDER}
        self.t_end = float(self.lt[-1])

    # ---------------------------------------------------------- montagem
    @staticmethod
    def _random_fill(drum, slab, d, gen):
        """Posições aleatórias no tambor com raios a 50 % (crescem no assentamento), longe das aletas."""
        N = len(d)
        R = drum.R
        pts = np.zeros((N, 3))
        filled = 0
        while filled < N:
            k = (N - filled) * 3
            P = np.c_[gen.uniform(0, slab, k), gen.uniform(-R, R, k), gen.uniform(-R, R, k)]
            P = P[np.hypot(P[:, 1], P[:, 2]) < R - 0.06]
            if drum.n_lifters:
                dist, _, _ = lifter_contacts(drum, torch.as_tensor(P, dtype=torch.float64), 0.0, slab)
                P = P[(dist.min(1).values > 0.05).numpy()]
            take = min(len(P), N - filled)
            pts[filled:filled + take] = P[:take]
            filled += take
        # os maiores embaixo (assentamento mais rápido e sem pedaços presos no alto)
        order = np.argsort(pts[:, 2])
        big = np.argsort(-d)
        out = np.empty_like(pts)
        out[big] = pts[order]
        return out

    # ---------------------------------------------------------- estado da batelada
    def _log_at(self, t_b):
        i = int(np.clip(np.searchsorted(self.lt, t_b), 1, len(self.lt) - 1))
        a = (t_b - self.lt[i - 1]) / max(self.lt[i] - self.lt[i - 1], 1e-9)
        r0, r1 = self.log[i - 1], self.log[i]
        return {k: (1 - a) * r0[k] + a * r1[k] for k in r0 if isinstance(r0[k], (int, float))}

    def _u(self, row, keys):
        """Fração não reagida de um grupo de componentes."""
        m0 = sum(self.log[0].get(f"m_{k}", 0.0) for k in keys)
        return sum(row.get(f"m_{k}", 0.0) for k in keys) / m0 if m0 > 0 else 1.0

    def update_state(self, t_b):
        """Atualiza massa, raio, fração fundida, pegajosidade e a poça para o instante t_b da batelada."""
        pc = self.pc
        self.t_b = t_b
        row = self._log_at(t_b)
        T_bed = row["T_carga"] + 273.15
        lab = self.label
        Tp = T_bed + self.dT
        self.Tp = Tp
        u_w = self._u(row, ["H2O"])
        m_new = torch.zeros_like(self.m0p)
        vol_ratio = torch.ones_like(self.m0p)
        stick = torch.zeros_like(self.m0p)
        f_m = torch.zeros_like(self.m0p)
        charf = torch.zeros_like(self.m0p)
        Tp_np = Tp.cpu().numpy()
        for li, (name, comps, _, _) in enumerate(MATERIALS):
            sel = (lab == li)
            if not bool(sel.any()):
                continue
            selnp = sel.cpu().numpy()
            if name in MELTERS:
                c = fd.COMPONENTS[MELTERS[name]]
                u = self._u(row, [MELTERS[name]])
                fm = torch.as_tensor(fd.melt_fraction(c, Tp_np[selnp]), dtype=self.ft, device=self.dev)
                ratio = self.piece_rho[name] / self.dense_rho[name]
                f_m[sel] = fm
                vol_ratio[sel] = u * ((1 - fm) + fm * ratio)
                m_new[sel] = self.m0p[sel] * (u + self.moist * u_w)
                stick[sel] = ((fm - 0.03) / 0.45).clamp(0.0, 1.0)
            elif name == "organico":
                u = self._u(row, ["CELL"])
                ych = 0.29
                m_new[sel] = self.m0p[sel] * (u + ych * (1 - u) + self.moist * u_w)
                vol_ratio[sel] = u + (1 - u) * pc.shrink_char["organico"]
                charf[sel] = ych * (1 - u) / max(u + ych * (1 - u), 1e-6)
            elif name == "borracha":
                u = self._u(row, ["RUBa", "RUBb", "RUBc"])
                m_new[sel] = self.m0p[sel] * (0.6 * u + 0.4 + self.moist * u_w)
                vol_ratio[sel] = u + (1 - u) * pc.shrink_char["borracha"]
                charf[sel] = 0.4 / max(0.6 * u + 0.4, 1e-6) * (1 - u) + 0.0
            elif name == "pvc":
                m0 = self.log[0].get("m_PVC", 1e-9)
                mP, mR = row.get("m_PVC", 0.0), row.get("m_PVCr", 0.0)
                consumed_r = max(0.417 * (m0 - mP) - mR, 0.0)
                frac = (mP + mR + 0.25 * consumed_r) / m0
                m_new[sel] = self.m0p[sel] * (frac + self.moist * u_w)
                done = 1 - mP / m0
                vol_ratio[sel] = 1 - done * (1 - pc.shrink_char["pvc"])
                charf[sel] = min(1.0, (mR + 0.25 * consumed_r) / max(mP + mR + 0.25 * consumed_r, 1e-9))
            elif name == "areia":
                m_new[sel] = self.m0p[sel]
        # grânulos de carvão já nascidos: massa e raio fixos
        gran = lab == len(MATERIALS)
        m_new[gran] = self.m[gran]
        charf[gran] = 1.0
        # ---- coalescência na poça: pedaços fundidos saem do DEM e entram no inventário
        newly = (f_m >= pc.f_merge) & (self.active > 0) & (~gran)
        if bool(newly.any()):
            for name, comp in MELTERS.items():
                li = NAMES.index(name)
                s = newly & (lab == li)
                if bool(s.any()):
                    u = self._u(row, [comp])
                    self.pool_m0[comp] += float(self.m0p[s].sum()) * u / max(u, 1e-9) if u > 0 else 0.0
            self.active[newly] = 0.0
            self.v[newly] = 0.0
            self.w[newly] = 0.0
            self.merged |= newly
            self._dirty = True
        # ---- atualiza massa e raio (só ativos)
        act = (self.active > 0) & (~gran)
        r_new = self.r0 * vol_ratio.clamp(min=1e-3) ** (1.0 / 3.0)
        self.r_final = torch.where(act, r_new, self.r_final)
        self.r = torch.where(act, r_new, self.r)
        V = 4.0 / 3.0 * math.pi * self.r ** 3
        m_eff = torch.where(act, m_new.clamp(min=1e-6), self.m)
        self.m = m_eff
        self.rho_p = torch.where(act, self.m / V, self.rho_p if torch.is_tensor(self.rho_p) else self.m / V)
        self.I = 0.4 * self.m * self.r ** 2
        self.stick = torch.where(act, stick, torch.zeros_like(stick))
        self.f_m, self.charf = f_m, charf
        # ---- poça: massa por polímero não craqueado, volume, viscosidade
        mass_pool, vol_pool, char_made = 0.0, 0.0, 0.0
        for comp, m0eq in self.pool_m0.items():
            if m0eq <= 0:
                continue
            u = self._u(row, [comp])
            # m0eq foi guardado como massa no momento da fusão / u(t_fusão) ≈ massa inicial equivalente
            m = m0eq * u
            mass_pool += m
            vol_pool += m / fd.COMPONENTS[comp].rho_dense
            char_made += m0eq * (1 - u) * fd.COMPONENTS[comp].yields.get("CHAR", 0.0)
        conv_po = 1.0 - sum(self.pool_m0[c] * self._u(row, [c]) for c in self.pool_m0) / max(sum(self.pool_m0.values()), 1e-9)
        area = vol_pool / pc.slab
        F = area / (math.pi * self.drum.R ** 2)
        eps = fill_half_angle(F) if F > 1e-5 else 0.0
        mu = float(melt_viscosity(T_bed, conv_po))
        rho_p = mass_pool / max(vol_pool, 1e-12) if vol_pool > 0 else 750.0
        # superfície da poça: o fundido viscoso é arrastado pela parede e se acumula no lado que sobe.
        # Inclinação ~ atan(μU/(ρgH²)) (U = ΩR, H = profundidade); regimes de Ashmore et al. (2003) pelo
        # λ = F²ρgR/(μΩ): λ < 2 o fundido gira com a parede (filme), λ > 5 poça plana no fundo.
        R = self.drum.R
        H = R * (1 - math.cos(eps))
        U = self.omega_real * R
        tilt = mu * U / (rho_p * G * H ** 2) if H > 1e-4 else 0.0
        beta = math.atan(min(tilt, math.tan(math.radians(pc.beta_max_deg))))
        lam = F ** 2 * rho_p * G * R / (mu * self.omega_real) if F > 0 else float("inf")
        self.pool.update(area=area, F=F, eps=eps, T=T_bed, mu=mu, rho=rho_p, conv=conv_po, mass=mass_pool,
                         beta=beta, lam=lam)
        # ---- grânulos de carvão nascem na poça
        m_g = pc.char_rho * math.pi / 6 * pc.char_d ** 3
        n_need = int((char_made - self.char_spawned) / m_g)
        if n_need > 0 and self.pool["eps"] > 0:
            free = torch.nonzero((self.active == 0) & gran & (~self.merged)).flatten()
            n_new = min(n_need, len(free))
            if n_new > 0:
                idx = free[:n_new]
                self._spawn(idx, m_g)
                self.char_spawned += n_new * m_g
            if n_new < n_need:
                # reserva esgotada: o carvão que falta engorda os grânulos que já existem (massa conservada)
                g_act = gran & (self.active > 0)
                m_g_act = float(self.m[g_act].sum())
                left = char_made - self.char_spawned
                if m_g_act > 0 and left > 0:
                    f = 1.0 + left / m_g_act
                    self.m[g_act] *= f
                    self.r[g_act] *= f ** (1.0 / 3.0)
                    self.r_final[g_act] = self.r[g_act]
                    self.I[g_act] = 0.4 * self.m[g_act] * self.r[g_act] ** 2
                    self.char_spawned += left
                    self._dirty = True
        if getattr(self, "_dirty", False):
            self._rebuild()
            self._dirty = False

    def _spawn(self, idx, m_g):
        pc, R = self.pc, self.drum.R
        n = len(idx)
        gen = self.gen
        eps, beta = self.pool["eps"], self.pool["beta"]
        nrm = np.array([-math.sin(beta), math.cos(beta)])
        d_surf = -R * math.cos(eps)
        pts = []
        while len(pts) < n:
            y, z = gen.uniform(-R, R, 2)
            dd = y * nrm[0] + z * nrm[1]
            if math.hypot(y, z) < R - pc.char_d and d_surf - 0.25 * R * (1 - math.cos(eps)) < dd < d_surf - 0.5 * pc.char_d:
                pts.append((gen.uniform(0, pc.slab), y, z))
        P = torch.as_tensor(np.array(pts), dtype=self.ft, device=self.dev)
        self.x[idx] = P
        Om = self.omega * pc.flow_frac
        self.v[idx] = torch.stack([torch.zeros(n, device=self.dev), -Om * P[:, 2], Om * P[:, 1]], -1)
        self.w[idx] = 0.0
        self.r[idx] = self.r_final[idx] = self.r0[idx] = 0.5 * pc.char_d
        self.m[idx] = m_g
        self.I[idx] = 0.4 * m_g * (0.5 * pc.char_d) ** 2
        self.active[idx] = 1.0
        self._dirty = True

    # ---------------------------------------------------------- forças da poça
    def _post_step(self, dt):
        p = self.pool
        if p.get("area", 0.0) <= 1e-6:
            return
        R = self.drum.R
        eps, beta = p["eps"], p["beta"]
        nrm = torch.tensor([-math.sin(beta), math.cos(beta)], dtype=self.ft, device=self.dev)
        depth = -R * math.cos(eps) - (self.x[:, 1] * nrm[0] + self.x[:, 2] * nrm[1])    # >0 abaixo da superfície
        sub = ((depth / self.r.clamp(min=1e-3)) * 0.5 + 0.5).clamp(0.0, 1.0) * self.active
        if not bool((sub > 0).any()):
            return
        V = 4.0 / 3.0 * math.pi * self.r ** 3
        # empuxo
        self.v[:, 2] += (p["rho"] * V * sub * G / self.m.clamp(min=1e-9)) * dt
        # arrasto de Stokes: relaxação exata para a velocidade do fundido
        Om = self.omega * self.pc.flow_frac
        vf = torch.stack([torch.zeros_like(self.r), -Om * self.x[:, 2], Om * self.x[:, 1]], -1)
        tau = self.m / (6 * math.pi * p["mu"] * self.r.clamp(min=1e-3) * sub.clamp(min=1e-6))
        a = torch.exp(-dt / tau)[:, None]
        a = torch.where(sub[:, None] > 0, a, torch.ones_like(a))
        self.v = vf + (self.v - vf) * a
        self.w = self.w * a

    # ---------------------------------------------------------- marcha
    def frame_state(self) -> dict:
        f = self.frame()
        act = self.active > 0
        f.update(m_dem=float(self.m[act].sum()), fill_dem=self.bed_fill())
        f.update(t_b=self.t_b, f_m=(self.f_m.clamp(0, 1) * 255).byte().cpu().numpy(),
                 charf=(self.charf.clamp(0, 1) * 255).byte().cpu().numpy(), Tp=self.Tp.half().cpu().numpy(),
                 pool={k: float(v) for k, v in self.pool.items()})
        return f

    def run_batch(self, verbose=True):
        """Assenta, grava o trecho frio e percorre a batelada inteira com o relógio comprimido."""
        pc = self.pc
        self.settle(1.0)
        self.update_state(0.0)
        self.spin_up(pc.rpm)
        T_rev = 60.0 / pc.rpm
        t_pre = pc.pre_rev * T_rev
        rec = lambda d: self.pre_records.append(d.frame_state())  # noqa: E731
        self.run(t_pre / 2, frame_dt=t_pre / pc.pre_frames, on_frame=rec)    # o leito desenvolve o ângulo
        self.reset_stats()
        self.run(t_pre / 2, frame_dt=t_pre / pc.pre_frames, on_frame=rec)
        self.pre_stats = self.stats()              # carga solta (ângulo, arco coberto, contato): validação do DEM
        self.reset_stats()
        T_run = pc.n_rev * T_rev
        n_upd = max(int(round(pc.upd_dt / self.dt)), 1)
        frame_every = T_run / pc.frames
        t0 = self.t
        next_frame = t0
        k = 0
        import time as _t
        wall0 = _t.time()
        while self.t - t0 < T_run:
            t_b = self.t_end * (self.t - t0) / T_run
            self.update_state(t_b)
            self.step(n_upd)
            if self.t >= next_frame:
                self.records.append(self.frame_state())
                next_frame += frame_every
            k += 1
            if verbose and k % max(int(T_run / pc.upd_dt / 10), 1) == 0:
                p = self.pool
                print(f"  DEM fase: batelada {t_b / 3600:4.1f} h  T {self.Tp.mean() - 273.15:5.0f} °C  pedaços ativos "
                      f"{int(self.active.sum())}  poça {p.get('mass', 0.0):6.1f} kg (η {p['mu']:.2g} Pa s)  "
                      f"grânulos {int(((self.label == len(MATERIALS)) & (self.active > 0)).sum())}  "
                      f"[{_t.time() - wall0:5.0f} s]", flush=True)
        self.update_state(self.t_end)
        self.records.append(self.frame_state())
        return self.records

    def bed_fill(self):
        """Enchimento equivalente do DEM (pedaços ativos / empacotamento + poça) para comparar com a batelada."""
        act = self.active > 0
        V = float((4.0 / 3.0 * math.pi * self.r[act] ** 3).sum()) / self.pc.phi_pack
        return (V / self.pc.slab + self.pool.get("area", 0.0)) / (math.pi * self.drum.R ** 2)

    def describe(self):
        counts = {NAMES[i]: int((self.label == i).sum()) for i in range(len(MATERIALS))}
        return (f"DEM com mudança de fase: {self.n_act0} pedaços na fatia de {self.pc.slab:.2f} m ("
                + ", ".join(f"{k} {v}" for k, v in counts.items()) + f"), {self.N - self.n_act0} reservados "
                f"para carvão; Δt = {self.dt * 1e3:.2f} ms; {self.pc.n_rev:.0f} voltas a {self.pc.rpm:.1f} rpm "
                "para a batelada inteira")

In [ ]:
%%writefile tamborkit/combustion.py
"""Termoquímica da combustão na fornalha: fração de mistura, equilíbrio (Cantera) e β-PDF.

Mesma abordagem do flare (química rápida; estado φ(Z) por equilíbrio HP, limite rico de flamabilidade e
integração na β-PDF de Favre), agora com a perda de calor tratada à parte: a LES transporta o defeito de
entalpia h_d e usa T = T_ad(Z̃, s) − h_d/c_p (a fornalha perde calor para o tambor, que é o objetivo).

Combustíveis: GLP (propano) para a partida e o gás de pirólise das poliolefinas a 500 °C (rico em CH4,
seguido de C2–C3 e H2; composição de projeto [H] dentro das faixas de Calero 2023 e Honus 2018).
"""
from __future__ import annotations

import math

import numpy as np
from scipy.special import betainc

P_ATM = 101325.0
R_U = 8.314462618
FUEL_X = {
    "GLP": {"C3H8": 1.0},
    "gas_pirolise": {"CH4": 0.45, "C2H6": 0.15, "C2H4": 0.15, "C3H8": 0.10, "H2": 0.10, "CO": 0.03, "CO2": 0.02},
}
_MW = {"C3H8": 44.097, "CH4": 16.043, "C2H6": 30.07, "C2H4": 28.054, "H2": 2.016, "CO": 28.01, "CO2": 44.01,
       "O2": 31.998, "N2": 28.014, "H2O": 18.015}
_CHO = {"C3H8": (3, 8, 0), "CH4": (1, 4, 0), "C2H6": (2, 6, 0), "C2H4": (2, 4, 0), "H2": (0, 2, 0),
        "CO": (1, 0, 1), "CO2": (1, 0, 2)}


def stoich(fuel_X: dict) -> dict:
    """Z_st, ar estequiométrico e produtos por kg de combustível."""
    tot = sum(fuel_X.values())
    X = {k: v / tot for k, v in fuel_X.items()}
    M = sum(X[k] * _MW[k] for k in X)
    nC = sum(X[k] * _CHO[k][0] for k in X)
    nH = sum(X[k] * _CHO[k][1] for k in X)
    nO = sum(X[k] * _CHO[k][2] for k in X)
    nO2 = nC + nH / 4 - nO / 2
    afr = nO2 * (_MW["O2"] + 3.76 * _MW["N2"]) / M
    return dict(Z_st=1 / (1 + afr), AFR=afr, M=M / 1000, nC=nC, nH=nH, nO2=nO2, X=X)


def state_relation(fuel_X: dict, T_fuel: float = 300.0, T_air: float = 300.0, n: int = 801,
                   phi_rich: float = 2.5) -> dict:
    """φ(Z) adiabático: T, ρ, M, x_H2O + x_CO2 e uma fuligem de relação de estado para o brilho da chama."""
    st = stoich(fuel_X)
    Zs = st["Z_st"]
    zb = min(4 * Zs, 0.5)
    Z = np.concatenate([np.linspace(0, zb, n // 2, endpoint=False), np.linspace(zb, 1, n - n // 2)])
    try:
        import cantera as ct
        gas = ct.Solution("gri30.yaml")
        gas.TPX = T_fuel, P_ATM, st["X"]
        hF, YF = gas.enthalpy_mass, gas.Y.copy()
        gas.TPX = T_air, P_ATM, {"O2": 1.0, "N2": 3.76}
        hA, YA = gas.enthalpy_mass, gas.Y.copy()
        Z_rich = phi_rich * Zs / (1 - Zs + phi_rich * Zs)
        gas.HPY = Z_rich * hF + (1 - Z_rich) * hA, P_ATM, Z_rich * YF + (1 - Z_rich) * YA
        gas.equilibrate("HP")
        h_r, Y_r = gas.enthalpy_mass, gas.Y.copy()
        T = np.empty_like(Z); M = np.empty_like(Z); pwc = np.empty_like(Z)
        iw, ic = gas.species_index("H2O"), gas.species_index("CO2")
        for i, z in enumerate(Z):
            if z <= Z_rich:
                gas.HPY = z * hF + (1 - z) * hA, P_ATM, z * YF + (1 - z) * YA
                if z > 1e-6:
                    gas.equilibrate("HP")
            else:
                w = (z - Z_rich) / (1 - Z_rich)
                gas.HPY = (1 - w) * h_r + w * hF, P_ATM, (1 - w) * Y_r + w * YF
            T[i] = gas.T; M[i] = gas.mean_molecular_weight / 1000; pwc[i] = gas.X[iw] + gas.X[ic]
        source = "Cantera (equilíbrio HP, gri30)"
    except Exception as e:   # noqa: BLE001
        cp = 1350.0
        lhv = 46e6
        burn = np.where(Z <= Zs, Z / Zs, (1 - Z) / (1 - Zs))
        T = T_air + (T_fuel - T_air) * Z + burn * Zs * lhv / cp
        M = np.full_like(Z, 0.0285)
        pwc = 0.21 * burn
        source = f"Burke–Schumann (Cantera indisponível: {type(e).__name__})"
    rho = P_ATM * M / (R_U * T)
    phi = (Z / np.maximum(1 - Z, 1e-12)) / (Zs / (1 - Zs))
    fv = 1e-6 * np.clip(np.minimum(phi - 1.0, (5.0 - phi) / 3.0), 0.0, 1.0)
    return dict(Z=Z, T=T, rho=rho, M=M, pwc=pwc, fv=fv, Z_st=Zs, AFR=st["AFR"], source=source,
                T_ad=float(np.interp(Zs, Z, T)))


def beta_table(sr: dict, nZ: int = 201, ns: int = 11, s_max: float = 0.8) -> dict:
    """Médias de Favre na β-PDF: ρ, ∂ρ/∂Z, T, p_wc e f_v em grade uniforme (Z̃, s)."""
    Zt = np.linspace(0, 1, nZ)
    s = np.linspace(0, s_max, ns)
    zf = np.linspace(0, 1, 2001)
    zm = 0.5 * (zf[1:] + zf[:-1])
    inv_rho = 1 / np.interp(zm, sr["Z"], sr["rho"])
    fields = {k: np.interp(zm, sr["Z"], sr[k]) for k in ("T", "pwc", "fv")}
    out = {k: np.zeros((nZ, ns)) for k in ("inv_rho", "T", "pwc", "fv")}
    for j, sj in enumerate(s):
        for i, z in enumerate(Zt):
            if sj > 1e-6 and 1e-6 < z < 1 - 1e-6:
                g = 1 / sj - 1
                w = np.diff(betainc(z * g, (1 - z) * g, zf))
                w = w / max(w.sum(), 1e-30)
                # média de Favre: φ̃ = ∫ φ P̃ dZ; ρ̄ = 1/∫(1/ρ) P̃ dZ
                out["inv_rho"][i, j] = w @ inv_rho
                for k in ("T", "pwc", "fv"):
                    out[k][i, j] = w @ fields[k]
            else:
                out["inv_rho"][i, j] = 1 / np.interp(z, sr["Z"], sr["rho"])
                for k in ("T", "pwc", "fv"):
                    out[k][i, j] = np.interp(z, sr["Z"], sr[k])
    rho = 1 / out["inv_rho"]
    return dict(Zt=Zt, s=s, rho=rho, drho=np.gradient(rho, Zt, axis=0), T=out["T"], pwc=out["pwc"],
                fv=out["fv"], Z_st=sr["Z_st"], T_ad=sr["T_ad"], rho_air=float(sr["rho"][0]), source=sr["source"])

In [ ]:
%%writefile tamborkit/furnace_les.py
"""LES 3D da fornalha: chamas dos queimadores sob o tambor, gases subindo pelos corredores e saindo no topo.

Baixo Mach com densidade variável, no espírito do FDS e do flare (mesma família de métodos):
  * malha MAC uniforme; momento em forma advectiva (upwind de 3ª ordem), empuxo, termo baroclínico,
    Smagorinsky; RK2 com projeção; Poisson direto por diagonalização (paredes com Neumann, saída com
    Dirichlet no fim do plenum de exaustão);
  * química rápida: fração de mistura Z̃ (TVD Superbee) + β-PDF do equilíbrio (combustion.py);
  * perda de calor: defeito de entalpia h_d transportado, T = T_ad(Z̃, s) − h_d/c_p, com fonte
        S_h = 4 κ σ (T⁴ − T_viz⁴)  (radiação; κ do gás pelo WSGG e da fuligem)  +  h (T − T_parede) A/V;
    a restrição de divergência inclui −S_h/(ρ c_p T) (o gás que esfria contrai);
  * tambor como fronteira imersa que gira devagar (Ω R ~0,06 m/s), a uma temperatura de casco dada.

Saídas: campos médios (T, Z, velocidade), fluxo de calor no casco por fatia × ângulo (relações de Hottel
locais com o gás vizinho ao casco — as mesmas do modelo zonal, agora com a distribuição resolvida pela
LES), distribuição axial do calor liberado e balanço de energia.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass

import numpy as np
import scipy.linalg
import torch

from .combustion import FUEL_X, beta_table, state_relation, stoich
from .furnace import wsgg_emissivity

G = 9.81
SIGMA = 5.670374419e-8
CP = 1300.0


def _sl(a, axis, start, stop=None):
    idx = [slice(None)] * a.ndim
    idx[axis] = slice(start, stop)
    return a[tuple(idx)]


def _pad(a, axis, n=1):
    lo, hi = _sl(a, axis, 0, 1), _sl(a, axis, -1, None)
    return torch.cat([lo] * n + [a] + [hi] * n, dim=axis)


def _avg(a, axis):
    return 0.5 * (_sl(a, axis, 1, None) + _sl(a, axis, 0, -1))


def _diff(a, axis):
    return _sl(a, axis, 1, None) - _sl(a, axis, 0, -1)


def _to_faces(c, axis):
    return _avg(_pad(c, axis), axis)


@dataclass
class FurnaceLESConfig:
    delta: float = 0.05          # tamanho da célula [m]
    L_drum: float = 4.9
    R_out: float = 0.712
    half_width: float = 1.0      # meia largura interna da caixa [m]
    z_bot: float = -1.30         # piso da câmara de combustão
    z_top: float = 0.95          # teto da fornalha
    plenum: float = 0.45         # comprimento do plenum de exaustão atrás do tambor [m]
    burners_x: tuple = (1.2, 3.7)
    burner_z: float = -1.0
    air_patch: float = 0.20      # lado da boca de ar do queimador [m]
    Q_fire: float = 400e3        # potência total [W]
    lam: float = 1.3
    fuel: str = "GLP"
    T_shell: float = 773.15      # casco do tambor [K]
    T_refr: float = 1050.0       # refratário (para a radiação de vizinhança) [K]
    eps_s: float = 0.85
    eps_r: float = 0.85
    rpm: float = 0.8
    Lm: float = 0.5              # comprimento médio de feixe para κ do gás [m]
    h_wall: float = 8.0
    Cs: float = 0.12
    Sc_t: float = 0.7
    C_Z: float = 0.15
    cfl: float = 0.45
    dt_max: float = 0.02
    device: str = "auto"
    seed: int = 0

    @staticmethod
    def preset(name: str, **kw) -> "FurnaceLESConfig":
        d = {"teste": dict(delta=0.10), "gpu": dict(delta=0.05), "gpu_fino": dict(delta=0.035)}[name]
        d.update(kw)
        return FurnaceLESConfig(**d)


class FurnaceLES:
    def __init__(self, cfg: FurnaceLESConfig, table: dict | None = None):
        self.cfg = c = cfg
        dev = c.device
        if dev == "auto":
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = torch.device(dev)
        self.ft = torch.float32
        torch.manual_seed(c.seed)
        t = lambda a: torch.as_tensor(np.array(a, np.float64), dtype=self.ft, device=self.dev)  # noqa: E731
        self._t = t
        h = c.delta
        self.h = h
        Lx = c.L_drum + c.plenum
        self.nx = int(round(Lx / h)); self.ny = int(round(2 * c.half_width / h))
        self.nz = int(round((c.z_top - c.z_bot) / h))
        self.xf = np.linspace(0, self.nx * h, self.nx + 1)
        self.yf = np.linspace(-c.half_width, -c.half_width + self.ny * h, self.ny + 1)
        self.zf = np.linspace(c.z_bot, c.z_bot + self.nz * h, self.nz + 1)
        self.xc, self.yc, self.zc = [0.5 * (f[1:] + f[:-1]) for f in (self.xf, self.yf, self.zf)]
        X, Y, Z = np.meshgrid(self.xc, self.yc, self.zc, indexing="ij")
        self.n_cells = X.size
        # ---- sólidos: tambor + parede traseira/piso do plenum (o gás só sai pelo topo, atrás do tambor)
        drum = (np.hypot(Y, Z) < c.R_out) & (X < c.L_drum)
        rear = (X >= c.L_drum) & (Z < c.R_out)
        solid = drum | rear
        self.drum_np, self.solid_np = drum, solid
        self.fluid = t(~solid)
        self.drum = t(drum)
        # faces fluidas (as duas células vizinhas fluidas) e velocidade do sólido nas faces
        self.mf = []
        for ax in range(3):
            f = _pad(self.fluid, ax)
            self.mf.append(_sl(f, ax, 0, -1) * _sl(f, ax, 1, None))
        Om = 2 * math.pi * c.rpm / 60
        yv = t(np.broadcast_to(self.yf[None, :, None], (self.nx, self.ny + 1, self.nz)))
        zw = t(np.broadcast_to(self.zf[None, None, :], (self.nx, self.ny, self.nz + 1)))
        zc_v = t(np.broadcast_to(self.zc[None, None, :], (self.nx, self.ny + 1, self.nz)))
        yc_w = t(np.broadcast_to(self.yc[None, :, None], (self.nx, self.ny, self.nz + 1)))
        dfv = _to_faces(self.drum, 1); dfw = _to_faces(self.drum, 2)
        self.v_solid = -Om * zc_v * (dfv > 0)
        self.w_solid = Om * yc_w * (dfw > 0)
        del yv, zw
        # ---- vizinhança de parede (convecção) e anel junto ao casco (fluxo no tambor)
        sol = 1 - self.fluid
        nsol = 0.0
        for ax in range(3):
            sp = torch.cat([torch.ones_like(_sl(sol, ax, 0, 1)), sol, torch.ones_like(_sl(sol, ax, 0, 1))], ax)
            nsol = nsol + _sl(sp, ax, 0, -2) + _sl(sp, ax, 2, None)
        self.wall_area_per_vol = nsol * self.fluid / h
        rho_c = np.hypot(Y, Z)
        self.near_drum = t((rho_c >= c.R_out) & (rho_c < c.R_out + 0.18) & (X < c.L_drum))
        self.phi_c = np.degrees(np.arctan2(Z, Y))
        # ---- termoquímica
        if table is None:
            table = beta_table(state_relation(FUEL_X[c.fuel]))
        self.tab = table
        self.tab_rho, self.tab_drho, self.tab_T = t(table["rho"]), t(table["drho"]), t(table["T"])
        self.tab_pwc, self.tab_fv = t(table["pwc"]), t(table["fv"])
        self.tab_dT = t(np.gradient(table["T"], table["Zt"], axis=0))
        self.nZt, self.ns = table["rho"].shape
        self.s_max = float(table["s"][-1])
        self.rho_air = table["rho_air"]
        self.Z_st = table["Z_st"]
        st = stoich(FUEL_X[c.fuel])
        from .furnace import FUELS
        LHV = FUELS["GLP"]["LHV"] if c.fuel == "GLP" else FUELS["G_HC"]["LHV"]
        self.LHV = LHV
        self.m_fuel = c.Q_fire / LHV
        self.m_air = self.m_fuel * c.lam * st["AFR"]
        # ---- queimadores: fontes volumétricas de massa junto à parede y = y_min (como no flare): o ar
        # entra num bloco quadrado de lado air_patch e o combustível na célula central, com a quantidade de
        # movimento dos jatos (velocidade real de saída, sentido +y)
        nb = len(c.burners_x)
        Sa = np.zeros((self.nx, self.ny, self.nz)); Sf = np.zeros_like(Sa)
        npa = max(int(round(c.air_patch / h)), 2)
        rho_f = table["rho"][-1, 0]
        self.u_air = 8.0                          # m/s na boca de ar [H]
        self.u_fuel = 25.0                        # m/s no bico de gás [H]
        self.burner_cells = []
        for xb in c.burners_x:
            i0 = int(np.argmin(np.abs(self.xc - xb))); k0 = int(np.argmin(np.abs(self.zc - c.burner_z)))
            ii = slice(i0 - npa // 2, i0 - npa // 2 + npa); kk = slice(k0 - npa // 2, k0 - npa // 2 + npa)
            Sa[ii, 0, kk] = 1.0
            Sf[i0, 0, k0] = 1.0
            self.burner_cells.append((i0, k0))
        Sa = Sa / Sa.sum() * self.m_air / h ** 3
        Sf = Sf / Sf.sum() * self.m_fuel / h ** 3
        self.Sa, self.Sf = t(Sa), t(Sf)
        self.Sm = self.Sa + self.Sf
        # quantidade de movimento dos jatos (faces v do lado de dentro das células-fonte)
        Smv = _to_faces(self.Sm, 1)
        self.Sm_v = Smv
        self.ujet_v = torch.where(_to_faces(self.Sf, 1) > 0.5 * _to_faces(self.Sm, 1), torch.full_like(Smv, self.u_fuel),
                                  torch.full_like(Smv, self.u_air))
        # ---- Poisson: x (N parede frontal, D saída), y (N, N), z (N, N)
        self.bc = (("N", "D"), ("N", "N"), ("N", "N"))
        lams, Vs, Ws = [], [], []
        for ax, n in enumerate((self.nx, self.ny, self.nz)):
            lam, V, W = self._lap1d(n, *self.bc[ax])
            lams.append(lam); Vs.append(V); Ws.append(W)
        self.V, self.Vi = Vs, Ws
        self.lam = lams[0].view(-1, 1, 1) + lams[1].view(1, -1, 1) + lams[2].view(1, 1, -1)
        self.lam = torch.where(self.lam.abs() < 1e-9, torch.full_like(self.lam, -1e-9), self.lam)
        # ---- campos
        z = lambda *s: torch.zeros(*s, dtype=self.ft, device=self.dev)  # noqa: E731
        self.u, self.v, self.w = z(self.nx + 1, self.ny, self.nz), z(self.nx, self.ny + 1, self.nz), z(self.nx, self.ny, self.nz + 1)
        self.Z, self.hd, self.Hp = z(self.nx, self.ny, self.nz), z(self.nx, self.ny, self.nz), z(self.nx, self.ny, self.nz)
        # refratário quase adiabático: rerradia para o tambor o que recebe, então o "sumidouro" efetivo da
        # radiação do gás é o casco (a perda pela fibra cerâmica é pequena e fica no modelo zonal)
        self.T_surr = c.T_shell
        # partida a quente: a caixa cheia de ar a T_surr (encurta o transitório; o ar que entra está frio)
        self.hd = self.hd - CP * (self.T_surr - 300.0) * self.fluid
        self.time, self.steps, self.wall, self.dt = 0.0, 0, 0.0, c.dt_max
        self._numax = 0.0
        self.avg = None
        self.history = {k: [] for k in ("t", "Tmax", "Q_loss", "Q_out", "dt")}
        self._props()

    # ------------------------------------------------------------ Poisson
    def _lap1d(self, n, lo, hi):
        h = self.h
        B = np.zeros((n, n))
        idx = np.arange(n - 1)
        B[idx, idx + 1] = B[idx + 1, idx] = 1.0 / h
        diag = np.zeros(n); diag[:-1] -= 1.0 / h; diag[1:] -= 1.0 / h
        if lo == "D":
            diag[0] -= 2.0 / h
        if hi == "D":
            diag[-1] -= 2.0 / h
        B[np.arange(n), np.arange(n)] = diag
        lam, V = scipy.linalg.eigh(B, np.diag(np.full(n, h)))
        return self._t(lam), self._t(V), self._t(V.T * h)

    def poisson(self, rhs):
        (Wx, Wy, Wz), (Vx, Vy, Vz) = self.Vi, self.V
        r = torch.einsum("ai,ijk->ajk", Wx, rhs)
        r = torch.einsum("bj,ajk->abk", Wy, r)
        r = torch.einsum("ck,abk->abc", Wz, r) / self.lam
        r = torch.einsum("ia,abc->ibc", Vx, r)
        r = torch.einsum("jb,ibc->ijc", Vy, r)
        return torch.einsum("kc,ijc->ijk", Vz, r)

    def _grad_face(self, Hc, axis):
        lo, hi = self.bc[axis]
        g_lo = _sl(Hc, axis, 0, 1) * (1.0 if lo == "N" else -1.0)
        g_hi = _sl(Hc, axis, -1, None) * (1.0 if hi == "N" else -1.0)
        return _diff(torch.cat([g_lo, Hc, g_hi], dim=axis), axis) / self.h

    def divergence(self, u, v, w):
        return (_diff(u, 0) + _diff(v, 1) + _diff(w, 2)) / self.h

    def _bc(self, u, v, w):
        """Paredes da caixa (normal nula), bocas dos queimadores e sólidos imersos."""
        u = u * self.mf[0]
        v = v * self.mf[1] + self.v_solid * (1 - self.mf[1])
        w = w * self.mf[2] + self.w_solid * (1 - self.mf[2])
        v = torch.cat([torch.zeros_like(v[:, :1]), v[:, 1:-1], torch.zeros_like(v[:, :1])], 1)
        w = torch.cat([torch.zeros_like(w[:, :, :1]), w[:, :, 1:-1], torch.zeros_like(w[:, :, :1])], 2)
        u = torch.cat([torch.zeros_like(u[:1]), u[1:]], 0)
        return u, v, w

    def _Aop(self, H):
        """Laplaciano só no fluido (faces de sólido sem fluxo); identidade nas células sólidas."""
        out = 0.0
        for ax in range(3):
            out = out + _diff(self.mf[ax] * self._grad_face(H, ax), ax) / self.h
        return out * self.fluid - (1 - self.fluid) * H

    def _solve_pressure(self, b, x0, tol=2e-3, maxit=60):
        """PCG para −A x = −b com o Poisson direto da caixa inteira como pré-condicionador."""
        x = x0.clone()
        r = -b + self._Aop(x)
        bn = float(b.norm()) + 1e-30
        z = -self.poisson(r)
        p = z.clone()
        rz = (r * z).sum()
        it = 0
        for it in range(maxit):
            if float(r.norm()) < tol * bn:
                break
            Kp = -self._Aop(p)
            alpha = rz / (p * Kp).sum()
            x = x + alpha * p
            r = r - alpha * Kp
            z = -self.poisson(r)
            rz_new = (r * z).sum()
            p = z + (rz_new / rz) * p
            rz = rz_new
        self.cg_iters = it
        return x

    def _rho_face(self, rho, axis):
        return _to_faces(rho, axis)

    def project(self, u, v, w, rho_new, rho_old, dt_rate, coef):
        """Projeção na continuidade discreta: ∇·(ρ_f u) = S − (ρ_novo − ρ_antigo)/Δt, u = u* − c ∇p/ρ_f.

        ρ vem da equação de estado (Z̃, h_d): a massa fecha por construção, mesmo com ρ(Z) muito não linear."""
        u, v, w = self._bc(u, v, w)
        rf = [self._rho_face(rho_new, a) for a in range(3)]
        divm = (_diff(rf[0] * u, 0) + _diff(rf[1] * v, 1) + _diff(rf[2] * w, 2)) / self.h
        b = (divm - self.Sm + (rho_new - rho_old) / dt_rate) / coef * self.fluid
        p = self._solve_pressure(b, self.Hp if self.Hp.shape == b.shape else torch.zeros_like(b))
        u = u - coef * self.mf[0] * self._grad_face(p, 0) / rf[0]
        v = v - coef * self.mf[1] * self._grad_face(p, 1) / rf[1]
        w = w - coef * self.mf[2] * self._grad_face(p, 2) / rf[2]
        u, v, w = self._bc(u, v, w)
        return u, v, w, p

    # ----------------------------------------------------------- propriedades
    def _lookup(self, tab, Z, s):
        fz = Z.clamp(0, 1) * (self.nZt - 1)
        fs = (s / self.s_max).clamp(0, 1) * (self.ns - 1)
        iz = fz.floor().clamp(max=self.nZt - 2).long(); js = fs.floor().clamp(max=self.ns - 2).long()
        wz, ws = fz - iz, fs - js
        f = tab.reshape(-1); b = iz * self.ns + js
        return ((1 - wz) * (1 - ws) * f[b] + wz * (1 - ws) * f[b + self.ns]
                + (1 - wz) * ws * f[b + 1] + wz * ws * f[b + self.ns + 1])

    def _cgrad(self, q, axis):
        qp = _pad(q, axis)
        return (_sl(qp, axis, 2, None) - _sl(qp, axis, 0, -2)) / (2 * self.h)

    def _props(self):
        Z = self.Z
        g2 = sum(self._cgrad(Z, a) ** 2 for a in range(3))
        s = (self.cfg.C_Z * self.h ** 2 * g2 / (Z * (1 - Z) + 1e-8)).clamp(0, self.s_max)
        self.s = s
        Tad = self._lookup(self.tab_T, Z, s)
        rho_ad = self._lookup(self.tab_rho, Z, s)
        self.hd = torch.minimum(self.hd, CP * (Tad - 300.0)).clamp(min=-CP * 1200.0)
        self.T = Tad - self.hd / CP
        self.rho = rho_ad * Tad / self.T
        # ∂ρ/∂Z a h_d constante (ρ = ρ_ad T_ad / T, T = T_ad − h_d/c_p)
        self.drho = (self._lookup(self.tab_drho, Z, s) * Tad / self.T
                     - rho_ad * self._lookup(self.tab_dT, Z, s) * self.hd / (CP * self.T ** 2))
        pwc = self._lookup(self.tab_pwc, Z, s)
        fv = self._lookup(self.tab_fv, Z, s)
        eps = self._wsgg(self.T, pwc * self.cfg.Lm)
        kg = -torch.log((1 - eps).clamp(min=1e-6)) / self.cfg.Lm
        self.kappa = kg + 1862.0 * fv * self.T
        self.Sh = (4 * self.kappa * SIGMA * (self.T ** 4 - self.T_surr ** 4)
                   + self.cfg.h_wall * self.wall_area_per_vol * (self.T - self.cfg.T_shell)) * self.fluid

    def _wsgg(self, T, pL):
        """WSGG de Smith et al. (1982) em PyTorch (mesmos coeficientes de furnace.py)."""
        from .furnace import _WSGG_B, _WSGG_K
        Tc = T.clamp(600.0, 2400.0)
        eps = 0.0
        for i in range(3):
            b = _WSGG_B[i]
            a = b[0] + Tc * (b[1] + Tc * (b[2] + Tc * b[3]))
            eps = eps + a * (1 - torch.exp(-float(_WSGG_K[i]) * pL))
        return eps

    # --------------------------------------------------------------- termos
    def _nu_t(self, u, v, w):
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        g = [[None] * 3 for _ in range(3)]
        for i, (q, qc) in enumerate(((u, uc), (v, vc), (w, wc))):
            for j in range(3):
                g[i][j] = _diff(q, j) / self.h if j == i else self._cgrad(qc, j)
        SS = 0.0
        for i in range(3):
            for j in range(i, 3):
                Sij = 0.5 * (g[i][j] + g[j][i])
                SS = SS + (1.0 if i == j else 2.0) * Sij * Sij
        return (self.cfg.Cs * self.h) ** 2 * torch.sqrt(2.0 * SS)

    def _adv_visc(self, q, adv, mu_q, rho_q):
        h = self.h
        A, Vs = 0.0, 0.0
        for axis, a in enumerate(adv):
            qp = _pad(q, axis, 2)
            n = q.shape[axis]
            qm2, qm1 = _sl(qp, axis, 0, n), _sl(qp, axis, 1, n + 1)
            qp1, qp2 = _sl(qp, axis, 3, n + 3), _sl(qp, axis, 4, n + 4)
            dpos = (2 * qp1 + 3 * q - 6 * qm1 + qm2) / (6 * h)
            dneg = (-qp2 + 6 * qp1 - 3 * q - 2 * qm1) / (6 * h)
            A = A + a * torch.where(a >= 0, dpos, dneg)
            mp = _pad(mu_q, axis)
            mu_p = 0.5 * (mu_q + _sl(mp, axis, 2, None)); mu_m = 0.5 * (mu_q + _sl(mp, axis, 0, -2))
            Vs = Vs + (mu_p * (qp1 - q) - mu_m * (q - qm1)) / (h * h)
        return A - Vs / rho_q

    def _momentum_rhs(self, u, v, w, rho):
        mu = rho * (self._nu_t(u, v, w) + 3e-5)
        uc, vc, wc = _avg(u, 0), _avg(v, 1), _avg(w, 2)
        ru, rv, rw = _to_faces(rho, 0), _to_faces(rho, 1), _to_faces(rho, 2)
        Fu = self._adv_visc(u, (u, _to_faces(vc, 0), _to_faces(wc, 0)), _to_faces(mu, 0), ru)
        Fv = self._adv_visc(v, (_to_faces(uc, 1), v, _to_faces(wc, 1)), _to_faces(mu, 1), rv)
        Fw = self._adv_visc(w, (_to_faces(uc, 2), _to_faces(vc, 2), w), _to_faces(mu, 2), rw)
        Fw = Fw + G * (rw - self.rho_air) / rw
        Fv = Fv - self.Sm_v * (self.ujet_v - v) / rv              # jatos dos queimadores
        Fu = Fu + _to_faces(self.Sm, 0) * u / ru
        Fw = Fw + _to_faces(self.Sm, 2) * w / rw
        return Fu, Fv, Fw, mu

    def _tvd_flux(self, q, a, axis, q_bnd_lo=None):
        """Fluxo a·q_face (Superbee). q_bnd_lo: valor de entrada na face de borda inferior (ou zero-grad.)."""
        n = q.shape[axis]
        qp = _pad(q, axis, 2)
        qLL, qL = _sl(qp, axis, 0, n + 1), _sl(qp, axis, 1, n + 2)
        qR, qRR = _sl(qp, axis, 2, n + 3), _sl(qp, axis, 3, n + 4)

        def sb(num, den):
            den = torch.where(den.abs() < 1e-12, torch.full_like(den, 1e-12), den)
            r = num / den
            return torch.maximum(torch.clamp(2 * r, max=1.0), torch.clamp(r, max=2.0)).clamp(min=0.0)
        qpos = qL + 0.5 * sb(qL - qLL, qR - qL) * (qR - qL)
        qneg = qR + 0.5 * sb(qR - qRR, qL - qR) * (qL - qR)
        qf = torch.where(a >= 0, qpos, qneg)
        if q_bnd_lo is not None:
            qf = torch.cat([q_bnd_lo.unsqueeze(axis), _sl(qf, axis, 1, None)], axis)
        return a * qf

    def _mass_fluxes(self, u, v, w, rho):
        """Fluxos de massa ρ_face·u nas faces (ρ a montante; nas bocas, a densidade do que entra)."""
        out = []
        for axis, a in enumerate((u, v, w)):
            out.append(self._rho_face(rho, axis) * a)
        return out

    def _scalar_rhs(self, q, F, rho, mu, q_in_y):
        """Forma de massa: [−∇·(F q_f) + q ∇·F + ∇·(ρD∇q)]/ρ, F = fluxos de massa (TVD Superbee em q_f)."""
        adv, divF = 0.0, 0.0
        for axis, Fa in enumerate(F):
            Fq = self._tvd_flux(q, Fa, axis, None)
            adv = adv + _diff(Fq, axis) / self.h
            divF = divF + _diff(Fa, axis) / self.h
        rhoD = mu / self.cfg.Sc_t
        dif = 0.0
        for axis in range(3):
            fl = _to_faces(rhoD, axis) * _diff(_pad(q, axis), axis) / self.h * self.mf[axis]
            dif = dif + _diff(fl, axis) / self.h
        return (-adv + q * divF) / rho, dif / rho

    def _compute_dt(self):
        rate = ((_avg(self.u, 0).abs() + _avg(self.v, 1).abs() + _avg(self.w, 2).abs()) / self.h).max().item()
        dt = self.cfg.cfl / max(rate, 1e-6)
        if self._numax > 0:
            dt = min(dt, 0.4 / self._numax)
        return min(dt, self.cfg.dt_max)

    def step(self):
        t0 = time.perf_counter()
        dt = self._compute_dt()
        self.dt = dt
        u0, v0, w0, Z0, h0, rho0 = self.u, self.v, self.w, self.Z, self.hd, self.rho

        def stage(u, v, w, Z, hd, rho):
            Fu, Fv, Fw, mu = self._momentum_rhs(u, v, w, rho)
            F = self._mass_fluxes(u, v, w, rho)
            aZ, dZ = self._scalar_rhs(Z, F, rho, mu, None)
            ah, dh = self._scalar_rhs(hd, F, rho, mu, None)
            RZ = (aZ + dZ + (self.Sf * (1 - Z) - self.Sa * Z) / rho) * self.fluid
            Rh = (ah + dh + (self.Sh - self.Sm * hd) / rho) * self.fluid
            return Fu, Fv, Fw, mu, RZ, Rh

        Fu, Fv, Fw, mu, RZ, Rh = stage(u0, v0, w0, Z0, h0, rho0)
        self.Z = (Z0 + dt * RZ).clamp(0, 1); self.hd = h0 + dt * Rh
        self._props()
        rho1 = self.rho
        u1, v1, w1, _ = self.project(u0 - dt * Fu, v0 - dt * Fv, w0 - dt * Fw, rho1, rho0, dt, dt)
        Z1, h1 = self.Z, self.hd
        Fu, Fv, Fw, mu1, RZ, Rh = stage(u1, v1, w1, Z1, h1, rho1)
        self.Z = (0.5 * (Z0 + Z1 + dt * RZ)).clamp(0, 1); self.hd = 0.5 * (h0 + h1 + dt * Rh)
        self._props()
        self.u, self.v, self.w, self.Hp = self.project(0.5 * (u0 + u1 - dt * Fu), 0.5 * (v0 + v1 - dt * Fv),
                                                       0.5 * (w0 + w1 - dt * Fw), self.rho, rho0, dt, 0.5 * dt)
        self._numax = ((mu1 / rho1) / self.cfg.Sc_t * 6.0 / self.h ** 2).max().item()
        if not torch.isfinite(self.w).all():
            raise FloatingPointError(f"a LES da fornalha divergiu em t = {self.time:.2f} s")
        self.time += dt
        self.steps += 1
        self.wall += time.perf_counter() - t0
        return dt

    # ------------------------------------------------------------ médias
    def accumulate(self):
        f = dict(T=self.T, Z=self.Z, Sh=self.Sh, u=_avg(self.u, 0), v=_avg(self.v, 1), w=_avg(self.w, 2))
        if self.avg is None:
            self.avg = {k: torch.zeros_like(v) for k, v in f.items()}
            self.avg_n = 0
        for k, v in f.items():
            self.avg[k] += v
        self.avg_n += 1

    def mean(self, k):
        return (self.avg[k] / max(self.avg_n, 1)).cpu().numpy()

    def energy_balance(self) -> dict:
        """Potência liberada, calor cedido ao tambor (∫S_h) e defeito de entalpia que sai na exaustão."""
        cfg = self.cfg
        V = self.h ** 3
        Q_loss = float((self.Sh * V).sum())
        rho_f = self.rho[-1]; u_f = self.u[-1]; fl = self.fluid[-1]
        hd_out = float((rho_f * u_f * self.hd[-1] * fl).sum() * self.h ** 2)
        m_out = float((rho_f * u_f).sum() * self.h ** 2)
        T_out = float((self.T[-1] * fl).sum() / fl.sum().clamp(min=1))
        return dict(Q_fire=cfg.Q_fire, Q_paredes=Q_loss, Q_exaustao=cfg.Q_fire - Q_loss, defeito_saida=hd_out,
                    m_out=m_out, m_in=self.m_fuel + self.m_air, T_max=float(self.T.max()), T_exaustao=T_out)

    def drum_flux_map(self, nphi: int = 36, nxb: int = 7) -> dict:
        """Fluxo no casco por fatia × ângulo, com o gás médio vizinho ao casco (Hottel local)."""
        cfg = self.cfg
        T = self.mean("T") if self.avg is not None else self.T.cpu().numpy()
        mask = self.near_drum.cpu().numpy() > 0
        phi = self.phi_c
        X = np.broadcast_to(self.xc[:, None, None], T.shape)
        xb = np.clip((X / cfg.L_drum * nxb).astype(int), 0, nxb - 1)
        pb = np.clip(((phi + 180.0) / 360.0 * nphi).astype(int), 0, nphi - 1)
        key = (xb * nphi + pb)[mask]
        T4 = np.bincount(key, weights=T[mask] ** 4, minlength=nxb * nphi)
        cnt = np.bincount(key, minlength=nxb * nphi)
        Tg = (T4 / np.maximum(cnt, 1)) ** 0.25
        Tg = np.where(cnt > 0, Tg, np.nan).reshape(nxb, nphi)
        # buracos (setores sem célula na malha grossa): média dos vizinhos em φ
        for _ in range(4):
            bad = ~np.isfinite(Tg)
            if not bad.any():
                break
            nb = np.nanmean(np.stack([np.roll(Tg, 1, 1), np.roll(Tg, -1, 1)]), 0)
            Tg[bad] = nb[bad]
        # forma local pelas relações de Hottel com o gás vizinho; total igual ao calor que a LES tirou do gás
        eg = wsgg_emissivity(Tg, 0.2 * 0.35)
        Ts = cfg.T_shell
        q = SIGMA * eg * (1 + cfg.eps_s) / 2 * (Tg ** 4 - Ts ** 4) + 5.0 * (Tg - Ts)
        A = 2 * np.pi * cfg.R_out / nphi * cfg.L_drum / nxb
        Q_les = float((self.mean("Sh") if self.avg is not None else self.Sh.cpu().numpy()).sum() * self.h ** 3)
        q = q / max(np.nansum(q) * A, 1e-9) * Q_les
        phi_c = -180 + (np.arange(nphi) + 0.5) * 360 / nphi
        return dict(q=q, Tg=Tg, phi=phi_c, x=(np.arange(nxb) + 0.5) * cfg.L_drum / nxb)

    # ------------------------------------------------------- visualização
    def _grad2_fluid(self, q):
        """|∇q|² só com vizinhos fluidos (diferença central, unilateral junto a sólido; o Z̃ das células
        sólidas fica parado no valor inicial e criaria gradientes falsos no casco)."""
        g2 = 0.0
        for ax in range(3):
            qp, fp = _pad(q, ax), _pad(self.fluid, ax)
            fm, fq = _sl(fp, ax, 0, -2), _sl(fp, ax, 2, None)
            d = ((_sl(qp, ax, 2, None) - q) * fq + (q - _sl(qp, ax, 0, -2)) * fm) / ((fq + fm).clamp(min=1.0) * self.h)
            g2 = g2 + d * d
        return g2 * self.fluid

    def flame_fields(self, fv_scale: float = 0.2) -> dict:
        """Campos 3D do estado atual para a renderização das chamas (tamborkit.flames), no dispositivo.

        * f_v: fuligem da relação de estado (tabela β-PDF; pico de 1 ppm em φ ≈ 2) × fv_scale;
        * q: liberação de calor da química rápida [W/m³], q = ρ D |∇Z̃|² · LHV/(1 − Z_st) · P(Z_st)
          (ρχ/2 com χ = 2D|∇Z̃|², D = (ν_t + ν)/Sc_t; P(Z_st) gaussiana com σ² = s Z̃(1 − Z̃) + (Z_st/2)²);
        * velocidade nos centros das células (para advectar o detalhe abaixo da malha) e a máscara fluida."""
        with torch.no_grad():
            Z, Zst = self.Z, float(self.Z_st)
            fv = self._lookup(self.tab_fv, Z, self.s) * fv_scale * self.fluid
            D = (self._nu_t(self.u, self.v, self.w) + 3e-5) / self.cfg.Sc_t
            var = self.s * Z * (1 - Z) + (0.5 * Zst) ** 2
            P = torch.exp(-0.5 * (Z - Zst) ** 2 / var) / torch.sqrt(2 * math.pi * var)
            q = self.rho * D * self._grad2_fluid(Z) * (self.LHV / (1 - Zst)) * P
            return dict(T=self.T, fv=fv, q=q, u=_avg(self.u, 0), v=_avg(self.v, 1), w=_avg(self.w, 2),
                        fluid=self.fluid)

    def run(self, t_end: float, t_avg: float = 0.0, frame_dt: float | None = None, on_frame=None,
            verbose: bool = True):
        next_frame = self.time
        t_print = self.time
        while self.time < t_end:
            self.step()
            if self.time >= t_end - t_avg:
                self.accumulate()
            if frame_dt and on_frame and self.time >= next_frame:
                on_frame(self); next_frame += frame_dt
            if verbose and self.time - t_print >= max(t_end / 10, 1.0):
                t_print = self.time
                eb = self.energy_balance()
                print(f"  LES fornalha t = {self.time:6.2f} s  Δt = {self.dt * 1e3:5.2f} ms  T_máx = {eb['T_max'] - 273:5.0f} °C"
                      f"  ao tambor {eb['Q_paredes'] / 1e3:5.0f} kW  T saída {eb['T_exaustao'] - 273:4.0f} °C"
                      f"  CG {self.cg_iters:2d}  ṁ saída/entrada {eb['m_out'] / eb['m_in']:4.2f}  massa {float((self.rho * self.fluid).sum()) * self.h ** 3:6.2f} kg"
                      f"  ({self.wall:5.0f} s de relógio, {self.n_cells / 1e3:.0f} mil células)", flush=True)

In [ ]:
%%writefile tamborkit/flames.py
"""Chamas da fornalha com base física: emissão da fuligem e quimiluminescência integradas na linha de visada.

Pós-processamento dos quadros 3D gravados da LES (furnace_les.FurnaceLES), no espírito do render HD do flare:

* fuligem: f_v da relação de estado (tabela β-PDF; pico 1 ppm em φ ≈ 2) × fv_scale. Queimadores de GLP
  com ar forçado (bico de mistura) são pouco fuliginosos: f_v de pico ~0,1–0,3 ppm (chamas de propano
  difusivas chegam a ~1 ppm; a mistura parcial do ar forçado reduz) → fv_scale = 0,2 por padrão;
* absorção da fuligem κ_s = C f_v T, C = 1862 m⁻¹K⁻¹ (média de Planck; a mesma da LES) e, por célula da
  linha de visada, dτ = κ_s Δs; emissão visível ∝ L_vis(T)·(1 − e^(−dτ)), composta da frente para trás
  com a transmitância acumulada. L_vis(T) e a cor saem do espectro λ^(−1,39)·B_λ(T) integrado nas funções
  de cor CIE 1931 (a mesma tabela do flare): ~1400 K laranja-avermelhado, ~1800 K amarelo, ~2200 K branco;
* quimiluminescência azul (CH*, C₂*) proporcional à liberação de calor da química rápida
  q = ρχ/2 · LHV/(1 − Z_st) · P(Z_st), fina e sem absorção, cor sRGB linear ≈ (0,22; 0,42; 1,0), só onde q
  passa de 10 % do pico (sem véu azul do gás morno). Sem medida de referência para a razão azul/fuligem, o
  ganho é visual: ~1 MW/m² de calor por área projetada vale ~0,3 % da luminância de um corpo negro a 1800 K
  (parâmetro `blue`);
* resolução abaixo da célula: T, f_v e q interpolados (trilinear) numa grade ~3 cm antes da emissão — como a
  luminância cresce ~T^20, as bordas da chama ficam finas em vez de quadradas;
* superfícies (vista lateral): casco de aço e refratário com a incandescência do próprio T (ponto de
  Draper ≈ 798 K, vermelho escuro quase invisível; ~1100 K cereja/laranja) e iluminados de forma difusa
  pela luz da chama (soma das células emissoras, ∝ cos θ/r², sem sombra);
* detalhe abaixo da malha (opcional): ruído fractal advectado com a velocidade projetada (advecção de
  textura em duas fases), que deforma e modula a chama só na escala da célula — efeito visual;
* exposição automática pelo percentil 99,9 da luminância (mediana móvel de ±20 quadros, independente da
  ordem de renderização), compressão HDR (radiância^0,5), bloom em duas escalas, ACES e sRGB.

Custo (CPU, 2 núcleos, malha de 5 cm 108×40×45): projeção ~0,2 s por quadro (uma vez, no início); depois
~0,2–0,4 s por quadro de 1080p (amostragem por pixel e bloom). Na GPU, ~10× mais rápido.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F

C_SOOT = 1862.0                  # κ_s = C f_v T [1/(m K)] (média de Planck; a mesma da LES)
BLUE_RGB = (0.22, 0.42, 1.0)     # CH* 431 nm + C₂* 516 nm (bandas de Swan), sRGB linear
T_DRAPER = 798.0                 # K: início do brilho visível no escuro


# ============================================================ gravação dos quadros
def _pool(a: torch.Tensor, stride: int) -> torch.Tensor:
    if stride <= 1:
        return a
    nx, ny, nz = (n // stride * stride for n in a.shape)
    return F.avg_pool3d(a[None, None, :nx, :ny, :nz].float(), stride)[0, 0]


def flame_meta(les, stride: int = 1, fv_scale: float = 0.2) -> dict:
    """Metadados estáticos dos quadros: coordenadas dos centros, h, geometria do tambor e da caixa."""
    c = les.cfg

    def coords(v):
        n = len(v) // stride * stride
        return np.asarray(v[:n], float).reshape(-1, stride).mean(1)
    fluid = (_pool(les.fluid, stride) > 0.5).cpu().numpy().astype(np.uint8)
    return dict(xc=coords(les.xc), yc=coords(les.yc), zc=coords(les.zc), h=float(les.h) * stride, stride=stride,
                R_out=float(c.R_out), L_drum=float(c.L_drum), half_width=float(c.half_width), z_bot=float(c.z_bot),
                z_top=float(c.z_top), Lx=float(les.xf[-1]), burners_x=tuple(c.burners_x), burner_z=float(c.burner_z),
                Q_fire=float(c.Q_fire), T_shell=float(c.T_shell), T_refr=float(c.T_refr), Z_st=float(les.Z_st),
                fv_scale=float(fv_scale), fluid=fluid)


def record_flame_frame(les, fv_scale: float = 0.2, stride: int = 1, fluid=None) -> dict:
    """Quadro 3D do estado atual da LES, em float16: T [K], f_v [ppm], q [MW/m³], u, v, w [m/s].

    `fluid` (uint8) é a máscara fluida estática; passe sempre o mesmo objeto (o pickle guarda uma vez só)."""
    f = les.flame_fields(fv_scale)
    cv = lambda a, s=1.0: (_pool(a, stride) * s).cpu().numpy().astype(np.float16)  # noqa: E731
    if fluid is None:
        fluid = (_pool(les.fluid, stride) > 0.5).cpu().numpy().astype(np.uint8)
    return dict(t=float(les.time), T=cv(f["T"]), fv=cv(f["fv"], 1e6), q=cv(f["q"], 1e-6), u=cv(f["u"]),
                v=cv(f["v"]), w=cv(f["w"]), fluid=fluid)


class FlameRecorder:
    """on_frame da LES: grava quadros a partir de t_start. Uso:
    rec = FlameRecorder(les, t_start=10.0); les.run(12.0, frame_dt=0.05, on_frame=rec)."""

    def __init__(self, les, t_start: float = 0.0, fv_scale: float = 0.2, stride: int = 1):
        self.t_start, self.fv_scale, self.stride = t_start, fv_scale, stride
        self.meta = flame_meta(les, stride, fv_scale)
        self.frames: list[dict] = []

    def __call__(self, les):
        if les.time >= self.t_start:
            self.frames.append(record_flame_frame(les, self.fv_scale, self.stride, self.meta["fluid"]))

    def data(self) -> dict:
        return dict(frames=self.frames, meta=self.meta)


# ============================================================ cor (CIE 1931 + Planck), portado do flare
def _cie_cmf(lam_nm: np.ndarray) -> np.ndarray:
    """Funções de cor CIE 1931 2° (ajuste analítico de Wyman, Sloan & Shirley, 2013)."""
    def g(x, mu, s1, s2):
        s = np.where(x < mu, s1, s2)
        return np.exp(-0.5 * ((x - mu) / s) ** 2)
    x = 1.056 * g(lam_nm, 599.8, 37.9, 31.0) + 0.362 * g(lam_nm, 442.0, 16.0, 26.7) \
        - 0.065 * g(lam_nm, 501.1, 20.4, 26.2)
    y = 0.821 * g(lam_nm, 568.8, 46.9, 40.5) + 0.286 * g(lam_nm, 530.9, 16.3, 31.1)
    z = 1.217 * g(lam_nm, 437.0, 11.8, 36.0) + 0.681 * g(lam_nm, 459.0, 26.0, 13.8)
    return np.stack([x, y, z], -1)


XYZ_TO_SRGB = np.array([[3.2406, -1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])


def radiance_lut(T0: float = 300.0, T1: float = 3000.0, n: int = 541, alpha: float = 1.39):
    """Tabela T → (cromaticidade sRGB linear, luminância visível Y) da fuligem (ε_λ ∝ λ^−α, normalizada em
    550 nm) e de um corpo negro (superfícies). Y em unidades arbitrárias comuns (as duas comparáveis)."""
    lam = np.linspace(380e-9, 780e-9, 401)
    cmf = _cie_cmf(lam * 1e9)
    T = np.linspace(T0, T1, n)
    c2 = 1.4388e-2
    out = {}
    for key, a in (("soot", alpha), ("bb", 0.0)):
        rgb = np.zeros((n, 3)); Y = np.zeros(n)
        for i, t in enumerate(T):
            spec = (lam / 550e-9) ** (-a) * lam ** -5.0 / np.expm1(np.minimum(c2 / (lam * t), 700.0))
            xyz = (spec[:, None] * cmf).sum(0)
            rgb[i] = np.clip(XYZ_TO_SRGB @ xyz, 0.0, None)
            Y[i] = xyz[1]
        rgb = rgb / np.maximum(rgb.max(1, keepdims=True), 1e-30)
        out[key] = (rgb, Y)
    ref = np.interp(1800.0, T, out["soot"][1])
    return T, out["soot"][0], out["soot"][1] / ref, out["bb"][0], out["bb"][1] / ref


_LUT = None


def _lut():
    global _LUT
    if _LUT is None:
        _LUT = radiance_lut()
    return _LUT


def incandescence_rgb(T_K):
    """Cor (sRGB linear) × luminância relativa de uma superfície negra a T (refratário, casco). Abaixo do ponto
    de Draper (~798 K) é praticamente invisível; ~1100 K cereja/laranja."""
    T, _, _, rgb, Y = _lut()
    Tq = np.clip(np.asarray(T_K, float), T[0], T[-1])
    c = np.stack([np.interp(Tq, T, rgb[:, i]) for i in range(3)], -1)
    return c * np.interp(Tq, T, Y)[..., None]


def aces(x: torch.Tensor) -> torch.Tensor:
    """Tone mapping fílmico ACES (aproximação de Narkowicz)."""
    return ((x * (2.51 * x + 0.03)) / (x * (2.43 * x + 0.59) + 0.14)).clamp(0.0, 1.0)


def to_srgb(x: torch.Tensor) -> torch.Tensor:
    return torch.where(x <= 0.0031308, 12.92 * x, 1.055 * x.clamp(min=0.0) ** (1 / 2.4) - 0.055)


def _gauss_kernel(sigma: float, dev, dtype):
    r = max(1, int(3 * sigma))
    x = torch.arange(-r, r + 1, device=dev, dtype=dtype)
    k = torch.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def blur(img: torch.Tensor, sigma: float) -> torch.Tensor:
    """Desfoque gaussiano separável de (C, H, W); sigmas grandes via pirâmide."""
    if sigma < 0.5:
        return img
    x = img[None]
    down = 1
    while sigma / down > 12 and min(x.shape[-2:]) > 64:
        x = F.avg_pool2d(x, 2)
        down *= 2
    k = _gauss_kernel(sigma / down, img.device, img.dtype)
    C = x.shape[1]
    pad = len(k) // 2
    x = F.conv2d(F.pad(x, (pad, pad, 0, 0), mode="replicate"), k.view(1, 1, 1, -1).expand(C, 1, 1, -1), groups=C)
    x = F.conv2d(F.pad(x, (0, 0, pad, pad), mode="replicate"), k.view(1, 1, -1, 1).expand(C, 1, -1, 1), groups=C)
    if down > 1:
        x = F.interpolate(x, size=img.shape[-2:], mode="bilinear", align_corners=False)
    return x[0]


def fractal_noise(n: int, lam_min: float, lam_max: float, beta: float, gen: torch.Generator) -> torch.Tensor:
    """Ruído periódico n×n com espectro ∝ k^-β entre os comprimentos de onda lam_min e lam_max (em px)."""
    w = torch.randn(n, n, generator=gen)
    k = torch.fft.fftfreq(n)
    kk = torch.sqrt(k[:, None] ** 2 + k[None, :] ** 2)
    band = (kk >= 1.0 / lam_max) & (kk <= 1.0 / lam_min)
    amp = torch.where(band, kk.clamp(min=1e-6) ** (-beta / 2), torch.zeros_like(kk))
    f = torch.fft.ifft2(torch.fft.fft2(w) * amp).real
    return (f - f.mean()) / f.std()


# ============================================================ projeção ao longo da linha de visada
def _lut_torch(dev):
    """(T0, ΔT, L_fuligem [n, 3], L_corpo_negro [n, 3]) como tensores: radiância visível relativa já com a cor."""
    T, c_s, Y_s, c_b, Y_b = _lut()
    ls = torch.as_tensor(c_s * Y_s[:, None], dtype=torch.float32, device=dev)
    lb = torch.as_tensor(c_b * Y_b[:, None], dtype=torch.float32, device=dev)
    return float(T[0]), float(T[1] - T[0]), ls, lb


def _lookup(table, T, T0, dT):
    n = table.shape[0]
    x = ((T - T0) / dT).clamp(0.0, n - 1.001)
    i0 = x.floor().long()
    f = (x - i0)[..., None]
    return table[i0] * (1.0 - f) + table[i0 + 1] * f


def project(frame: dict, meta: dict, view: str = "corte", blue: float = 0.003, T_shell: float | None = None,
            T_refr: float | None = None, up: int = 3, device: str = "cpu"):
    """Integra emissão e absorção ao longo da visada. Devolve (rgb linear das chamas [A, Z, 3], opacidade [A, Z],
    rgb linear das superfícies vistas [A, Z, 3] — zero na vista 'corte') com
    A = y (vista 'corte', visada +x) ou x (vista 'lateral', visada +y a partir da parede dos queimadores), numa
    grade `up` vezes mais fina que a da LES nos dois eixos da imagem (centros em a0 − h/2 + (j + ½)·h/up).

    Resolução abaixo da célula: T, f_v e q são interpolados (trilinear) antes de calcular a emissão; como a
    luminância visível cresce ~T^20 perto de 1500 K, isso afina as bordas da chama em vez de borrar a emissão
    célula a célula.

    Vista lateral: o raio para no casco do tambor (opaco), que entra com a sua incandescência e iluminado pela
    luz das chamas (difusa, ρ/π·L·Ω ~ 1 % da radiância da chama, mais a luz local borrada); a face de baixo,
    voltada para os queimadores, recebe mais luz (sombreamento de cilindro). Se o raio atravessa tudo, aparece
    o refratário do fundo (incandescência de T_refr, também iluminado)."""
    dev = torch.device(device)
    T0, dT, L_s, _ = _lut_torch(dev)
    h = float(meta["h"])
    fl0 = frame.get("fluid", meta["fluid"])
    f = torch.stack([torch.as_tensor(np.asarray(a, np.float32), device=dev)
                     for a in (frame["T"], frame["fv"], frame["q"], fl0)])
    ax = 0 if view == "corte" else 1
    img_axes = (1, 2) if view == "corte" else (0, 2)
    size = list(f.shape[1:])
    for a in img_axes:
        size[a] *= up
    f = F.interpolate(f[None], size=size, mode="trilinear", align_corners=False)[0]
    T, fv, q, fl = f[0], f[1].clamp(min=0.0) * 1e-6, f[2].clamp(min=0.0), f[3].clamp(0.0, 1.0)

    def centres(c, n):
        c = np.asarray(c, float)
        a0 = c[0] - h / 2
        return torch.as_tensor(a0 + (np.arange(n) + 0.5) * (h * len(c) / n), dtype=torch.float32, device=dev)
    xs, ys, zs = (centres(meta[k], size[i]) for i, k in enumerate(("xc", "yc", "zc")))

    dtau = C_SOOT * fv * T * h * fl
    emis = (1.0 - torch.exp(-dtau))[..., None] * _lookup(L_s, T, T0, dT)
    # quimiluminescência só onde a liberação de calor passa de 10 % do pico (sem véu azul do gás morno), com o pico
    # limitado ao percentil 99,5 (a borda do jato na boca do queimador tem |∇Z|² de uma célula só)
    qs = q.flatten()[::max(1, q.numel() // 200000)]
    q_hi = float(torch.quantile(qs, 0.995)) + 1e-30
    qe = (q.clamp(max=q_hi) - 0.10 * q_hi).clamp(min=0.0) * fl
    if view == "lateral":
        X, Y, Z = torch.meshgrid(xs, ys, zs, indexing="ij")
        drum = (torch.hypot(Y, Z) < meta["R_out"]) & (X < meta["L_drum"])
        blocked = torch.cumsum(drum.int(), dim=1) > 0                   # da parede dos queimadores para dentro
        prev = torch.cat([torch.zeros_like(blocked[:, :1]), blocked[:, :-1]], 1)
        first = blocked & ~prev
        dtau = torch.where(blocked, torch.zeros_like(dtau), dtau)
        emis = torch.where(blocked[..., None], torch.zeros_like(emis), emis)
    cum = torch.cumsum(dtau, dim=ax)
    trans_before = torch.exp(-(cum - dtau))
    rgb = (emis * trans_before[..., None]).sum(dim=ax)
    trans = torch.exp(-cum.select(ax, -1))
    if blue > 0:
        if view == "lateral":
            qe = torch.where(blocked, torch.zeros_like(qe), qe)
        bl = blur((blue * h * (qe * trans_before).sum(dim=ax))[None], 0.6 * up)[0]   # posição da folha: ± ½ célula
        rgb = rgb + bl[..., None] * torch.tensor(BLUE_RGB, device=dev)
    surf_rgb = torch.zeros_like(rgb)
    if view == "lateral":
        T_sh = meta["T_shell"] if T_shell is None else T_shell
        T_rf = meta["T_refr"] if T_refr is None else T_refr
        hit = first.any(dim=1)
        t_hit = (trans_before * first).sum(dim=1)
        lum = rgb.sum(-1)
        peak = float(torch.quantile(lum.flatten()[::max(1, lum.numel() // 200000)], 0.995)) + 1e-30
        sig = 0.35 / (h / up)                                           # luz local: ~35 cm de alcance
        glow = blur(rgb.permute(2, 0, 1), sig).permute(1, 2, 0).sum(-1, keepdim=True)
        nz = (zs / meta["R_out"]).clamp(-1, 1)[None, :, None]           # normal do casco no ponto visto (z/R)
        shade = 0.35 + 0.65 * (-nz).clamp(min=0.0)
        steel = torch.tensor([0.50, 0.52, 0.56], device=dev)
        brick = torch.tensor([0.60, 0.33, 0.22], device=dev)
        inc_sh = torch.as_tensor(incandescence_rgb(T_sh), dtype=torch.float32, device=dev)
        inc_rf = torch.as_tensor(incandescence_rgb(T_rf), dtype=torch.float32, device=dev)
        surf = inc_sh + (0.01 * peak + 0.25 * glow) * shade * steel
        back = inc_rf + (0.006 * peak + 0.10 * glow) * brick
        surf_rgb = torch.where(hit[..., None], t_hit[..., None] * surf, trans[..., None] * back)
        trans = torch.where(hit, torch.zeros_like(trans), trans)
    return (rgb.cpu().numpy().astype(np.float32), (1.0 - trans).cpu().numpy().astype(np.float32),
            surf_rgb.cpu().numpy().astype(np.float32))


class FlameRenderer:
    """Quadros das chamas a partir dos quadros 3D gravados (FlameRecorder.data())."""

    def __init__(self, data: dict, view: str = "corte", exposure: float = 1.0, bloom: float = 0.5,
                 detail: float = 0.15, blue: float = 0.003, seed: int = 3, device: str = "cpu", gamma: float = 0.5,
                 up: int | None = None):
        self.frames, self.meta = data["frames"], data["meta"]
        self.gamma = gamma
        self.view, self.exposure, self.bloom, self.detail, self.blue = view, exposure, bloom, detail, blue
        self.dev = torch.device(device)
        m = self.meta
        # grade da imagem ~3 cm: 0,10 m → ×3; 0,05 m → ×2; 0,035 m → ×1
        self.up = up if up is not None else max(1, int(round(m["h"] / 0.033)))
        self.proj = [project(f, m, view, blue, up=self.up, device=device) for f in self.frames]
        lum = np.array([float(np.percentile(p[0].sum(-1), 99.9)) for p in self.proj])   # pelas chamas
        k = 20
        self.norm = np.array([np.median(lum[max(0, i - k):i + k + 1]) for i in range(len(lum))])
        gen = torch.Generator().manual_seed(seed)
        self.noise = [fractal_noise(256, 3.0, 40.0, 3.0, gen) for _ in range(2)]
        self.hp = m["h"] / self.up
        self.ax0 = np.asarray(m["yc"] if view == "corte" else m["xc"])
        self.ax1 = np.asarray(m["zc"])

    def _sample(self, img: np.ndarray, A, B):
        """Amostra bilinear (sem sobressinal nas bordas) de img [n0, n1, C] nas coordenadas físicas (A, B) dos pixels."""
        t = torch.as_tensor(img, dtype=torch.float32, device=self.dev).permute(2, 0, 1)
        t = blur(t, 0.8)
        n0, n1 = img.shape[:2]
        h = self.meta["h"]
        a0, b0 = self.ax0[0] - h / 2, self.ax1[0] - h / 2
        gx = 2 * ((torch.as_tensor(np.array(B), dtype=torch.float32) - b0) / (n1 * self.hp)) - 1
        gy = 2 * ((torch.as_tensor(np.array(A), dtype=torch.float32) - a0) / (n0 * self.hp)) - 1
        grid = torch.stack([gx, gy], -1)[None].to(self.dev)
        out = F.grid_sample(t[None], grid, mode="bilinear", padding_mode="zeros", align_corners=False)[0]
        return out.clamp(min=0.0)

    def _detail(self, A, B, i):
        """Modulação abaixo da malha (efeito visual): ruído fractal com comprimentos de onda de ~3 a 33 cm,
        em duas fases que sobem com o empuxo (~1,5 m/s) e se alternam sem emenda."""
        texel = 120.0                                       # texels por metro
        t = i / 30.0
        P = 0.8
        ph = (t % P) / P
        A_t = torch.as_tensor(np.array(A), dtype=torch.float32)
        B_t = torch.as_tensor(np.array(B), dtype=torch.float32)
        tot = 0.0
        for k, off in enumerate((0.0, 0.5)):
            s = (ph + off) % 1.0
            wgt = 1 - abs(2 * s - 1)
            u = torch.remainder(A_t * texel, 256.0)
            v = torch.remainder((B_t - 1.2 * s * P) * texel + 97.0 * k, 256.0)
            nz = F.pad(self.noise[k][None, None], (0, 1, 0, 1), mode="circular")
            grid = torch.stack([(v + 0.5) / 257 * 2 - 1, (u + 0.5) / 257 * 2 - 1], -1)[None]
            tot = tot + wgt * F.grid_sample(nz, grid, mode="bilinear", align_corners=False)[0, 0]
        return (1.0 + self.detail * tot).clamp(min=0.0).to(self.dev)

    def render_on_grid(self, i: int, A, B, gain: float = 1.0):
        """(rgb [H, W, 3] em 0–1 com tone mapping, opacidade [H, W]) nas coordenadas físicas A (y ou x), B (z)."""
        i = i % len(self.proj)
        rgb, op, surf = self.proj[i]
        img = np.concatenate([rgb, op[..., None], surf], -1)
        s = self._sample(img, A, B)
        lin, alpha = s[:3], s[3].clamp(0, 1)
        if self.detail > 0:
            lin = lin * self._detail(A, B, i)[None]        # o detalhe só deforma a chama, não as superfícies
        lin = lin + s[4:]
        # compressão da faixa dinâmica (câmera HDR): radiância^γ antes do tone mapping, para a chama não virar
        # só o pico mais quente (a luminância visível cresce ~T^20 perto de 1500 K)
        lin = (lin * (gain / max(self.norm[i], 1e-30))).clamp(min=0.0) ** self.gamma * (self.exposure * 1.6)
        if self.bloom > 0:
            bright = (lin - 0.6).clamp(min=0.0)
            lin = lin + self.bloom * (0.6 * blur(bright, 6.0) + 0.4 * blur(bright, 22.0))
        out = to_srgb(aces(lin)).permute(1, 2, 0).cpu().numpy()
        a = np.clip(np.asarray(alpha.cpu().numpy()) * 8.0 + out.max(-1), 0, 1)
        return out, a

    def render_image(self, i: int, extent: tuple, size: tuple, gain: float = 1.0):
        """Imagem isolada (W×H) cobrindo extent = (a0, a1, b0, b1) em metros (a horizontal, b vertical)."""
        W, H = size
        a0, a1, b0, b1 = extent
        A = np.ascontiguousarray(np.broadcast_to(np.linspace(a0, a1, W)[None, :], (H, W)))
        B = np.ascontiguousarray(np.broadcast_to(np.linspace(b1, b0, H)[:, None], (H, W)))
        return self.render_on_grid(i, A, B, gain)

In [ ]:
%%writefile tamborkit/video.py
"""Vídeo em HD (1080p): cartões de título, texto e números-chave, figuras com legenda e montagem final.

Mesma linguagem visual do estudo do flare: fundo escuro, textos com PIL (vírgula decimal), entrada suave e
codificação H.264 pelo ffmpeg, quadro a quadro, sem passar pelo matplotlib.
"""
from __future__ import annotations

import math
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np

_DEC = re.compile(r"(?<=\d)\.(?=\d)")


def br(text: str) -> str:
    """Vírgula decimal (norma brasileira) em todo número de um texto: '6.8 m/s' → '6,8 m/s'."""
    return _DEC.sub(",", text) if isinstance(text, str) else text


def _font(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    path = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name
    try:
        return ImageFont.truetype(str(path), size)
    except OSError:
        return ImageFont.load_default()


def _mono(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSansMono-Bold.ttf" if bold else "DejaVuSansMono.ttf"
    try:
        return ImageFont.truetype(str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name), size)
    except OSError:
        return _font(size, bold)


class FFmpegPipe:
    """Escreve quadros RGB (H, W, 3) uint8 direto no ffmpeg (H.264, CRF baixo, yuv420p)."""

    def __init__(self, path: str, size: tuple, fps: int = 30, crf: int = 14):
        ff = shutil.which("ffmpeg")
        if ff is None:
            raise RuntimeError("ffmpeg não encontrado")
        W, H = size
        self.p = subprocess.Popen(
            [ff, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
             "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "slow", "-crf", str(crf),
             "-tune", "film", "-pix_fmt", "yuv420p", "-movflags", "+faststart", path],
            stdin=subprocess.PIPE)
        self.path = path

    def write(self, frame: np.ndarray):
        self.p.stdin.write(np.ascontiguousarray(frame, dtype=np.uint8).tobytes())

    def close(self):
        self.p.stdin.close()
        if self.p.wait() != 0:
            raise RuntimeError(f"ffmpeg falhou ao gravar {self.path}")
        return self.path



_GLUE = re.compile(r"^(\([A-Z]{2}\)|m|m/s|s|kg/h|kg/s|kW|MW|t/a|W/m²K|Pa\u00a0s|mPa\u00a0s|rpm|°C|K|%|h)[.,;:)]?$")


def _words(text):
    """Palavras para quebra de linha, sem separar '(SP)' do nome nem a unidade do número."""
    out = []
    for w in text.split():
        if out and _GLUE.match(w):
            out[-1] += "\u00a0" + w
        else:
            out.append(w)
    return out


def _wrap(draw, text, font, width):
    words, lines, cur = _words(text), [], ""
    for w in words:
        nxt = (cur + " " + w).strip()
        if draw.textlength(nxt, font=font) <= width:
            cur = nxt
        else:
            if cur:
                lines.append(cur)
            cur = w
    return lines + ([cur] if cur else [])


def _wrap_balanced(draw, text, font, width):
    """Como _wrap, mas com linhas de comprimento parecido (evita uma palavra sozinha na última linha)."""
    lines = _wrap(draw, text, font, width)
    if len(lines) < 2:
        return lines
    lo, hi = 0.3 * width, float(width)
    for _ in range(18):                      # menor largura que mantém o mesmo número de linhas
        mid = 0.5 * (lo + hi)
        if len(_wrap(draw, text, font, mid)) <= len(lines):
            hi = mid
        else:
            lo = mid
    return _wrap(draw, text, font, hi)


def _caption_alpha(captions, t, fade=0.45):
    """Legenda ativa no instante t (s) e sua opacidade (entrada e saída suaves)."""
    for t0, t1, text in captions:
        if t0 <= t < t1:
            return text, float(np.clip(min(t - t0, t1 - t) / fade, 0, 1))
    return None, 0.0


def even_captions(texts, duration, lead=0.3):
    """Distribui as legendas igualmente na duração do segmento: [(t0, t1, texto)]."""
    texts = [t for t in texts if t]
    if not texts:
        return []
    d = (duration - lead) / len(texts)
    return [(lead + k * d, lead + (k + 1) * d, t) for k, t in enumerate(texts)]


def draw_lower_third(im, text, alpha, s, accent=(255, 176, 64), y_bottom=None, max_w=None, x0=None):
    """Legenda em caixa escura translúcida no terço inferior (PIL RGBA)."""
    from PIL import Image, ImageDraw
    if not text or alpha <= 0:
        return im
    W, H = im.size
    font = _font(int(34 * s))
    layer = Image.new("RGBA", im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    max_w = max_w or int(W * 0.62)
    lines = _wrap(d, br(text), font, max_w)
    lh = int(font.size * 1.28)
    pad = int(22 * s)
    box_h = lh * len(lines) + 2 * pad
    box_w = max(d.textlength(ln, font=font) for ln in lines) + 2 * pad + int(10 * s)
    y1 = y_bottom if y_bottom is not None else H - int(150 * s)
    y0 = y1 - box_h
    x0 = int(60 * s) if x0 is None else x0
    d.rounded_rectangle([x0, y0, x0 + box_w, y1], radius=int(10 * s), fill=(6, 9, 15, int(185 * alpha)))
    d.rectangle([x0, y0, x0 + int(6 * s), y1], fill=accent + (int(255 * alpha),))
    for k, ln in enumerate(lines):
        d.text((x0 + pad + int(10 * s), y0 + pad + k * lh), ln, font=font, fill=(240, 244, 250, int(255 * alpha)))
    im.alpha_composite(layer)
    return im



def _gradient(W: int, H: int) -> np.ndarray:
    bg = np.zeros((H, W, 3), np.float32)
    yy = np.linspace(0, 1, H)[:, None]
    for c, (a, b) in enumerate(zip((8, 11, 20), (14, 20, 34))):
        bg[..., c] = a + (b - a) * yy
    return bg


def _bright_cx(im: np.ndarray) -> float:
    """Posição horizontal (fração da largura) do centro luminoso da imagem."""
    lum = np.asarray(im, np.float32).mean(2)
    wgt = np.clip(lum - np.percentile(lum, 50), 0, None)
    return float((wgt.sum(0) * np.arange(lum.shape[1])).sum() / max(wgt.sum(), 1e-6)) / lum.shape[1]


def _compose(bg: np.ndarray, im: np.ndarray, place: float, strength: float, ramp_from: float,
             cx: float) -> np.ndarray:
    """Chama deslocada para a fração `place` da largura, esmaecida e com rampa da esquerda para a direita."""
    W = bg.shape[1]
    im = np.asarray(im, np.float32)
    sh = int((place - cx) * W)
    out = np.zeros_like(im)
    if sh >= 0:
        out[:, sh:] = im[:, :W - sh]
    else:
        out[:, :W + sh] = im[:, -sh:]
    ramp = np.clip((np.linspace(0, 1, W) - ramp_from) / 0.35, 0, 1)[None, :, None] ** 1.5
    return np.clip(np.maximum(bg, strength * ramp * out), 0, 255)


def _card_background(size, backdrop=None, place: float = 0.78, strength: float = 0.5,
                     ramp_from: float = 0.42) -> np.ndarray:
    """Fundo escuro em gradiente; com `backdrop` (quadro HD do reator, sem textos), a imagem é deslocada
    para a fração `place` da largura e entra esmaecida, sem competir com o texto à esquerda."""
    W, H = size
    bg = _gradient(W, H)
    if backdrop is not None:
        from PIL import Image
        im = np.asarray(Image.fromarray(np.asarray(backdrop, np.uint8)).resize((W, H), Image.LANCZOS), np.float32)
        bg = _compose(bg, im, place, strength, ramp_from, _bright_cx(im))
    return np.clip(bg, 0, 255)


def _probe(src: str):
    """(largura, altura, duração [s]) de um vídeo, lidos da saída do ffmpeg."""
    txt = subprocess.run([shutil.which("ffmpeg"), "-i", src], capture_output=True, text=True).stderr
    m = re.search(r", (\d{2,5})x(\d{2,5})", txt)
    d = re.search(r"Duration: (\d+):(\d+):([\d.]+)", txt)
    dur = 3600 * int(d.group(1)) + 60 * int(d.group(2)) + float(d.group(3)) if d else 0.0
    return int(m.group(1)), int(m.group(2)), dur


class Backdrop:
    """Fundo dos cartões. Aceita None, um quadro RGB (fundo fixo) ou um vídeo limpo (sem textos) do
    reator girando: aí a imagem se move atrás dos textos, em câmera lenta (`speed`) e em
    laço sem emenda (fusão de `cross` quadros). O relógio continua de um segmento para o outro. Os quadros
    ficam na memória em resolução reduzida (`width`), o bastante para um fundo esmaecido."""

    def __init__(self, src=None, size=(1920, 1080), speed: float = 0.5, width: int = 960, max_mb: float = 500.0,
                 cross: int = 24):
        from PIL import Image
        self.size, self.speed, self.k = tuple(size), speed, 0
        self.still, self.frames, self._cache = None, None, {}
        if src is None:
            return
        if not isinstance(src, (str, Path)):
            self.still = np.asarray(src, np.uint8)
            return
        src = str(src)
        if src.lower().endswith((".png", ".jpg", ".jpeg")):
            self.still = np.asarray(Image.open(src).convert("RGB"))
            return
        w0, h0, dur = _probe(src)
        bw = min(width, w0)
        bh = max(2, int(round(h0 * bw / w0 / 2)) * 2)
        nmax = max(8, int(max_mb * 1e6 / (bw * bh * 3)))
        fps = min(30.0, nmax / max(dur, 1e-3))          # clipes longos são subamostrados para caber
        p = subprocess.Popen([shutil.which("ffmpeg"), "-loglevel", "error", "-i", src, "-vf",
                              f"fps={fps:.5f},scale={bw}:{bh}", "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                             stdout=subprocess.PIPE)
        n, fr = bw * bh * 3, []
        while len(fr) < nmax:
            buf = p.stdout.read(n)
            if len(buf) < n:
                break
            fr.append(np.frombuffer(buf, np.uint8).reshape(bh, bw, 3))
        p.stdout.close()
        p.wait()
        if not fr:
            raise ValueError(f"nenhum quadro lido de {src}")
        self.frames = fr
        self.speed = speed * fps / 30.0                  # mesma velocidade aparente com quadros mais espaçados
        self.cross = min(cross, len(fr) // 3)
        self.period = len(fr) - self.cross
        self.cx = _bright_cx(np.mean([f.astype(np.float32) for f in fr[::max(1, len(fr) // 8)]], axis=0))
        self.grad = _gradient(bw, bh)

    @property
    def animated(self) -> bool:
        return self.frames is not None

    def _seq(self, j: int) -> np.ndarray:
        """Quadro j do laço: os primeiros `cross` quadros fundem o fim do clipe ao começo."""
        F, C, Lp = self.frames, self.cross, self.period
        j %= Lp
        if j < C:
            w = j / C
            return (1 - w) * F[Lp + j].astype(np.float32) + w * F[j].astype(np.float32)
        return F[j].astype(np.float32)

    def next(self, place: float = 0.78, strength: float = 0.5, ramp_from: float = 0.42) -> np.ndarray:
        """Próximo fundo (uint8, H×W×3); cada chamada avança um quadro do vídeo final."""
        from PIL import Image
        if not self.animated:
            key = (place, strength, ramp_from)
            if key not in self._cache:
                self._cache[key] = _card_background(self.size, self.still, place, strength,
                                                    ramp_from).astype(np.uint8)
            return self._cache[key]
        pos = self.k * self.speed
        self.k += 1
        i0, w = int(pos), pos - int(pos)
        im = (1 - w) * self._seq(i0) + w * self._seq(i0 + 1)
        bg = _compose(self.grad, im, place, strength, ramp_from, self.cx)
        return np.asarray(Image.fromarray(bg.astype(np.uint8)).resize(self.size, Image.BILINEAR))


def _backdrop(backdrop, size) -> Backdrop:
    return backdrop if isinstance(backdrop, Backdrop) else Backdrop(backdrop, size)


def still_of(src: str, path: str, t: float) -> str:
    """Salva o quadro do instante t [s] de um vídeo como PNG (ex.: capa do post)."""
    subprocess.run([shutil.which("ffmpeg"), "-loglevel", "error", "-y", "-ss", f"{t:.2f}", "-i", src,
                    "-frames:v", "1", path], check=True)
    return path


def render_title_card(path: str, title: str, subtitle: str, credit: str = "", note: str = "",
                      size=(1920, 1080), fps: int = 30, seconds: float = 4.5, backdrop=None,
                      accent=(255, 176, 64), contact: str = "") -> str:
    """Abertura (ou encerramento) em HD: título e subtítulo em linhas equilibradas, autoria, contato
    (ex.: endereço do LinkedIn) e aviso, com entrada suave. backdrop: None, quadro RGB ou Backdrop."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bd = _backdrop(backdrop, size)
    probe = ImageDraw.Draw(Image.new("RGBA", (8, 8)))
    title, subtitle, credit, note = br(title), br(subtitle), br(credit), br(note)
    f_t, f_s, f_c, f_n = _font(int(72 * s), True), _font(int(36 * s)), _font(int(28 * s)), _font(int(22 * s))
    t_lines = _wrap_balanced(probe, title, f_t, int(0.86 * W))
    s_lines = _wrap_balanced(probe, subtitle, f_s, int(0.80 * W))
    blocks = []      # (linhas, fonte, cor, altura de linha, atraso)
    blocks.append((t_lines, f_t, (240, 244, 250), int(f_t.size * 1.18), 0.0))
    blocks.append(([""], f_c, (0, 0, 0), int(24 * s), 0.0))
    blocks.append((s_lines, f_s, accent, int(f_s.size * 1.3), 0.4))
    if credit:
        blocks.append(([""], f_c, (0, 0, 0), int(30 * s), 0.0))
        blocks.append(([credit], f_c, (178, 186, 198), int(f_c.size * 1.3), 0.8))
    if contact:
        blocks.append(([""], f_c, (0, 0, 0), int(12 * s), 0.0))
        blocks.append(([contact], _font(int(30 * s), True), accent, int(f_c.size * 1.4), 1.0))
    total = sum(len(l) * lh for l, _, _, lh, _ in blocks)
    y0 = (H - total) / 2 - 20 * s
    pipe = FFmpegPipe(path, size, fps, crf=16)
    n = int(seconds * fps)
    for i in range(n):
        t = i / fps
        im = Image.fromarray(bd.next(place=0.5, strength=0.38, ramp_from=-1.0)).convert("RGBA")
        d = ImageDraw.Draw(im)
        y = y0
        for lines, font, col, lh, delay in blocks:
            a = float(np.clip((t - delay) / 0.6, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            ease = 1 - (1 - a) ** 3
            for ln in lines:
                if ln:
                    d.text((W / 2, y + lh / 2 + (1 - ease) * 18 * s), ln, font=font,
                           fill=tuple(col) + (int(255 * a),), anchor="mm")
                y += lh
        if note:
            a = float(np.clip((t - 1.1) / 0.6, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            for k, ln in enumerate(_wrap(d, note, f_n, int(0.86 * W))[:2]):
                d.text((W / 2, H - int((92 - 30 * k) * s)), ln, font=f_n, fill=(140, 148, 160, int(255 * a)),
                       anchor="mm")
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


def render_text_cards(path: str, cards: list[dict], size=(1920, 1080), fps: int = 30,
                      seconds: float | list = 7.0, fade: float = 0.6, accent=(255, 176, 64),
                      backdrop=None) -> str:
    """Cartões de texto (kicker, título, tópicos, nota opcional) com entrada suave, em HD.

    cards = [{"kicker": "O DESAFIO", "headline": "...", "bullets": ["...", "..."], "footnote": "..."}]
    backdrop: None, quadro RGB (ex.: um quadro HD do reator) ou Backdrop (reator em movimento), mostrado
    esmaecido à direita."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    pipe = FFmpegPipe(path, size, fps, crf=16)
    f_k, f_h, f_b, f_n = _font(int(30 * s), True), _font(int(62 * s), True), _font(int(38 * s)), _font(int(24 * s))
    secs = seconds if isinstance(seconds, (list, tuple)) else [seconds] * len(cards)
    bd = _backdrop(backdrop, size)
    wrap = _wrap

    for card, sec in zip(cards, secs):
        n = int(sec * fps)
        base = Image.new("RGBA", (8, 8))
        layers = []   # (y_alvo, imagem RGBA, atraso em s)
        x0, wmax = int(150 * s), int(W - 300 * s)
        y = int(230 * s)
        probe = ImageDraw.Draw(base)

        def layer(lines, font, color, gap, delay, bullet=False):
            nonlocal y
            h = int(font.size * 1.25)
            im = Image.new("RGBA", (W, h * len(lines) + int(gap)), (0, 0, 0, 0))
            d = ImageDraw.Draw(im)
            for k, ln in enumerate(lines):
                xo = x0 + (int(48 * s) if bullet else 0)
                if bullet and k == 0:
                    d.ellipse([x0 + int(6 * s), int(h * 0.42 - 8 * s), x0 + int(22 * s), int(h * 0.42 + 8 * s)],
                              fill=accent + (255,))
                d.text((xo, k * h), ln, font=font, fill=color)
            layers.append((y, im, delay))
            y += im.height

        layer([br(card.get("kicker", "")).upper()], f_k, accent + (255,), 18 * s, 0.0)
        layer(_wrap_balanced(probe, br(card["headline"]), f_h, wmax), f_h, (238, 242, 248, 255), 46 * s, 0.25)
        for j, bl in enumerate(card.get("bullets", [])):
            layer(wrap(probe, br(bl), f_b, wmax - int(48 * s)), f_b, (200, 208, 220, 255), 22 * s, 0.7 + 0.45 * j,
                  bullet=True)
        foot = br(card.get("footnote") or "")
        y_top = int(230 * s)
        shift = int((H - (y - y_top)) / 2 - y_top - 20 * s)      # centraliza o bloco na vertical
        layers = [(yl + shift, L, dl) for (yl, L, dl) in layers]
        bar = (y_top + shift, y + shift)
        icard = cards.index(card)
        for i in range(n):
            t = i / fps
            im = Image.fromarray(bd.next()).convert("RGBA")
            dr = ImageDraw.Draw(im)
            dr.rectangle([x0 - int(40 * s), bar[0], x0 - int(32 * s), bar[1]], fill=accent + (255,))
            if len(cards) > 1:   # indicador de progresso (bolinhas)
                for j in range(len(cards)):
                    cx = W - int((60 + 28 * (len(cards) - 1 - j)) * s)
                    r = int((7 if j == icard else 5) * s)
                    col = accent + (255,) if j == icard else (90, 98, 112, 255)
                    dr.ellipse([cx - r, H - int(60 * s) - r, cx + r, H - int(60 * s) + r], fill=col)
            for (yl, L, delay) in layers:
                a = float(np.clip((t - delay) / fade, 0, 1))
                a *= float(np.clip((sec - t) / fade, 0, 1))
                if a <= 0:
                    continue
                ease = 1 - (1 - a) ** 3
                Lc = L.copy()
                Lc.putalpha(Image.eval(L.getchannel("A"), lambda v, a=a: int(v * a)))
                im.alpha_composite(Lc, (0, int(yl + (1 - ease) * 24 * s)))
            if foot:
                a = float(np.clip((t - 1.2) / fade, 0, 1) * np.clip((sec - t) / fade, 0, 1))
                dr.text((x0, H - int(90 * s)), foot, font=f_n, fill=(140, 148, 160, int(255 * a)))
            pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


# ------------------------------------------------------------ figuras com legenda
def _frames_of(src: str, seconds: float | None, fps: int):
    """Quadros RGB de um vídeo (decodificado pelo ffmpeg) ou de uma imagem repetida por `seconds`."""
    from PIL import Image
    if src.lower().endswith((".png", ".jpg", ".jpeg")):
        im = np.asarray(Image.open(src).convert("RGB"))
        for _ in range(int(round((seconds or 5.0) * fps))):
            yield im
        return
    ff = shutil.which("ffmpeg")
    probe = subprocess.run([ff, "-i", src], capture_output=True, text=True).stderr
    import re
    m = re.search(r", (\d{2,5})x(\d{2,5})", probe)
    w, h = int(m.group(1)), int(m.group(2))
    p = subprocess.Popen([ff, "-loglevel", "error", "-i", src, "-vf", f"fps={fps}", "-f", "rawvideo",
                          "-pix_fmt", "rgb24", "-"], stdout=subprocess.PIPE)
    n = w * h * 3
    while True:
        buf = p.stdout.read(n)
        if len(buf) < n:
            break
        yield np.frombuffer(buf, np.uint8).reshape(h, w, 3)
    p.wait()


def caption_segment(src: str, path: str, captions, chapter: str = "", size=(1920, 1080), fps: int = 30,
                    seconds: float | None = None, band: float = 0.14, accent=(255, 176, 64),
                    handle: str = "") -> str:
    """Coloca um vídeo ou imagem (figura do estudo) acima de uma faixa de legenda: rótulo do capítulo e
    textos que se alternam com transição suave. captions: lista de textos (distribuídos igualmente) ou
    [(t0, t1, texto)]; handle: assinatura discreta no canto da faixa (ex.: endereço do LinkedIn)."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bh = int(band * H)
    frames = list(_frames_of(src, seconds, fps))
    dur = len(frames) / fps
    caps = captions or []
    if caps and isinstance(caps[0], str):
        caps = even_captions(caps, dur)
    f_k, f_c = _font(int(24 * s), True), _font(int(34 * s))
    base = Image.new("RGBA", size, (13, 17, 23, 255))
    d0 = ImageDraw.Draw(base)
    d0.rectangle([0, H - bh, W, H], fill=(9, 12, 18, 255))
    d0.rectangle([int(60 * s), H - bh + int(22 * s), int(66 * s), H - int(22 * s)], fill=accent + (255,))
    if chapter:
        d0.text((int(84 * s), H - bh + int(20 * s)), br(chapter).upper(), font=f_k, fill=accent + (255,))
    if handle:
        d0.text((W - int(60 * s), H - bh + int(22 * s)), handle, font=_font(int(22 * s)), fill=(118, 126, 140, 255),
                anchor="ra")
    pipe = FFmpegPipe(path, size, fps, crf=16)
    area_h = H - bh
    for i, fr in enumerate(frames):
        im = base.copy()
        src_im = Image.fromarray(fr)
        sc = min(W / src_im.width, area_h / src_im.height)
        fit = src_im.resize((int(src_im.width * sc), int(src_im.height * sc)), Image.LANCZOS)
        im.paste(fit, ((W - fit.width) // 2, (area_h - fit.height) // 2))
        text, a = _caption_alpha(caps, i / fps)
        if text:
            layer = Image.new("RGBA", size, (0, 0, 0, 0))
            d = ImageDraw.Draw(layer)
            lines = _wrap(d, br(text), f_c, int(W - 180 * s))[:2]
            y = H - bh + int((56 if chapter else 30) * s)
            for k, ln in enumerate(lines):
                d.text((int(84 * s), y + k * int(f_c.size * 1.2)), ln, font=f_c, fill=(236, 240, 246, int(255 * a)))
            im.alpha_composite(layer)
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


# --------------------------------------------------------------- números-chave
def render_kpi_card(path: str, title: str, kpis: list[dict], size=(1920, 1080), fps: int = 30,
                    seconds: float = 8.0, backdrop=None, note: str = "", kicker: str = "",
                    accent=(255, 176, 64)) -> str:
    """Cartão de números-chave com contagem animada.

    kpis = [{"value": 169, "fmt": "{:.0f}", "unit": "kg/h", "label": "plástico processado",
             "color": (255, 176, 64)}, ...] — value pode ser texto (sem animação)."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bd = _backdrop(backdrop, size)
    base = Image.new("RGBA", size, (0, 0, 0, 0))      # título e caixas, compostos sobre o fundo a cada quadro
    d0 = ImageDraw.Draw(base)
    y_t = int(170 * s)
    if kicker:
        d0.text((W // 2, y_t - int(48 * s)), br(kicker).upper(), font=_font(int(28 * s), True), fill=accent + (255,),
                anchor="mm")
    d0.text((W // 2, y_t + int(10 * s)), br(title), font=_font(int(56 * s), True), fill=(240, 244, 250, 255),
            anchor="mm")
    if note:
        d0.text((W // 2, H - int(70 * s)), br(note), font=_font(int(22 * s)), fill=(140, 148, 160, 255),
                anchor="mm")
    n = len(kpis)
    cols = 2 if n == 4 else n
    rows = int(math.ceil(n / cols))
    cw, ch = int(W * 0.40), int(250 * s)
    gx = (W - cols * cw) // (cols + 1)
    y0 = int(300 * s)
    boxes = []
    for k, kp in enumerate(kpis):
        r, c = divmod(k, cols)
        x = gx + c * (cw + gx)
        y = y0 + r * (ch + int(40 * s))
        d0.rounded_rectangle([x, y, x + cw, y + ch], radius=int(18 * s), fill=(18, 24, 33, 230),
                             outline=(40, 52, 68, 255), width=max(1, int(2 * s)))
        boxes.append((x, y))
    f_v, f_u, f_l = _font(int(92 * s), True), _font(int(40 * s), True), _font(int(30 * s))
    pipe = FFmpegPipe(path, size, fps, crf=16)
    N = int(seconds * fps)
    for i in range(N):
        t = i / fps
        im = Image.fromarray(bd.next(strength=0.35)).convert("RGBA")
        im.alpha_composite(base)
        d = ImageDraw.Draw(im)
        for k, (kp, (x, y)) in enumerate(zip(kpis, boxes)):
            a = float(np.clip((t - 0.3 - 0.35 * k) / 0.5, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            if a <= 0:
                continue
            col = tuple(kp.get("color", accent))
            v = kp["value"]
            if isinstance(v, (int, float)):
                prog = 1 - (1 - min(1.0, max(0.0, (t - 0.3 - 0.35 * k) / 1.4))) ** 3
                vs = br(kp.get("fmt", "{:.1f}").format(v * prog))
            else:
                vs = br(str(v))
            unit = kp.get("unit", "")
            wv = d.textlength(vs, font=f_v)
            wu = d.textlength(" " + unit, font=f_u) if unit else 0
            xs = x + (cw - wv - wu) / 2
            d.text((xs, y + int(40 * s)), vs, font=f_v, fill=col + (int(255 * a),))
            if unit:
                d.text((xs + wv, y + int(84 * s)), " " + unit, font=f_u, fill=col + (int(255 * a),))
            for j, ln in enumerate(_wrap(d, br(kp.get("label", "")), f_l, cw - int(40 * s))[:2]):
                d.text((x + cw / 2, y + int(170 * s) + j * int(36 * s)), ln, font=f_l,
                       fill=(200, 208, 220, int(255 * a)), anchor="mm")
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()

def render_still(src_png: str, path: str, seconds: float = 5.0, fps: int = 30):
    """Segmento de vídeo a partir de uma imagem fixa (1920×1080)."""
    ff = shutil.which("ffmpeg")
    subprocess.run([ff, "-y", "-loop", "1", "-framerate", str(fps), "-t", f"{seconds}", "-i", src_png,
                    "-vf", "scale=1920:1080,setsar=1,format=yuv420p", "-c:v", "libx264", "-crf", "16",
                    "-preset", "medium", path], check=True, capture_output=True)
    return path


def assemble(segments: list[str], out: str, fps: int = 30, size: tuple = (1920, 1080), crf: int = 16) -> str:
    """Concatena os segmentos em um MP4 final na resolução `size` (os de outra resolução são reescalados
    com Lanczos, preservando o aspecto)."""
    ff = shutil.which("ffmpeg")
    if ff is None:
        raise RuntimeError("ffmpeg não encontrado")
    segs = [p for p in segments if p and Path(p).exists()]
    cmd = [ff, "-y"]
    for p in segs:
        cmd += ["-i", p]
    W, H = size
    chains = "".join(f"[{i}:v]scale={W}:{H}:flags=lanczos:force_original_aspect_ratio=decrease,"
                     f"pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},format=yuv420p[v{i}];"
                     for i in range(len(segs)))
    cmd += ["-filter_complex", chains + "".join(f"[v{i}]" for i in range(len(segs)))
            + f"concat=n={len(segs)}:v=1:a=0[v]", "-map", "[v]", "-c:v", "libx264", "-crf", str(crf),
            "-preset", "slow", "-tune", "film", "-movflags", "+faststart", out]
    subprocess.run(cmd, check=True, capture_output=True)
    return out

In [ ]:
%%writefile tamborkit/scene.py
"""Cena em corte transversal do tambor dentro da fornalha (estilo dos infográficos de forno rotativo).

Camadas, de trás para a frente: fundo; caixa da fornalha (aço + refratário aquecido); gás de combustão e
chamas (fatia da LES no plano de um queimador, cor de corpo negro); casco do tambor colorido pela
temperatura (setores no referencial do tambor, girando); aletas; carga (partículas do DEM projetadas ao
longo do eixo, com oclusão por z-buffer e sombreamento de esfera; ou a poça de fundido); vapores.
Tudo em NumPy + PIL; sem matplotlib no quadro principal.
"""
from __future__ import annotations

import math

import numpy as np
from PIL import Image, ImageDraw, ImageFilter

from .video import _font, br

# ------------------------------------------------------------------ cores
_TSTOPS = [(20, (40, 70, 160)), (100, (40, 150, 190)), (200, (80, 190, 150)), (300, (230, 210, 80)),
           (400, (245, 150, 40)), (480, (225, 70, 30)), (560, (170, 20, 30)), (650, (255, 235, 220))]


def thermal_rgb(T_C):
    """Mapa de cor de temperatura da carga e do casco (azul frio → amarelo → vermelho → branco)."""
    T = np.asarray(T_C, float)
    xs = np.array([s[0] for s in _TSTOPS], float)
    cs = np.array([s[1] for s in _TSTOPS], float)
    return np.stack([np.interp(T, xs, cs[:, i]) for i in range(3)], -1)


def flame_rgba(T_K):
    """Cor e opacidade de chama/gás quente (aprox. corpo negro, intensidade ~ T⁴)."""
    T = np.asarray(T_K, float)
    stops = np.array([900, 1100, 1300, 1500, 1700, 2000], float)
    cols = np.array([(120, 20, 5), (200, 50, 10), (245, 110, 20), (255, 170, 50), (255, 215, 120),
                     (255, 245, 210)], float)
    rgb = np.stack([np.interp(T, stops, cols[:, i]) for i in range(3)], -1)
    a = np.clip((T - 850.0) / 550.0, 0, 1) ** 1.1
    return rgb, a


def colorbar(width, height, Tmin=20, Tmax=600, label="temperatura da carga e do casco (°C)", s=1.0):
    im = Image.new("RGBA", (width, height), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)
    bar_h = int(14 * s)
    xs = np.linspace(Tmin, Tmax, width - 2)
    rgb = thermal_rgb(xs).astype(np.uint8)
    bar = np.repeat(rgb[None], bar_h, 0)
    im.paste(Image.fromarray(bar), (1, int(26 * s)))
    f = _font(int(15 * s))
    d.text((0, 0), label, font=f, fill=(225, 225, 225, 255))
    for T in range(int(Tmin // 100 * 100 + 100), int(Tmax) + 1, 100):
        x = int(1 + (T - Tmin) / (Tmax - Tmin) * (width - 3))
        d.text((x - 12 * s, int(26 * s) + bar_h + 2), f"{T}", font=_font(int(13 * s)), fill=(200, 200, 200, 255))
    return im


# ------------------------------------------------------------- esferas
def splat_spheres(H, W, cx, cy, r, depth, rgb, light=(-0.45, -0.6, 0.66)):
    """Desenha esferas (centros em pixels, raio em pixels) com z-buffer e sombreamento difuso + brilho.

    depth: menor = mais perto da câmera. Devolve (imagem RGB float [H, W, 3], alfa [H, W])."""
    n = len(cx)
    out = np.zeros((H, W, 3), np.float32)
    alpha = np.zeros((H, W), np.float32)
    if n == 0:
        return out, alpha
    zbuf = np.full(H * W, np.inf, np.float32)
    owner = np.full(H * W, -1, np.int64)
    Rm = int(np.ceil(r.max())) + 1
    oy, ox = np.mgrid[-Rm:Rm + 1, -Rm:Rm + 1]
    ox = ox.ravel().astype(np.float32); oy = oy.ravel().astype(np.float32)
    L = np.array(light, np.float32); L /= np.linalg.norm(L)
    # processa em blocos para limitar memória
    nn = np.zeros((H * W, 3), np.float32)
    for b0 in range(0, n, 2048):
        sl = slice(b0, min(n, b0 + 2048))
        px = np.floor(cx[sl, None]) + ox[None]
        py = np.floor(cy[sl, None]) + oy[None]
        dx = (px + 0.5 - cx[sl, None]) / r[sl, None]
        dy = (py + 0.5 - cy[sl, None]) / r[sl, None]
        q = dx * dx + dy * dy
        ok = (q < 1.0) & (px >= 0) & (px < W) & (py >= 0) & (py < H)
        dz = np.sqrt(np.clip(1 - q, 0, 1))
        d = depth[sl, None] - dz * r[sl, None] * 1e-3
        ids = (py * W + px).astype(np.int64)
        ids_ok, d_ok = ids[ok], d[ok]
        pid = np.broadcast_to(np.arange(sl.start, sl.stop)[:, None], ok.shape)[ok]
        # ordena por profundidade decrescente e escreve: o mais perto fica por último (vence)
        order = np.argsort(-d_ok, kind="stable")
        ids_o, d_o, pid_o = ids_ok[order], d_ok[order], pid[order]
        closer = d_o < zbuf[ids_o]
        ids_c, d_c, pid_c = ids_o[closer], d_o[closer], pid_o[closer]
        zbuf[ids_c] = d_c
        owner[ids_c] = pid_c
        nx_ = dx[ok][order][closer]; ny_ = dy[ok][order][closer]; nz_ = dz[ok][order][closer]
        nn[ids_c] = np.stack([nx_, ny_, nz_], -1)
    hit = owner >= 0
    nrm = nn[hit]
    lam = np.clip(nrm @ L, 0, 1)
    Hv = L + np.array([0, 0, 1], np.float32); Hv /= np.linalg.norm(Hv)
    spec = np.clip(nrm @ Hv, 0, 1) ** 24
    col = rgb[owner[hit]].astype(np.float32)
    shade = 0.32 + 0.78 * lam
    pix = col * shade[:, None] + 90.0 * spec[:, None]
    out.reshape(-1, 3)[hit] = np.clip(pix, 0, 255)
    alpha.reshape(-1)[hit] = 1.0
    return out, alpha


# ------------------------------------------------------------- a cena
class CrossSection:
    """Corte do conjunto tambor + fornalha no plano (y, z), em pixels."""

    def __init__(self, drum, furnace_box, size=(1080, 1080), center=None, scale=None):
        self.drum = drum
        self.box = furnace_box            # dict(half_width, z_bot, z_top, wall)
        self.W, self.H = size
        hw, zb, zt = furnace_box["half_width"], furnace_box["z_bot"], furnace_box["z_top"]
        wall = furnace_box.get("wall", 0.18)
        span = max(2 * (hw + wall), (zt - zb) + 2 * wall) * 1.06
        self.s = scale or min(self.W, self.H) / span
        cz = 0.5 * (zt + zb)
        self.c = center or (self.W / 2, self.H / 2 + cz * self.s)
        Y, Z = np.meshgrid(np.arange(self.W) + 0.5, np.arange(self.H) + 0.5)
        self.Yw = (Y - self.c[0]) / self.s
        self.Zw = -(Z - self.c[1]) / self.s
        self.rho = np.hypot(self.Yw, self.Zw)
        self.phi = np.arctan2(self.Zw, self.Yw)

    def px(self, y, z):
        return self.c[0] + np.asarray(y) * self.s, self.c[1] - np.asarray(z) * self.s

    # ---------------------------------------------------------- partes
    def furnace_layer(self, T_refr_C=700.0, gas=None, flame=None, flame_img=None):
        """Caixa da fornalha, refratário e gás. gas/flame: (T_K [ny, nz], yc, zc) da LES no plano do queimador."""
        hw, zb, zt = self.box["half_width"], self.box["z_bot"], self.box["z_top"]
        wall = self.box.get("wall", 0.18)
        Y, Zw = self.Yw, self.Zw
        img = np.zeros((self.H, self.W, 3), np.float32)
        # fundo
        g = np.clip(1 - (Zw - zb) / (zt - zb + 2 * wall), 0, 1)
        img[:] = (np.array([14, 17, 24]) * (0.7 + 0.3 * g[..., None]))
        inside = (np.abs(Y) < hw) & (Zw > zb) & (Zw < zt)
        lining = (np.abs(Y) < hw + wall) & (Zw > zb - wall) & (Zw < zt + wall) & ~inside
        shell = (np.abs(Y) < hw + wall + 0.035) & (Zw > zb - wall - 0.035) & (Zw < zt + wall + 0.035) & ~inside & ~lining
        # refratário: tijolo com brilho conforme a temperatura (quente por dentro, frio por fora)
        depth_in = np.minimum.reduce([hw + wall - np.abs(Y), Zw - (zb - wall), (zt + wall) - Zw]) / wall
        heat = np.clip(1 - depth_in, 0, 1)
        brick = np.array([120, 72, 50], np.float32)
        glow = thermal_rgb(np.full(1, 0.0) + T_refr_C)[0]
        # tijolos: fiadas de 6,5 cm, juntas escuras finas
        row = np.floor(Zw / 0.065)
        jx = np.abs(((Y + 0.11 * (row % 2)) / 0.22) % 1 - 0.5) > 0.47
        jz = np.abs((Zw / 0.065) % 1 - 0.5) > 0.44
        tex = np.where(jx | jz, 0.62, 0.92 + 0.08 * np.sin(row * 12.9898) ** 2)
        ref = brick * (0.55 + 0.25 * heat[..., None]) * tex[..., None]
        ref = ref * (1 - 0.35 * heat[..., None]) + glow * 0.35 * heat[..., None] * min(1.0, (T_refr_C - 250) / 600)
        img[lining] = ref[lining]
        img[shell] = np.array([150, 158, 168])
        img[inside] = np.array([28, 22, 22])
        a_gas = np.zeros((self.H, self.W), np.float32)
        if gas is not None:
            T, yc, zc = gas[:3]
            gain = gas[3] if len(gas) > 3 else 1.0
            Ts = self._sample(T, yc, zc)
            rgb, a = flame_rgba(Ts)
            a = np.where(inside & (self.rho > self.drum.R + 0.01), a * gain, 0.0)
            img = img * (1 - 0.9 * a[..., None]) + rgb * 0.9 * a[..., None]
            # brilho (bloom) da chama
            glow = Image.fromarray(np.clip(rgb * a[..., None], 0, 255).astype(np.uint8)).filter(
                ImageFilter.GaussianBlur(radius=max(4, int(0.035 * self.s))))
            img = img + 0.55 * np.asarray(glow, np.float32)
            a_gas = a
        if flame_img is not None:
            # chamas renderizadas com base física (flames.FlameRenderer.render_on_grid): luz somada à cena
            rgb_f, a_f = flame_img
            a_f = np.where(inside & (self.rho > self.drum.R + 0.01), a_f, 0.0)
            img = img * (1 - 0.6 * a_f[..., None]) + 255.0 * np.asarray(rgb_f, np.float32) * \
                (inside & (self.rho > self.drum.R + 0.01))[..., None]
            a_gas = a_f
        # bocais dos queimadores na parede lateral
        for zb_ in self.box.get("burners_z", (-1.0,)):
            bx, bz = self.px(-hw, zb_)
            r_ = 0.06 * self.s
            m = (np.abs(self.Yw + hw + 0.03) < 0.05) & (np.abs(self.Zw - zb_) < 0.07)
            img[m] = np.array([70, 74, 82])
        self.inside = inside
        return img, a_gas

    def _sample(self, f, yc, zc):
        """Amostra bilinear de um campo (ny, nz) na malha de pixels."""
        fy = np.interp(self.Yw, yc, np.arange(len(yc)))
        fz = np.interp(self.Zw, zc, np.arange(len(zc)))
        j0 = np.clip(np.floor(fy).astype(int), 0, len(yc) - 2); k0 = np.clip(np.floor(fz).astype(int), 0, len(zc) - 2)
        wy, wz = fy - j0, fz - k0
        return ((1 - wy) * (1 - wz) * f[j0, k0] + wy * (1 - wz) * f[j0 + 1, k0]
                + (1 - wy) * wz * f[j0, k0 + 1] + wy * wz * f[j0 + 1, k0 + 1])

    def drum_layer(self, img, Ts_C_mat, theta, wall_px=None):
        """Casco colorido pela temperatura de cada setor (referencial do tambor) e aletas girando."""
        dr = self.drum
        R = dr.R
        t = max((wall_px or 9) / self.s, 0.03)
        ring = (self.rho >= R) & (self.rho < R + t)
        nphi = Ts_C_mat.shape[-1]
        phi_mat = (self.phi - theta + math.pi) % (2 * math.pi) - math.pi
        k = np.clip(((phi_mat + math.pi) / (2 * math.pi) * nphi).astype(int), 0, nphi - 1)
        col = thermal_rgb(Ts_C_mat[k])
        steel = np.array([95, 100, 110], np.float32)
        hot = np.clip((Ts_C_mat[k] - 350) / 250, 0, 1)[..., None]
        ringcol = steel * (1 - 0.75 * hot) + col * 0.75 * hot
        img[ring] = ringcol[ring]
        img[(self.rho >= R + t) & (self.rho < R + t + 0.006)] = np.array([30, 30, 34])
        inner = self.rho < R
        img[inner] = np.array([18, 16, 18]) + 10 * np.clip(1 - self.rho[inner] / R, 0, 1)[:, None]
        self.inner = inner
        return img

    def lifters_polys(self, theta):
        dr = self.drum
        out = []
        for a in dr.lifter_angles() + theta:
            ca, sa = math.cos(a), math.sin(a)
            for (u0, u1, w0, w1) in dr.lifter_boxes():
                u1 = min(u1, dr.R)
                if w1 - w0 < 0.03:          # espessura mínima visível
                    wc = 0.5 * (w0 + w1); w0, w1 = wc - 0.015, wc + 0.015
                pts = [(u0, w0), (u1, w0), (u1, w1), (u0, w1)]
                out.append([self.px(u * ca - w * sa, u * sa + w * ca) for u, w in pts])
        return out

    def bed_particles(self, img, xyz, radii, colors, view_depth=None):
        """Partículas do DEM vistas ao longo do eixo do tambor (y→direita, z→cima; x = profundidade)."""
        cx, cy = self.px(xyz[:, 1], xyz[:, 2])
        r = np.maximum(radii * self.s, 1.2)
        depth = xyz[:, 0] if view_depth is None else view_depth
        rgb, a = splat_spheres(self.H, self.W, cx.astype(np.float32), cy.astype(np.float32), r.astype(np.float32),
                               depth.astype(np.float32), colors)
        return img * (1 - a[..., None]) + rgb * a[..., None], a

    def pool(self, img, F, beta, T_C, char=0.0):
        """Poça de fundido: segmento circular com superfície inclinada de β, cor da temperatura."""
        from .geometry import fill_half_angle
        R = self.drum.R
        eps = fill_half_angle(max(F, 1e-4))
        # superfície: reta a distância R cos ε do centro, normal apontando para cima girada de β
        nrm = np.array([-math.sin(beta), math.cos(beta)])
        d = self.Yw * nrm[0] + self.Zw * nrm[1]
        m = (d < -R * math.cos(eps)) & (self.rho < R)
        col = thermal_rgb(T_C) * (1 - 0.8 * char)
        depth = (-R * math.cos(eps) - d) / max(R * (1 - math.cos(eps)), 1e-3)
        shade = 0.75 + 0.25 * np.clip(1 - depth, 0, 1)
        gloss = np.exp(-((-R * math.cos(eps) - d) * self.s / 3.0) ** 2)
        pix = col[None, None, :] * shade[..., None] + 70 * gloss[..., None]
        img[m] = np.clip(pix[m], 0, 255)
        return img, m

    def bed_raster(self, img, F, beta, T_fn, char_fn=None):
        """Leito como mapa de temperatura: T_fn(σ) com σ = 0 na parede molhada e 1 na superfície livre."""
        from .geometry import fill_half_angle
        R = self.drum.R
        eps = fill_half_angle(max(F, 1e-4))
        nrm = np.array([-math.sin(beta), math.cos(beta)])
        d = self.Yw * nrm[0] + self.Zw * nrm[1]
        hs = -R * math.cos(eps) - d                       # distância até a superfície
        m = (hs > 0) & (self.rho < R)
        dw = R - self.rho                                 # distância até a parede
        sig = np.clip(dw / np.maximum(dw + hs, 1e-6), 0, 1)
        T = T_fn(sig[m])
        col = thermal_rgb(T)
        if char_fn is not None:
            col = col * (1 - 0.8 * char_fn(sig[m]))[:, None]
        img[m] = col
        return img, m


def draw_overlay(canvas: Image.Image, cs: CrossSection, theta, labels=True, s=1.0, lifter_col=(165, 170, 180)):
    """Aletas (polígonos) e rótulos sobre a imagem da cena."""
    d = ImageDraw.Draw(canvas, "RGBA")
    for poly in cs.lifters_polys(theta):
        d.polygon([(float(x), float(y)) for x, y in poly], fill=lifter_col + (255,), outline=(60, 60, 66, 255))
    return canvas


# ----------------------------------------------------------- carga com mudança de fase (PhaseDEM)
def phase_colors(rec, label, mode="temperatura", T_bed_C=None):
    """Cor de cada pedaço: material → (aquecimento) → fundido âmbar → carvão. mode 'material' mantém a cor do
    material (segmento da química); 'temperatura' usa o mapa de temperatura da barra de cores."""
    from .phasedem import MATERIALS, CHAR_RGB, MELT_RGB
    base = np.array([m[3] for m in MATERIALS] + [CHAR_RGB], float)[label]
    Tp = np.asarray(rec["Tp"], float) - 273.15
    fm = np.asarray(rec["f_m"], float) / 255.0
    ch = np.asarray(rec["charf"], float) / 255.0
    if mode == "temperatura":
        w = np.clip((Tp - 40.0) / 60.0, 0, 1)[:, None] * 0.85
        col = base * (1 - w) + thermal_rgb(Tp) * w
    else:
        col = base.copy()
    col = col * (1 - fm[:, None] * 0.7) + np.array(MELT_RGB, float) * fm[:, None] * 0.7
    col = col * (1 - ch[:, None] * 0.9) + np.array(CHAR_RGB, float) * ch[:, None] * 0.9
    return col


def draw_phase_bed(cs, img, rec, label, mode="temperatura"):
    """Pedaços ativos do PhaseDEM + a poça de fundido translúcida por cima dos submersos."""
    act = np.asarray(rec["active"], bool)
    x = np.asarray(rec["x"], np.float32)[act]
    r = np.asarray(rec["r"], np.float32)[act]
    col = phase_colors(rec, label, mode)[act]
    if len(x):
        img, _ = cs.bed_particles(img, x, r, col)
    p = rec["pool"]
    if p.get("area", 0.0) > 1e-5:
        from .geometry import fill_half_angle
        from .phasedem import MELT_RGB
        R = cs.drum.R
        eps = fill_half_angle(max(p["area"] / (math.pi * R ** 2), 1e-5))
        beta = p.get("beta", 0.1)
        nrm = np.array([-math.sin(beta), math.cos(beta)])
        d = cs.Yw * nrm[0] + cs.Zw * nrm[1]
        depth = -R * math.cos(eps) - d
        m = (depth > 0) & (cs.rho < R)
        conv = float(p.get("conv", 0.0))
        if mode == "temperatura":
            c = thermal_rgb(p["T"] - 273.15) * 0.75 + np.array(MELT_RGB) * 0.25
        else:
            c = np.array(MELT_RGB, float)
        c = c * (1 - 0.55 * conv) + np.array([60, 32, 14], float) * 0.55 * conv
        gloss = np.exp(-(depth * cs.s / 3.0) ** 2)
        a = 0.72
        pix = c[None, None, :] * (0.8 + 0.2 * np.clip(1 - depth / max(R * (1 - math.cos(eps)), 1e-3), 0, 1))[..., None] \
            + 80 * gloss[..., None]
        img[m] = img[m] * (1 - a) + np.clip(pix[m], 0, 255) * a
    return img

In [ ]:
%%writefile tamborkit/timelapse.py
"""Time-lapse da batelada: o corte do tambor na fornalha à esquerda e o gráfico que se atualiza à direita.

Como no flare (altura da chama ao vivo), o gráfico anda junto com a simulação: temperaturas (carga, casco,
vapor) em cima e massas acumuladas (óleo, gás, água, carvão) embaixo, com o cursor no instante mostrado.
A carga é desenhada com as partículas do DEM (lixo solto; carvão no fim) ou como poça de fundido, conforme
o estado calculado pelo modelo da batelada; as chamas são quadros da LES da fornalha.
"""
from __future__ import annotations

import io
import math

import numpy as np
from PIL import Image, ImageDraw

from .scene import CrossSection, colorbar, draw_overlay, thermal_rgb
from .video import FFmpegPipe, _font, br, draw_lower_third

MATERIAL_RGB = {"filme": (225, 225, 215), "rigido": (70, 130, 200), "isopor": (245, 245, 245),
                "pet": (120, 200, 170), "organico": (150, 105, 60), "borracha": (40, 40, 40),
                "areia": (205, 185, 140), "carga": (180, 180, 170)}


def _interp_log(log, t_h, key):
    ts = np.array([r["t_h"] for r in log])
    return float(np.interp(t_h, ts, np.array([r[key] for r in log], float)))


def batch_events(log) -> list:
    """Marcos da batelada (para as legendas): fim da secagem, carga fundida, primeiro óleo, troca de
    combustível (o gás próprio assume) e fim."""
    t = np.array([r["t_h"] for r in log])
    agua = np.array([r["agua_kg"] for r in log])
    oleo = np.array([r["oleo_kg"] for r in log])
    melt = np.array([r["frac_fundido"] for r in log])
    qn = np.array([r["Q_ncg_kW"] for r in log]); qf = np.array([r["Q_fire_kW"] for r in log])

    def first(mask, default=None):
        i = np.argmax(mask)
        return float(t[i]) if mask.any() else default
    ev = [(0.0, "Partida: os queimadores ligam com GLP e o tambor começa a girar")]
    t_dry0 = first(np.array([r["T_carga"] for r in log]) > 98)
    if t_dry0:
        ev.append((t_dry0, "Secagem: a água do lixo segura a carga em 100 °C"))
    t_melt = first(melt > 0.4)
    if t_melt:
        ev.append((t_melt, "O plástico funde e a carga desaba: de 2/3 do tambor para uma poça"))
    t_oil = first(oleo > 0.01 * max(oleo[-1], 1))
    if t_oil:
        ev.append((t_oil, "Pirólise: as cadeias quebram e o vapor vira óleo no condensador"))
    t_sw = first((qn > 0.95 * qf) & (t > (t_oil or 0)))
    if t_sw:
        ev.append((t_sw, "O gás da própria pirólise assume o queimador; o excedente vai para o flare"))
    ev.append((float(t[-1]) * 0.93, "Fim: o gás para de sair; no tambor sobram carvão e cinzas"))
    ev.sort(key=lambda e: e[0])
    return ev


class LiveChart:
    """Gráfico ao vivo (matplotlib) desenhado até o instante t."""

    def __init__(self, log, size=(820, 860), dpi=100):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        self.plt = plt
        self.log = log
        self.size, self.dpi = size, dpi
        self.t = np.array([r["t_h"] for r in log])
        g = lambda k: np.array([r[k] for r in log], float)  # noqa: E731
        self.series_T = [("carga (média)", g("T_carga"), "#ffb347"), ("casco do tambor", g("T_casco"), "#e8e8e8"),
                         ("ponto mais quente do casco", g("T_casco_max"), "#ff5e5e"),
                         ("vapor na saída", g("T_vapor"), "#7fd3ff")]
        self.series_m = [("óleo coletado", g("oleo_kg"), "#ffcc33"), ("água (fase aquosa)", g("aquoso_kg"), "#5fa8ff"),
                         ("gás não condensável", g("gas_kg"), "#b0ff7a"),
                         ("carvão (inclui o negro de fumo da borracha)", g("carvao_kg"), "#c0a080")]
        self.t_end = float(self.t[-1])
        self.Tmax = max(650.0, float(np.nanmax(g("T_casco_max"))) + 30)
        self.mmax = max(float(np.nanmax(s[1])) for s in self.series_m) * 1.08

    def render(self, t_h) -> Image.Image:
        plt = self.plt
        W, H = self.size
        fig = plt.figure(figsize=(W / self.dpi, H / self.dpi), dpi=self.dpi)
        fig.patch.set_alpha(0.0)
        axs = [fig.add_axes([0.13, 0.56, 0.84, 0.38]), fig.add_axes([0.13, 0.09, 0.84, 0.38])]
        k = np.searchsorted(self.t, t_h, side="right")
        for ax, series, ylab, ymax in ((axs[0], self.series_T, "temperatura (°C)", self.Tmax),
                                       (axs[1], self.series_m, "massa acumulada (kg)", self.mmax)):
            ax.set_facecolor((0, 0, 0, 0))
            for name, y, c in series:
                ax.plot(self.t, y, color=c, lw=1.0, alpha=0.18)
                ax.plot(self.t[:k], y[:k], color=c, lw=2.6, label=name)
                if k > 0:
                    ax.plot([self.t[k - 1]], [y[k - 1]], "o", color=c, ms=7)
            ax.axvline(t_h, color="w", lw=1, alpha=0.5)
            ax.set_xlim(0, self.t_end); ax.set_ylim(0, ymax)
            ax.set_ylabel(ylab, color="w", fontsize=13)
            ax.tick_params(colors="w", labelsize=11)
            for sp in ax.spines.values():
                sp.set_color((1, 1, 1, 0.35))
            ax.grid(alpha=0.15)
            leg = ax.legend(loc="upper left", fontsize=11, frameon=False, labelcolor="w", ncol=1)
            del leg
        axs[0].axhline(500, color="#ff9e9e", lw=0.8, ls="--", alpha=0.5)
        axs[1].set_xlabel("tempo de batelada (h)", color="w", fontsize=13)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", transparent=True, dpi=self.dpi)
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).convert("RGBA")


class BatchScene:
    """Quadros do time-lapse: corte + gráfico + números + legendas."""

    def __init__(self, batch, dem_sets: dict, les_frames=None, size=(1920, 1080), fps=30, seconds=40.0,
                 vis_rev_s=6.0, title="Pirólise em batelada num tambor rotativo", subtitle="", flames=None,
                 flames_Q=400e3):
        self.b = batch
        self.log, self.snaps = batch.log, batch.snaps
        self.size, self.fps, self.seconds = size, fps, seconds
        self.W, self.H = size
        s = self.H / 1080
        self.s = s
        self.cs = CrossSection(batch.drum, dict(half_width=1.0, z_bot=-1.30, z_top=0.95, wall=0.12),
                               size=(int(1000 * s), int(960 * s)))
        self.chart = LiveChart(self.log, size=(int(800 * s), int(690 * s)), dpi=int(100 * s))
        self.dem = dem_sets
        self.les = les_frames          # dict(T=[n, ny, nz], yc, zc, Q) — fatia 2D (modo antigo)
        self.flames, self.flames_Q = flames, flames_Q   # FlameRenderer (vista 'corte'): chamas com base física
        self.vis_rev_s = vis_rev_s
        self.events = batch_events(self.log)
        self.t_end = float(self.log[-1]["t_h"])
        self.title, self.subtitle = title, subtitle
        self.cbar = colorbar(int(430 * s), int(60 * s), s=s)
        for k, d in self.dem.items():
            if k != "fase":
                d["rgb_mat"] = np.array([MATERIAL_RGB.get(d["names"][i], (180, 180, 170)) for i in d["label"]], float)

    # --------------------------------------------------------------- estado
    def _snap(self, t_h):
        ts = np.array([sn["t"] / 3600 for sn in self.snaps])
        i = int(np.clip(np.searchsorted(ts, t_h), 1, len(ts) - 1))
        a = (t_h - ts[i - 1]) / max(ts[i] - ts[i - 1], 1e-9)
        return self.snaps[i - 1] if a < 0.5 else self.snaps[i]

    def _bed_particles(self, key, frame_idx, F, beta, T_bed_C, Ts_lab_C, char, phi_lab, mat_w):
        """Partículas de um quadro do DEM: mantém as mais baixas até o enchimento F do modelo da batelada."""
        d = self.dem[key]
        fr = d["frames"][frame_idx % len(d["frames"])]
        x = fr["x"].astype(np.float32)
        theta_dem = fr["theta"]
        # gira o quadro do DEM para alinhar as aletas com o ângulo visual
        r = d["r"].astype(np.float32)
        nrm = np.array([-math.sin(beta), math.cos(beta)], np.float32)
        hgt = x[:, 1] * nrm[0] + x[:, 2] * nrm[1]
        F_dem = d["fill"]
        keep_n = int(np.clip(F / max(F_dem, 1e-6), 0.02, 1.0) * len(x))
        order = np.argsort(hgt)[:keep_n]
        xs, rs = x[order], r[order]
        # cor: material no início → temperatura ao aquecer → escurece ao carbonizar; mais quente junto à parede
        dw = self.b.drum.R - np.hypot(xs[:, 1], xs[:, 2])
        ph = np.arctan2(xs[:, 2], xs[:, 1])
        Tw = np.interp(ph, phi_lab, Ts_lab_C, period=2 * math.pi)
        Tp = T_bed_C + (Tw - T_bed_C) * np.exp(-np.clip(dw, 0, None) / 0.04) * 0.6
        col = thermal_rgb(Tp)
        col = (1 - mat_w) * d["rgb_mat"][order] + mat_w * col
        col = col * (1 - 0.82 * char)
        return xs, rs, col, theta_dem

    # ---------------------------------------------------------------- quadro
    def frame(self, k: int) -> np.ndarray:
        n = int(self.seconds * self.fps)
        t_h = self.t_end * k / max(n - 1, 1)
        sn = self._snap(t_h)
        g = lambda key: _interp_log(self.log, t_h, key)  # noqa: E731
        cs = self.cs
        b = self.b
        mid = b.cfg.nx // 2
        # casco no referencial do laboratório
        phi_mat = b.phi0
        phi_lab = (phi_mat + sn["theta"] + math.pi) % (2 * math.pi) - math.pi
        order = np.argsort(phi_lab)
        Ts_lab = (sn["Ts"][mid] - 273.15)[order]
        phi_sorted = phi_lab[order]
        # giro mostrado = o dos quadros do DEM (as partículas e as aletas ficam sincronizadas; o trecho gravado
        # cobre um número inteiro de passagens de aleta, então o laço não tem emenda visível)
        dem0 = self.dem.get("solto")
        theta_vis = float(dem0["frames"][k % len(dem0["frames"])]["theta"]) if dem0 else 0.0
        # ---- fornalha e chamas
        gas = None
        flame_img = None
        if getattr(self, "flames", None) is not None:
            Q = max(g("Q_fire_kW") * 1e3, 1.0)
            scale = float(np.clip(Q / self.flames_Q, 0.05, 1.2))
            flame_img = self.flames.render_on_grid(k, cs.Yw, cs.Zw, gain=scale ** 0.8)
        elif self.les is not None:
            T = self.les["T"][k % len(self.les["T"])]
            Q = max(g("Q_fire_kW") * 1e3, 1.0)
            scale = float(np.clip(Q / self.les["Q"], 0.08, 1.2))
            gas = (T, self.les["yc"], self.les["zc"], scale ** 0.6)
        T_ref = float(np.mean(sn["Tr_fur"])) - 273.15
        img, _ = cs.furnace_layer(T_refr_C=T_ref, gas=gas, flame_img=flame_img)
        # casco (desenhado no laboratório: o padrão térmico fica parado; as aletas giram)
        nphi = len(Ts_lab)
        phi_bins = (np.arange(nphi) + 0.5) * 2 * math.pi / nphi - math.pi
        Ts_bins = np.interp(phi_bins, phi_sorted, Ts_lab, period=2 * math.pi)
        img = cs.drum_layer(img, Ts_bins, 0.0, wall_px=10 * self.s)
        # ---- carga
        lo, me, re = (float(sn["frac"][i][mid]) for i in range(3))
        F = float(sn["F"][mid])
        beta = float(sn["beta"][mid])
        T_bed = float(np.mean(b._slice_view(sn["T"])[mid])) - 273.15
        char_m = float(sn["char"][min(mid, len(sn["char"]) - 1)])
        tot_m = float(sn["total"][min(mid, len(sn["total"]) - 1)])
        char = np.clip(char_m / max(tot_m, 1e-9), 0, 1)
        mat_w = float(np.clip((T_bed - 35.0) / 50.0, 0, 1))
        layers = []
        if "fase" in self.dem:
            # DEM com mudança de fase: os mesmos pedaços da partida ao carvão (sem troca de conjunto)
            from .scene import draw_phase_bed
            fz = self.dem["fase"]
            tb = fz.setdefault("_tb", np.array([r["t_b"] for r in fz["recs"]]) / 3600.0)
            rec = fz["recs"][int(np.argmin(np.abs(tb - t_h)))]
            img = draw_phase_bed(cs, img, rec, fz["label"], mode="temperatura")
            theta_vis = float(rec["theta"])
        elif "solto" in self.dem and (lo > 0.25 or me < 0.35) and re < 0.65:
            xs, rs, col, th_dem = self._bed_particles("solto", k, F, beta, T_bed, Ts_lab, char * 0.5, phi_sorted, mat_w)
            w = float(np.clip((lo + 0.2 - me) / 0.4, 0, 1)) if me > 0.2 else 1.0
            layers.append(("p", xs, rs, col, w))
        if "fase" not in self.dem and me >= 0.2 and re < 0.75:
            w = float(np.clip((me - 0.2) / 0.3, 0, 1))
            layers.append(("pool", F, beta, T_bed, char, w))
        if "fase" not in self.dem and "carvao" in self.dem and re >= 0.45:
            xs, rs, col, th_dem = self._bed_particles("carvao", k, F, beta, T_bed, Ts_lab, min(char * 1.6, 0.9),
                                                      phi_sorted, 1.0)
            w = float(np.clip((re - 0.45) / 0.25, 0, 1))
            layers.append(("p", xs, rs, col, w))
            if w > 0.5:
                theta_vis = float(th_dem)
        for L_ in layers:
            if L_[0] == "p":
                _, xs, rs, col, w = L_
                new, a = cs.bed_particles(img.copy(), xs, rs, col)
            else:
                _, F_, beta_, Tb_, ch_, w = L_
                new, m = cs.pool(img.copy(), F_, beta_ * 0.3, Tb_, ch_)
            img = img * (1 - w) + new * w
        canvas = Image.new("RGBA", self.size, (12, 14, 20, 255))
        scene = Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))
        scene = draw_overlay(scene, cs, theta_vis)
        canvas.paste(scene, (int(40 * self.s), int(92 * self.s)))
        # ---- gráfico ao vivo
        ch = self.chart.render(t_h)
        canvas.paste(ch, (int(1095 * self.s), int(180 * self.s)), ch)
        d = ImageDraw.Draw(canvas)
        s = self.s
        d.text((int(36 * s), int(18 * s)), br(self.title), font=_font(int(34 * s), True), fill=(245, 245, 245))
        if self.subtitle:
            d.text((int(38 * s), int(60 * s)), br(self.subtitle), font=_font(int(18 * s)), fill=(190, 195, 205))
        # números grandes
        big = [(f"{t_h:4.1f} h".replace(".", ","), "de batelada"), (f"{g('T_carga'):.0f} °C", "carga"),
               (f"{g('oleo_kg'):.0f} kg", "de óleo"), (f"{g('Q_fire_kW'):.0f} kW", "queima")]
        x0 = int(1110 * s)
        for i, (v, lab) in enumerate(big):
            xx = x0 + int(i * 200 * s)
            d.text((xx, int(96 * s)), v, font=_font(int(36 * s), True), fill=(255, 210, 120) if i == 2 else (245, 245, 245))
            d.text((xx, int(142 * s)), lab, font=_font(int(16 * s)), fill=(185, 190, 200))
        frac_ncg = g("Q_ncg_kW") / max(g("Q_fire_kW"), 1e-6)
        fuel = "gás da própria pirólise" if frac_ncg > 0.9 else ("GLP + gás de pirólise" if frac_ncg > 0.1 else "GLP (partida)")
        d.text((int(44 * s), int(1056 * s)), f"queimador: {fuel}   ·   giro do tambor ilustrativo no time-lapse".replace(".", ","),
               font=_font(int(17 * s)), fill=(200, 205, 215))
        canvas.paste(self.cbar, (int(560 * s), int(985 * s)), self.cbar)
        # legenda do marco atual (embaixo do gráfico)
        cur = [e for e in self.events if e[0] <= t_h + 1e-6]
        if cur:
            draw_lower_third(canvas, cur[-1][1], 1.0, s * 0.82, y_bottom=int(1060 * s), max_w=int(760 * s),
                             x0=int(1080 * s))
        return np.asarray(canvas.convert("RGB"))

    def render(self, path: str) -> str:
        n = int(self.seconds * self.fps)
        pipe = FFmpegPipe(path, self.size, self.fps)
        for k in range(n):
            pipe.write(self.frame(k))
        return pipe.close()

In [ ]:
%%writefile tamborkit/compare.py
"""Parado × girando: dois cortes lado a lado no mesmo instante da batelada (o tema da matéria da KINTEK).

Mesma carga, mesma fornalha e mesmo controle; só muda a rotação. No tambor parado a pilha só conduz calor:
a camada junto à parede quente funde, carboniza e isola, o miolo continua frio e o casco sob a chama
passa do limite do aço. Girando, cada volta renova o contato da carga com a parede e a temperatura fica
uniforme na seção.
"""
from __future__ import annotations

import math

import numpy as np
from PIL import Image, ImageDraw

from .scene import CrossSection, colorbar, thermal_rgb
from .video import FFmpegPipe, _font, br, draw_lower_third


def _snap_at(b, t_h):
    ts = np.array([sn["t"] / 3600 for sn in b.snaps])
    i = int(np.argmin(np.abs(ts - t_h)))
    return b.snaps[i], ts[i] >= t_h - 0.05 or i == len(ts) - 1


def bed_T_function(b, sn, mid):
    """T(σ) da carga na fatia do meio: σ = 0 na parede molhada, 1 na superfície livre."""
    T = b._slice_view(sn["T"])[mid] - 273.15
    if T.size == 1:
        return lambda sig: np.full_like(sig, T[0])
    v = b._slice_view(sn["vol"])[mid]
    c = np.cumsum(v) / max(v.sum(), 1e-12)
    centers = c - 0.5 * v / max(v.sum(), 1e-12)
    return lambda sig: np.interp(sig, centers, T)


def char_function(b, sn, mid):
    ch = b._slice_view(sn["char"])[mid]
    tot = b._slice_view(sn["total"])[mid]
    f = ch / np.maximum(tot, 1e-12)
    if f.size == 1:
        return lambda sig: np.full_like(sig, min(f[0] * 1.5, 0.9))
    v = b._slice_view(sn["vol"])[mid]
    c = np.cumsum(v) / max(v.sum(), 1e-12)
    centers = c - 0.5 * v / max(v.sum(), 1e-12)
    return lambda sig: np.clip(np.interp(sig, centers, f) * 1.5, 0, 0.9)


def draw_section(cs: CrossSection, b, sn, theta_vis=0.0, gas=None, flame_img=None):
    mid = b.cfg.nx // 2
    img, _ = cs.furnace_layer(T_refr_C=float(np.mean(sn["Tr_fur"])) - 273.15, gas=gas, flame_img=flame_img)
    phi_lab = (b.phi0 + sn["theta"] + math.pi) % (2 * math.pi) - math.pi
    o = np.argsort(phi_lab)
    nphi = len(phi_lab)
    bins = (np.arange(nphi) + 0.5) * 2 * math.pi / nphi - math.pi
    Ts = np.interp(bins, phi_lab[o], sn["Ts"][mid][o] - 273.15, period=2 * math.pi)
    img = cs.drum_layer(img, Ts, 0.0, wall_px=12)
    img, _ = cs.bed_raster(img, float(sn["F"][mid]), float(sn["beta"][mid]), bed_T_function(b, sn, mid),
                           char_function(b, sn, mid))
    return img, float(Ts.max())


def render_compare(b_rot, b_est, path, seconds=18.0, fps=30, size=(1920, 1080), t_end_h=None, les=None,
                   flames=None, flames_Q=400e3):
    W, H = size
    s = H / 1080
    t_end = t_end_h or max(b_est.log[-1]["t_h"], b_rot.log[-1]["t_h"])
    box = dict(half_width=1.0, z_bot=-1.30, z_top=0.95, wall=0.17)
    cs = CrossSection(b_rot.drum, box, size=(int(820 * s), int(820 * s)))
    cbar = colorbar(int(520 * s), int(64 * s), s=s)
    pipe = FFmpegPipe(path, size, fps)
    n = int(seconds * fps)
    lr = np.array([r["t_h"] for r in b_rot.log]); le = np.array([r["t_h"] for r in b_est.log])
    for k in range(n):
        t_h = t_end * k / (n - 1)
        canvas = Image.new("RGBA", size, (12, 14, 20, 255))
        d = ImageDraw.Draw(canvas)
        d.text((int(40 * s), int(22 * s)), "Por que girar? Mesma carga, mesma fornalha, só muda a rotação",
               font=_font(int(36 * s), True), fill=(245, 245, 245))
        for j, (b, lab, ts_log) in enumerate(((b_est, "tambor parado", le), (b_rot, f"girando a {b_rot.rpm:.1f} rpm, 6 aletas".replace(".", ","), lr))):
            sn, _ = _snap_at(b, t_h)
            gas, flame_img = None, None
            if flames is not None and t_h <= ts_log[-1]:
                gain = float(np.clip(sn["Q_fire"] / flames_Q, 0.05, 1.2)) ** 0.8
                flame_img = flames.render_on_grid(k, cs.Yw, cs.Zw, gain=gain)
            elif les is not None and t_h <= ts_log[-1]:
                gain = float(np.clip(sn["Q_fire"] / les["Q"], 0.08, 1.2)) ** 0.6
                gas = (les["T"][k % len(les["T"])], les["yc"], les["zc"], gain)
            img, Tmax = draw_section(cs, b, sn, gas=gas, flame_img=flame_img)
            x0 = int((60 + j * 920) * s)
            canvas.paste(Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)), (x0, int(120 * s)))
            done = t_h > ts_log[-1] + 1e-6
            T_c = float(np.interp(min(t_h, ts_log[-1]), ts_log, [r["T_carga"] for r in b.log]))
            oil = float(np.interp(min(t_h, ts_log[-1]), ts_log, [r["oleo_kg"] for r in b.log]))
            Tsm = float(np.interp(min(t_h, ts_log[-1]), ts_log, [r["T_casco_max"] for r in b.log]))
            d.text((x0, int(90 * s)), lab, font=_font(int(28 * s), True), fill=(255, 205, 120) if j else (160, 200, 255))
            y = int(950 * s)
            d.text((x0, y), br(f"carga {T_c:.0f} °C · óleo {oil:.0f} kg"), font=_font(int(26 * s), True), fill=(240, 240, 240))
            hot = (255, 110, 100) if Tsm > 475 else (200, 205, 215)
            d.text((x0, y + int(36 * s)), br(f"casco no ponto mais quente: {Tsm:.0f} °C"), font=_font(int(22 * s)), fill=hot)
            if done:
                d.text((x0 + int(520 * s), int(90 * s)), br(f"terminou em {ts_log[-1]:.1f} h"), font=_font(int(26 * s), True),
                       fill=(140, 255, 160))
        d.text((int(1700 * s), int(14 * s)), br(f"{t_h:4.1f} h"), font=_font(int(48 * s), True), fill=(255, 255, 255))
        canvas.paste(cbar, (int(700 * s), int(1010 * s)), cbar)
        pipe.write(np.asarray(canvas.convert("RGB")))
    return pipe.close()

In [ ]:
%%writefile tamborkit/story.py
"""Segmentos do vídeo e figuras do relatório: DEM em tempo real, chamas da LES, curvas e tabela de validação."""
from __future__ import annotations

import io
import math

import numpy as np
from PIL import Image, ImageDraw

from .scene import CrossSection, colorbar, draw_overlay, thermal_rgb
from .timelapse import MATERIAL_RGB
from .video import FFmpegPipe, _font, _wrap, br, draw_lower_third, even_captions, _caption_alpha


def _title(d, text, x, y, s, width=800, size=36):
    f = _font(int(size * s), True)
    lines = _wrap(d, br(text), f, int(width * s))
    for i, ln in enumerate(lines):
        d.text((x, y + i * int(size * 1.25 * s)), ln, font=f, fill=(245, 245, 245))
    return y + len(lines) * int(size * 1.25 * s)

BOX = dict(half_width=1.0, z_bot=-1.30, z_top=0.95, wall=0.17)


def render_dem_segment(drum, dem: dict, path, seconds=9.0, fps=30, size=(1920, 1080), title="", captions=(),
                       speed_label=""):
    """Partículas do DEM (cores dos materiais) girando com o tambor e as aletas, em corte."""
    W, H = size
    s = H / 1080
    cs = CrossSection(drum, dict(half_width=0.80, z_bot=-0.80, z_top=0.80, wall=0.04), size=(int(980 * s), int(980 * s)))
    from .phasedem import MATERIALS as _PM
    palette = dict(MATERIAL_RGB)
    palette.update({m[0]: m[3] for m in _PM})
    rgb_mat = np.array([palette.get(dem["names"][i], (30, 26, 24)) for i in dem["label"]], float)
    frames = dem["frames"]
    n = int(seconds * fps)
    caps = even_captions(list(captions), seconds) if captions else []
    pipe = FFmpegPipe(path, size, fps)
    for k in range(n):
        fr = frames[k % len(frames)]
        img = np.zeros((cs.H, cs.W, 3), np.float32) + np.array([14, 16, 22], np.float32)
        img = cs.drum_layer(img, np.full(36, 120.0), 0.0, wall_px=14 * s)
        act = np.asarray(fr.get("active", np.ones(len(fr["x"]), bool)), bool)
        rr = np.asarray(fr.get("r", dem["r"]), np.float32)
        img, _ = cs.bed_particles(img, fr["x"].astype(np.float32)[act], rr[act], rgb_mat[act])
        scene = draw_overlay(Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)), cs, fr["theta"])
        canvas = Image.new("RGBA", size, (12, 14, 20, 255))
        canvas.paste(scene, (int(60 * s), int(60 * s)))
        d = ImageDraw.Draw(canvas)
        y = _title(d, title, int(1080 * s), int(70 * s), s) + int(24 * s)
        present = set(dem["names"][i] for i in np.unique(dem["label"][np.asarray(frames[0].get("active", np.ones(len(dem["label"]), bool)), bool)]))
        for name, c in palette.items():
            if name not in present:
                continue
            d.ellipse([int(1090 * s), y, int(1112 * s), y + int(22 * s)], fill=tuple(int(v) for v in c))
            d.text((int(1125 * s), y - 2), {"filme": "filmes e sacolas (PEBD)", "rigido": "plástico rígido",
                                            "rigido_pe": "frascos e tampas (PEAD)", "rigido_pp": "tampas, cordas, redes (PP)",
                                            "pvc": "PVC (tubos, cabos)", "isopor": "isopor (EPS)",
                                            "pet": "garrafas PET", "organico": "madeira, papel, coco", "borracha": "borracha",
                                            "areia": "areia e conchas"}.get(name, name), font=_font(int(22 * s)),
                   fill=(225, 228, 235))
            y += int(36 * s)
        st = dem.get("stats", {})
        lines = []
        if st:
            lines = [f"ângulo dinâmico do leito: {st['beta_deg']:.0f}°",
                     f"parede coberta pela carga: {100 * st['covered_frac']:.0f} % do perímetro",
                     f"cada ponto do casco passa {st['passive_contact_s'] * dem['rpm'] / 0.8:.0f} s sob a carga por volta",
                     f"em queda nas cortinas: {100 * st['airborne_frac']:.1f} % dos pedaços",
                     f"mistura (índice de Lacey) no trecho mostrado: {dem.get('lacey', float('nan')):.2f}"]
        for i, ln in enumerate(lines):
            d.text((int(1090 * s), int(470 * s) + i * int(44 * s)), br(ln), font=_font(int(25 * s)), fill=(240, 240, 240))
        if speed_label:
            for i, ln in enumerate(_wrap(d, br(speed_label), _font(int(20 * s)), int(780 * s))):
                d.text((int(1090 * s), int(740 * s) + i * int(26 * s)), ln, font=_font(int(20 * s)), fill=(170, 175, 185))
        if caps:
            txt, a = _caption_alpha(caps, k / fps)
            draw_lower_third(canvas, txt, a, s * 0.85, y_bottom=int(1030 * s), max_w=int(760 * s), x0=int(1070 * s))
        pipe.write(np.asarray(canvas.convert("RGB")))
    return pipe.close()


def render_flame_segment(drum, les: dict, path, Ts_C=500.0, seconds=8.0, fps=30, size=(1920, 1080), title="",
                         captions=(), info=()):
    """Chamas da LES (fatia no plano de um queimador) sob o tambor, com o casco a T_s."""
    W, H = size
    s = H / 1080
    cs = CrossSection(drum, BOX, size=(int(1000 * s), int(1000 * s)))
    n = int(seconds * fps)
    caps = even_captions(list(captions), seconds) if captions else []
    pipe = FFmpegPipe(path, size, fps)
    for k in range(n):
        T = les["T"][k % len(les["T"])]
        img, _ = cs.furnace_layer(T_refr_C=750.0, gas=(T, les["yc"], les["zc"]))
        img = cs.drum_layer(img, np.full(36, Ts_C), 0.0, wall_px=12 * s)
        scene = draw_overlay(Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)), cs, 2 * math.pi * k / fps / 20)
        canvas = Image.new("RGBA", size, (12, 14, 20, 255))
        canvas.paste(scene, (int(40 * s), int(50 * s)))
        d = ImageDraw.Draw(canvas)
        y = _title(d, title, int(1080 * s), int(70 * s), s) + int(30 * s)
        for i, ln in enumerate(info):
            d.text((int(1090 * s), y + i * int(46 * s)), br(ln), font=_font(int(26 * s)), fill=(235, 235, 235))
        if caps:
            txt, a = _caption_alpha(caps, k / fps)
            draw_lower_third(canvas, txt, a, s * 0.85, y_bottom=int(1030 * s), max_w=int(760 * s), x0=int(1070 * s))
        pipe.write(np.asarray(canvas.convert("RGB")))
    return pipe.close()


# ----------------------------------------------------------------- figuras
def _style(plt):
    plt.rcParams.update({"figure.facecolor": "#0c0e14", "axes.facecolor": "#0c0e14", "axes.edgecolor": "#888",
                         "axes.labelcolor": "w", "xtick.color": "w", "ytick.color": "w", "text.color": "w",
                         "font.size": 13, "axes.grid": True, "grid.alpha": 0.15})


def curves_figure(runs: dict, path, title="Temperatura da carga e óleo coletado"):
    """runs: {rótulo: log}."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _style(plt)
    fig, axs = plt.subplots(1, 2, figsize=(16, 7), dpi=120)
    cols = ["#ffb347", "#7fd3ff", "#b0ff7a", "#ff7a9c", "#d0a0ff"]
    for (lab, log), c in zip(runs.items(), cols):
        t = [r["t_h"] for r in log]
        axs[0].plot(t, [r["T_carga"] for r in log], color=c, lw=2.4, label=lab)
        axs[0].plot(t, [r["T_casco_max"] for r in log], color=c, lw=1.2, ls="--", alpha=0.8)
        axs[1].plot(t, [r["oleo_kg"] for r in log], color=c, lw=2.4, label=lab)
    axs[0].axhline(475, color="#ff5e5e", lw=0.8, alpha=0.6)
    axs[0].text(0.2, 482, "limite de serviço do aço do casco (Q345R)", color="#ff9e9e", fontsize=11)
    axs[0].set_ylabel("°C (cheia: carga; tracejada: ponto mais quente do casco)")
    axs[1].set_ylabel("óleo coletado (kg)")
    for ax in axs:
        ax.set_xlabel("tempo de batelada (h)")
        ax.legend(frameon=False)
    fig.suptitle(title, fontsize=18)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return path


def validation_figure(rows, path, title="Validação"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _style(plt)
    n = len(rows)
    fig, ax = plt.subplots(figsize=(16, 0.52 * n + 1.6), dpi=110)
    ax.axis("off")
    k = sum(r["ok"] for r in rows)
    ax.set_title(br(f"{title}: {k} de {n} dentro do alvo"), fontsize=19, loc="left")
    for i, r in enumerate(rows):
        y = 1 - (i + 0.8) / (n + 0.5)
        v = r["modelo"]
        vs = br(f"{v:.3g}") + (f" {r['unidade']}" if r["unidade"] else "")
        ax.text(0.0, y, "✔" if r["ok"] else "✘", color="#7dff9a" if r["ok"] else "#ff7a7a", fontsize=15,
                transform=ax.transAxes, va="center")
        ax.text(0.03, y, r["grupo"], fontsize=12, color="#aab", transform=ax.transAxes, va="center")
        ax.text(0.16, y, br(r["item"]), fontsize=13, transform=ax.transAxes, va="center")
        ax.text(0.62, y, vs, fontsize=13, transform=ax.transAxes, va="center", color="#ffd27a")
        ax.text(0.74, y, br(f"alvo {r['alvo']}"), fontsize=12, transform=ax.transAxes, va="center", color="#ccd")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def flux_figure(flux: dict, les: dict, path):
    """Mapa de fluxo no casco (fatia × ângulo) e campo médio de T da LES num corte longitudinal."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _style(plt)
    fig, axs = plt.subplots(1, 2, figsize=(16, 6.5), dpi=120)
    q = flux["q"] / 1e3
    im = axs[0].pcolormesh(flux["x"], flux["phi"], q.T, cmap="inferno", shading="nearest")
    axs[0].set_xlabel("posição ao longo do tambor (m)"); axs[0].set_ylabel("ângulo no casco (°; −90 = embaixo)")
    plt.colorbar(im, ax=axs[0], label="fluxo para o casco (kW/m²)")
    Tm = les["Tmean"]
    j = Tm.shape[1] // 2
    im2 = axs[1].pcolormesh(les["xc"], les["zc"], (Tm[:, j, :] - 273.15).T, cmap="inferno", shading="nearest",
                            vmin=200, vmax=1500)
    axs[1].set_xlabel("x (m)"); axs[1].set_ylabel("z (m)")
    axs[1].set_aspect("equal")
    plt.colorbar(im2, ax=axs[1], label="T média do gás (°C), plano y = 0")
    fig.suptitle("LES da fornalha: onde o calor entra no tambor", fontsize=17)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)
    return path


# ------------------------------------------------- taxa de pirólise ao vivo (o "DTG" da batelada)
GROUP_STYLE = [("água", "#5fa8ff", "água (secagem)"), ("PVC", "#c9c9d6", "PVC → HCl"),
               ("celulósicos", "#b07a45", "papel, madeira"), ("PS", "#f2f2f2", "isopor (PS)"),
               ("PP", "#f0b840", "PP"), ("PE", "#7fb2ff", "PE (filmes, frascos)"), ("PET", "#7fd6b3", "PET"),
               ("borracha", "#8a8a8a", "borracha")]
GROUP_MSG = {"água": "Secagem: a água do lixo sai a 100 °C",
             "PVC": "O PVC solta HCl (≈ 250–320 °C), que vai para a fase aquosa",
             "celulósicos": "Papel e madeira viram gás, alcatrão e carvão (≈ 300–380 °C)",
             "PS": "O isopor (PS) volta a estireno (≈ 370–400 °C)",
             "PP": "O PP craqueia em óleo leve (≈ 400 °C)",
             "PE": "Filmes e frascos de PE: o grosso do óleo e da cera (≈ 410–430 °C)",
             "PET": "PET: ácidos e CO/CO₂; deixa carvão",
             "borracha": "Borracha: óleo e negro de fumo"}


def _smooth_rate(t, r, w):
    """Média móvel centrada de largura w (h) de uma taxa, pela integral acumulada (conserva a massa)."""
    I = np.concatenate([[0.0], np.cumsum(0.5 * (r[1:] + r[:-1]) * np.diff(t))])
    lo, hi = np.clip(t - w / 2, t[0], t[-1]), np.clip(t + w / 2, t[0], t[-1])
    return (np.interp(hi, t, I) - np.interp(lo, t, I)) / np.maximum(hi - lo, 1e-9)


class RatesChart:
    """Taxas por grupo (empilhadas) e conversões. A taxa é suavizada em ~6 min (a secagem avança fatia a fatia
    e, passo a passo, sai serrilhada; uma termobalança também integra no intervalo de amostragem)."""

    def __init__(self, log, size=(820, 760), dpi=100, smooth_h=0.1):
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        self.plt, self.log, self.size, self.dpi = plt, log, size, dpi
        self.t = np.array([r["t_h"] for r in log])
        self.rates = np.array([[r[f"taxa_{g}"] for g, _, _ in GROUP_STYLE] for r in log])
        if smooth_h > 0 and len(self.t) > 2:
            self.rates = np.stack([_smooth_rate(self.t, c, smooth_h) for c in self.rates.T], 1)
        self.conv = np.array([[r[f"conv_{g}"] for g, _, _ in GROUP_STYLE] for r in log])
        self.rmax = float(self.rates.sum(1).max()) * 1.1

    def render(self, t_h):
        plt = self.plt
        W, H = self.size
        fig = plt.figure(figsize=(W / self.dpi, H / self.dpi), dpi=self.dpi)
        fig.patch.set_alpha(0.0)
        a1 = fig.add_axes([0.12, 0.47, 0.85, 0.45]); a2 = fig.add_axes([0.12, 0.08, 0.85, 0.28])
        k = int(np.searchsorted(self.t, t_h, side="right"))
        cols = [c for _, c, _ in GROUP_STYLE]
        labs = [lab for _, _, lab in GROUP_STYLE]
        a1.stackplot(self.t, self.rates.T, colors=cols, alpha=0.12)
        if k > 1:
            a1.stackplot(self.t[:k], self.rates[:k].T, colors=cols, labels=labs, alpha=0.95)
        for j, c in enumerate(cols):
            a2.plot(self.t, self.conv[:, j], color=c, lw=1, alpha=0.15)
            a2.plot(self.t[:k], self.conv[:k, j], color=c, lw=2.4)
        for ax in (a1, a2):
            ax.set_facecolor((0, 0, 0, 0)); ax.axvline(t_h, color="w", lw=1, alpha=0.5)
            ax.set_xlim(0, self.t[-1]); ax.tick_params(colors="w", labelsize=11); ax.grid(alpha=0.15)
            for sp in ax.spines.values():
                sp.set_color((1, 1, 1, 0.35))
        a1.set_ylim(0, self.rmax); a2.set_ylim(0, 1.02)
        a1.set_ylabel("taxa de reação (kg/h)", color="w", fontsize=13)
        a2.set_ylabel("convertido", color="w", fontsize=13)
        a2.set_xlabel("tempo de batelada (h)", color="w", fontsize=13)
        if k > 1:
            a1.legend(loc="upper left", fontsize=10, frameon=False, labelcolor="w", ncol=2)
        buf = io.BytesIO()
        fig.savefig(buf, format="png", transparent=True, dpi=self.dpi)
        plt.close(fig)
        buf.seek(0)
        return Image.open(buf).convert("RGBA")


def render_rates_segment(drum, log, fase: dict, path, seconds=20.0, fps=30, size=(1920, 1080), title="",
                         subtitle=""):
    """Esquerda: os pedaços mudando de fase (DEM); direita: a taxa de pirólise por componente se atualizando."""
    from .scene import draw_phase_bed
    W, H = size
    s = H / 1080
    cs = CrossSection(drum, dict(half_width=0.78, z_bot=-0.78, z_top=0.78, wall=0.04), size=(int(960 * s), int(960 * s)))
    chart = RatesChart(log, size=(int(820 * s), int(740 * s)), dpi=int(100 * s))
    recs, label = fase["recs"], fase["label"]
    tb = np.array([r["t_b"] for r in recs]) / 3600.0
    t_end = float(log[-1]["t_h"])
    lt = np.array([r["t_h"] for r in log])
    n = int(seconds * fps)
    pipe = FFmpegPipe(path, size, fps)
    for k in range(n):
        t_h = t_end * k / max(n - 1, 1)
        rec = recs[int(np.argmin(np.abs(tb - t_h)))]
        img = np.zeros((cs.H, cs.W, 3), np.float32) + np.array([14, 16, 22], np.float32)
        img = cs.drum_layer(img, np.full(36, 130.0), 0.0, wall_px=14 * s)
        img = draw_phase_bed(cs, img, rec, label, mode="material")
        scene = draw_overlay(Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)), cs, rec["theta"])
        canvas = Image.new("RGBA", size, (12, 14, 20, 255))
        canvas.paste(scene, (int(40 * s), int(90 * s)))
        d = ImageDraw.Draw(canvas)
        _title(d, title, int(40 * s), int(18 * s), s, width=1800, size=34)
        if subtitle:
            d.text((int(42 * s), int(62 * s)), br(subtitle), font=_font(int(18 * s)), fill=(190, 195, 205))
        ch = chart.render(t_h)
        canvas.paste(ch, (int(1080 * s), int(190 * s)), ch)
        row = {kk: float(np.interp(t_h, lt, [r[kk] for r in log])) for kk in ("T_carga", "taxa_total", "oleo_kg")}
        conv_tot = 1 - sum(float(np.interp(t_h, lt, [r[f"m_{c}"] for r in log])) for c in
                           ("LDPE", "HDPE", "PP", "PS", "PET", "PVC", "CELL", "RUBa", "RUBb", "RUBc")) / \
            max(sum(log[0][f"m_{c}"] for c in ("LDPE", "HDPE", "PP", "PS", "PET", "PVC", "CELL", "RUBa", "RUBb", "RUBc")), 1e-9)
        big = [(br(f"{t_h:.1f} h"), "de batelada"), (f"{row['T_carga']:.0f} °C", "carga"),
               (f"{row['taxa_total']:.0f} kg/h", "taxa de pirólise"), (f"{100 * conv_tot:.0f} %", "convertido")]
        for i, (v, lab) in enumerate(big):
            xx = int((1100 + i * 200) * s)
            d.text((xx, int(96 * s)), v, font=_font(int(34 * s), True), fill=(255, 210, 120) if i == 2 else (245, 245, 245))
            d.text((xx, int(140 * s)), lab, font=_font(int(16 * s)), fill=(185, 190, 200))
        # legenda: o grupo que mais reage agora
        rates = {g: float(np.interp(t_h, lt, [r[f"taxa_{g}"] for r in log])) for g, _, _ in GROUP_STYLE}
        g_max = max(rates, key=rates.get)
        if rates[g_max] > 5.0:
            msg = GROUP_MSG[g_max]
        elif t_h < 0.5 * t_end:
            msg = "Aquecendo: os pedaços ainda estão inteiros"
        else:
            msg = "Fim: sobram carvão (pedaços escuros e grânulos) e areia"
        p = rec["pool"]
        info = br(f"poça de fundido: {p.get('mass', 0.0):.1f} kg na fatia · viscosidade {p['mu']:.2g} Pa·s") \
            if p.get("mass", 0.0) > 0.05 else ""
        if info:
            d.text((int(60 * s), int(1050 * s)), info, font=_font(int(18 * s)), fill=(210, 190, 160))
        draw_lower_third(canvas, msg, 1.0, s * 0.82, y_bottom=int(1060 * s), max_w=int(760 * s), x0=int(1080 * s))
        pipe.write(np.asarray(canvas.convert("RGB")))
    return pipe.close()


def render_flame_views(drum, flames_corte, flames_lateral, meta, path, Ts_C=500.0, seconds=9.0, fps=30,
                       size=(1920, 1080), title="", captions=(), info=()):
    """Chamas com base física em duas vistas: o corte (ao longo do eixo) e a lateral (os dois queimadores ao
    longo do tambor, vistos da parede dos queimadores, com o casco incandescente na frente das chamas de trás)."""
    W, H = size
    s = H / 1080
    cs = CrossSection(drum, BOX, size=(int(760 * s), int(760 * s)))
    Lx, zb, zt = meta["Lx"], meta["z_bot"], meta["z_top"]
    side_w = int(1080 * s)
    side_h = int(side_w * (zt - zb) / Lx)
    n = int(seconds * fps)
    caps = even_captions(list(captions), seconds) if captions else []
    pipe = FFmpegPipe(path, size, fps)
    for k in range(n):
        img, _ = cs.furnace_layer(T_refr_C=750.0, flame_img=flames_corte.render_on_grid(k, cs.Yw, cs.Zw))
        img = cs.drum_layer(img, np.full(36, Ts_C), 0.0, wall_px=10 * s)
        scene = draw_overlay(Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)), cs, 2 * math.pi * k / fps / 20)
        rgb, a = flames_lateral.render_image(k, (0.0, Lx, zb, zt), (side_w, side_h))
        side = Image.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8))
        ds = ImageDraw.Draw(side)
        for xb in meta["burners_x"]:
            px = int(xb / Lx * side_w); pz = int((zt - meta["burner_z"]) / (zt - zb) * side_h)
            ds.rectangle([px - int(14 * s), pz - int(14 * s), px + int(14 * s), pz + int(14 * s)], outline=(160, 165, 175), width=2)
        canvas = Image.new("RGBA", size, (12, 14, 20, 255))
        canvas.paste(scene, (int(30 * s), int(150 * s)))
        canvas.paste(side, (int(810 * s), int(150 * s) + (int(760 * s) - side_h) // 2))
        d = ImageDraw.Draw(canvas)
        y = _title(d, title, int(40 * s), int(30 * s), s, width=1800, size=36)
        d.text((int(40 * s), int(118 * s)), "corte (ao longo do eixo)", font=_font(int(20 * s)), fill=(190, 195, 205))
        d.text((int(810 * s), int(118 * s)), "vista lateral: os dois queimadores ao longo do tambor",
               font=_font(int(20 * s)), fill=(190, 195, 205))
        y_info = int(150 * s) + (int(760 * s) + side_h) // 2 + int(50 * s)
        for i, ln in enumerate(info):
            d.text((int(810 * s), y_info + i * int(40 * s)), "· " + br(ln), font=_font(int(22 * s)),
                   fill=(225, 225, 230))
        if caps:
            txt, al = _caption_alpha(caps, k / fps)
            draw_lower_third(canvas, txt, al, s * 0.8, y_bottom=int(1060 * s), max_w=int(740 * s), x0=int(30 * s))
        pipe.write(np.asarray(canvas.convert("RGB")))
    return pipe.close()

In [ ]:
%%writefile tamborkit/validation.py
"""Escada de validação do tambor em batelada (cada item: valor do modelo × faixa da literatura).

N1 componentes  — cinética (picos de DTG a 10 K/min; T para k = 10⁻³ s⁻¹), radiação (WSGG), DEM (regime,
                  ângulo dinâmico, arco coberto), balanços de massa e energia, conservação de massa da LES.
N2 processo     — coeficiente parede-carga (Suzuki 2008: 22–55 W/m²K), ganho das aletas (7–26 %),
                  ganho da rotação acima de ~1 rpm pequeno, duração da batelada (IVIG: 6–8 h),
                  combustível auxiliar por tonelada (Beston: 2,1–2,9 GJ/t), eficiência da fornalha (45–70 %),
                  queima autossuficiente depois da partida, rendimentos (Faussone 2022; Fulgencio-Medrano 2022).
N3 sistema      — tambor parado × girando: mancha quente acima do limite do aço, batelada muito mais longa.
"""
from __future__ import annotations

from . import feed as fd

OK, WARN = "dentro", "fora"


def _row(group, name, model, target, lo, hi, unit="", source=""):
    ok = (model >= lo) and (model <= hi)
    return dict(grupo=group, item=name, modelo=model, alvo=target, ok=ok, unidade=unit, fonte=source)


def chemistry_checks() -> list:
    rows = []
    for k, tgt in (("HDPE", 475), ("LDPE", 465), ("PP", 455), ("PS", 420)):
        p = fd.dtg_peak(k, 10)
        rows.append(_row("N1 cinética", f"pico de DTG do {k} a 10 K/min", p, f"{tgt} ± 10 °C", tgt - 10, tgt + 10,
                         "°C", "picos de consenso (Aboulkas 2010; Westerhout 1997)"))
    for k, tgt, tol in (("PVC", 251, 15), ("PET", 395, 15), ("PP", 402, 10), ("LDPE", 417, 10), ("HDPE", 429, 10),
                        ("PS", 370, 10)):
        v = fd.t_k1e3(k)
        rows.append(_row("N1 cinética", f"T com k = 10⁻³ s⁻¹ ({k})", v, f"{tgt} ± {tol} °C", tgt - tol, tgt + tol,
                         "°C", "pesquisa R2 (conjuntos cinéticos da literatura)"))
    y = fd.ultimate_yields(fd.CHARGES["plastico_triado"])
    oil = sum(y[k] for k in fd.CONDENSABLE) * 100
    rows.append(_row("N1 cinética", "óleo + cera, plástico triado (poliolefinas)", oil, "70–90 %", 70, 90, "%",
                     "Fulgencio-Medrano 2022 (filme triado 70,6 %); Beston PEAD > 85 %"))
    return rows


def furnace_checks() -> list:
    from .furnace import wsgg_emissivity
    rows = []
    e = float(wsgg_emissivity(1200.0, 0.2 * 0.35))
    rows.append(_row("N1 radiação", "emissividade do gás de combustão no corredor (WSGG)", e, "0,10–0,18", 0.10, 0.18,
                     "", "Smith, Shen & Friedman 1982; pesquisa R4"))
    return rows


def batch_checks(rot: dict, est: dict | None = None, sweep: dict | None = None, log=None) -> list:
    """rot/est: resumos (BatchDrum.summary()) + extras h_mean, Tmax; sweep: {'sem_aletas':…, 'rpm2':…}."""
    rows = []
    rows.append(_row("N1 balanços", "fecho de massa da batelada", abs(rot["fecho_massa"]) * 100, "< 1 %", 0, 1, "%"))
    rows.append(_row("N1 balanços", "fecho de energia do casco", abs(rot["fecho_casco"]) * 100, "< 1 %", 0, 1, "%"))
    rows.append(_row("N2 processo", "duração da batelada (1 t, parede a 500 °C)", rot["duracao_h"], "6–8 h", 5.5, 8.5,
                     "h", "IVIG/UFRJ (vídeo: 6 a 8 h a partir da carga)"))
    if "h_mean" in rot:
        rows.append(_row("N2 processo", "coeficiente parede → carga médio", rot["h_mean"], "22–55 W/m²K", 22, 55,
                         "W/m²K", "Suzuki et al. 2008 (resíduo em forno rotativo com aletas)"))
    rows.append(_row("N2 processo", "combustível auxiliar por tonelada de carga", rot["aux_GJ_por_t"], "2,1–2,9 GJ/t (± 25 %)",
                     1.6, 3.6, "GJ/t", "Beston (GLP 50–55 kg/t; diesel 55–65 kg/t)"))
    rows.append(_row("N2 processo", "eficiência da fornalha", rot["eficiencia_fornalha"] * 100, "45–70 %", 45, 72, "%",
                     "pesquisa R4 (balanço da fornalha; Hamm 2009)"))
    if log:
        import numpy as np
        t = np.array([r["t_h"] for r in log]); qn = np.array([r["Q_ncg_kW"] for r in log])
        qf = np.array([r["Q_fire_kW"] for r in log])
        on = np.nonzero(qn > 0.5 * qf)[0]
        if len(on):
            sel = slice(on[0], None)
            trap = getattr(np, "trapezoid", None) or np.trapz
            frac = 100 * trap(qn[sel], t[sel]) / max(trap(qf[sel], t[sel]), 1e-9)
            rows.append(_row("N2 processo", "gás próprio cobre a queima depois da troca", frac, "≈ 100 %", 95, 100.1, "%",
                             "Beston; Ma & Zhong 2018: autossuficiente após a partida"))
    oil = rot["oleo"] * 100
    rows.append(_row("N2 rendimentos", "óleo (base úmida, lixo marinho misto)", oil, "45–70 %", 45, 70, "%",
                     "Faussone 2022 (45 %, 400 °C); Fulgencio 2022 (61 %); IVIG ~70 %"))
    rows.append(_row("N2 rendimentos", "carvão + cinzas", (rot["carvao"] + rot["cinzas"]) * 100, "10–26 %", 10, 26, "%",
                     "Faussone 2022 (26 %); Fulgencio 2022 (16,6–24,7 %)"))
    if sweep and "sem_aletas" in sweep:
        gain = 100 * (sweep["sem_aletas"]["duracao_h"] - rot["duracao_h"]) / sweep["sem_aletas"]["duracao_h"]
        rows.append(_row("N2 processo", "redução do tempo com 6 aletas", gain, "7–26 %", 5, 26, "%",
                         "Suzuki et al. 2008 (42 → 31–35 min)"))
    if sweep and "rpm2" in sweep:
        g2 = 100 * (rot["duracao_h"] - sweep["rpm2"]["duracao_h"]) / rot["duracao_h"]
        rows.append(_row("N2 processo", "ganho de 0,8 → 2 rpm", g2, "pequeno (< 15 %)", 0, 15, "%",
                         "pesquisa R3 (ganho modesto acima de ~1–2 rpm)"))
    if est is not None:
        rows.append(_row("N3 sistema", "tambor parado: batelada mais longa", est["duracao_h"] / rot["duracao_h"],
                         "> 1,5×", 1.5, 100, "×", "KINTEK; penetração × condução pura (brief §2)"))
        rows.append(_row("N3 sistema", "tambor parado: mancha quente do casco", est["Tmax"], "> limite do aço (475 °C)",
                         475, 2000, "°C", "Q345R ≤ 475 °C de serviço; caso P4O (brief)"))
    return rows


def dem_checks(st: dict, fill: float, drum) -> list:
    from .geometry import fill_half_angle
    rows = []
    rows.append(_row("N1 DEM", "número de Froude (regime de rolamento)", drum.froude(), "10⁻⁴–10⁻²", 1e-4, 1e-2, "",
                     "Henein et al. 1983; Mellmann 2001"))
    rows.append(_row("N1 DEM", "ângulo dinâmico do leito", st["beta_deg"], "30–50°", 30, 50, "°",
                     "resíduos/pedaços grossos (pesquisa R5)"))
    eps = fill_half_angle(fill)
    cov = st["covered_frac"]
    geo = 2 * eps / (2 * 3.14159265)
    rows.append(_row("N1 DEM", "arco do casco coberto: DEM / geometria", cov / geo, "1 ± 0,2", 0.8, 1.2, "",
                     "segmento circular (base do modelo de penetração)"))
    return rows


def summary(rows) -> str:
    n = len(rows); k = sum(r["ok"] for r in rows)
    lines = [f"{k} de {n} verificações dentro do alvo"]
    for r in rows:
        v = r["modelo"]
        vs = f"{v:.3g}" if abs(v) < 1e4 else f"{v:.2e}"
        lines.append(f"  [{'ok' if r['ok'] else '!!'}] {r['grupo']:<14} {r['item']:<52} {vs:>8} {r['unidade']:<5} alvo {r['alvo']}")
    from .video import br
    return br("\n".join(lines))


def phase_checks(fase: dict, log: list, drum, slab: float) -> list:
    """DEM com mudança de fase × modelo da batelada (massa e enchimento) e a ordem física das reações."""
    import numpy as np
    from .phasedem import melt_viscosity
    rows = []
    recs = fase["recs"]
    lt = np.array([r["t_h"] for r in log]) * 3600.0
    frac = slab / drum.L
    errs, fills = [], []
    for rec in recs[len(recs) // 10::max(len(recs) // 10, 1)]:
        m_bed = float(np.interp(rec["t_b"], lt, [r["carga_kg"] for r in log])) * frac
        m_dem = rec["m_dem"] + rec["pool"].get("mass", 0.0)
        errs.append(abs(m_dem - m_bed) / max(m_bed, 1e-9))
        F_b = float(np.interp(rec["t_b"], lt, [r["enchimento"] for r in log]))
        fills.append(rec["fill_dem"] / max(F_b, 1e-9))
    rows.append(_row("N1 mudança de fase", "massa do DEM (pedaços + poça) × batelada", 100 * float(np.max(errs)),
                     "< 5 %", 0, 5, "%", "acoplamento em uma via (o DEM segue a conversão de cada componente)"))
    rows.append(_row("N1 mudança de fase", "enchimento do DEM / enchimento da batelada (mediana)",
                     float(np.median(fills)), "1 ± 0,35", 0.65, 1.35, "",
                     "colapso solto → denso por material × volume aparente do modelo da batelada"))
    order = ["água", "PVC", "celulósicos", "PS", "PP", "PE"]
    t = np.array([r["t_h"] for r in log])
    peaks = [float(t[int(np.argmax([r[f"taxa_{g}"] for r in log]))]) for g in order]
    ok = all(b >= a for a, b in zip(peaks, peaks[1:]))
    rows.append(_row("N2 cinética", "ordem dos picos de taxa: água < PVC < celulósicos < PS < PP < PE", 1.0 if ok else 0.0,
                     "sim", 0.5, 1.5, "", "ordem dos picos de TGA (Westerhout 1997; Aboulkas 2010; pesquisa R2)"))
    eta_melt = float(melt_viscosity(473.15, 0.0))
    eta_wax = float(melt_viscosity(693.15, 0.3))
    rows.append(_row("N1 mudança de fase", "viscosidade do PE fundido a 200 °C", eta_melt, "10²–10⁴ Pa·s", 1e2, 1e4,
                     "Pa·s", "PE/PP fundidos (faixa de catálogo; reator agitado, Cross)"))
    rows.append(_row("N1 mudança de fase", "viscosidade do fundido craqueado (420 °C, 30 % convertido)", eta_wax,
                     "≤ 0,1 Pa·s (cera)", 1e-4, 0.1, "Pa·s", "ceras de PE (POLYWAX: mPa·s); Westerhout"))
    return rows

In [ ]:
%%writefile video_roteiro.json
{
 "timelapse": {
  "title": "Pirólise de lixo marinho num tambor rotativo",
  "subtitle": "{massa} t por batelada · tambor de {D} m × {L} m a {rpm} rpm com 6 aletas · parede a {Tw} °C · simulação acelerada"
 },
 "title_card": {
  "title": "Por que o tambor de pirólise gira?",
  "subtitle": "Combustão, troca térmica, cinética e o movimento da carga num só modelo, validado com a literatura",
  "credit": "{autor}",
  "disclaimer": "Tambor de referência do tamanho do equipamento da UFRJ/IVIG (classe Beston BLJ-3). Dados de literatura aberta e de catálogo; sem dados de projeto."
 },
 "context_cards": [
  {
   "kicker": "O problema",
   "headline": "Lixo plástico misto conduz muito mal o calor",
   "bullets": [
    "O lixo recolhido na Baía de Guanabara chega úmido, com areia, papel e vários plásticos.",
    "Solto, ele é quase só ar: a parede quente esquenta o que encosta nela, e o resto fica frio.",
    "O tambor gira para renovar esse contato a cada volta, e as aletas levantam a carga."
   ]
  },
  {
   "kicker": "O que foi simulado",
   "headline": "Uma batelada inteira, da partida ao carvão",
   "bullets": [
    "Fornalha: chamas de GLP e do próprio gás da pirólise (LES 3D) e radiação para o casco.",
    "Carga: DEM com mudança de fase (encolhe, gruda, funde numa poça, vira carvão), cinética de 9 componentes.",
    "Vapores: craqueamento no espaço livre, condensador e o gás voltando para o queimador."
   ]
  }
 ],
 "compare": {
  "chapter": "Parado × girando",
  "captions": [
   "Parado: a camada junto à parede quente funde e isola; o miolo da pilha fica frio por horas.",
   "Girando: cada volta renova o contato com a parede e a carga aquece por igual.",
   "Resultado: {t_rot} h girando contra mais de {t_est} h parado, e o casco sem a mancha quente de {Tmax_est} °C."
  ]
 },
 "dem": {
  "title": "Dentro do tambor: as aletas levantam a carga",
  "captions": [
   "Cada pedaço é uma partícula com a massa real do seu material: filme, frasco, isopor, PET, papel, borracha, areia.",
   "O DEM mede o que o modelo térmico precisa: parede coberta, tempo de contato e cortinas."
  ],
  "speed": "DEM a {rpm_dem} rpm (mesmo regime de rolamento do tambor real), mostrado acelerado"
 },
 "flame": {
  "title": "A fornalha: chamas da LES 3D renderizadas com base física",
  "captions": [
   "A luz da chama é a fuligem incandescente (cor de corpo negro) e a base azul vem da liberação de calor.",
   "Os queimadores queimam GLP na partida e, depois, o gás não condensável da própria pirólise."
  ],
  "info": [
   "fração de mistura + equilíbrio químico (Cantera)",
   "radiação: gás (WSGG) e fuligem",
   "{Q_les} kW de queima, chama a {T_chama} °C",
   "{Q_tambor} kW entram no tambor ({ef} %)"
  ]
 },
 "curves": {
  "chapter": "Rotação e aletas",
  "captions": [
   "Girar muda tudo; girar mais rápido muda pouco: de 0,8 para 2 rpm o ganho é de {g_rpm} %.",
   "As aletas encurtam a batelada em {g_aleta} % (Suzuki et al. mediram de 7 a 26 %)."
  ]
 },
 "validation": {
  "chapter": "Validação",
  "captions": [
   "{n_ok} de {n} verificações dentro do alvo.",
   "Cinética por termogravimetria, troca parede-carga de Suzuki, duração do IVIG, combustível auxiliar da Beston e balanços fechados."
  ]
 },
 "kpis": {
  "title": "Uma batelada de {massa} t de lixo marinho",
  "labels": [
   "de batelada",
   "de óleo",
   "de carvão + cinzas",
   "de GLP por tonelada"
  ]
 },
 "closing": {
  "kicker": "Para a gestão",
  "headline": "Uma ferramenta para decidir rotação, aletas e queima antes de mexer no equipamento",
  "bullets": [
   "Mostra quanto tempo cada batelada leva e onde o casco esquenta demais.",
   "Compara cargas (lixo marinho, plástico triado, rejeitos) e o consumo de combustível.",
   "Aberta e auditável: cada hipótese está no código e no relatório."
  ]
 },
 "end_card": {
  "line": "Combustão, troca térmica e cinética de um tambor de pirólise, no navegador",
  "author": "{autor}"
 },
 "caption": "Por que um reator de pirólise de lixo plástico gira? Simulei uma batelada inteira num tambor rotativo do tamanho do equipamento da UFRJ/IVIG, que processa lixo da Baía de Guanabara.\n\nO modelo junta a combustão na fornalha (LES 3D, com o gás da própria pirólise voltando para o queimador), o movimento da carga (DEM, com as aletas levantando o lixo e cada pedaço mudando de fase: amolece, funde numa poça, craqueia e vira carvão), a troca de calor parede-carga e a cinética de nove componentes do lixo: polietileno, polipropileno, isopor, PET, PVC, borracha, papel e madeira, além da areia.\n\nResultado: {massa} t de lixo úmido rendem {oleo} kg de óleo em {t_rot} h, com o gás próprio sustentando a queima depois da partida. Com o tambor parado, a mesma carga levaria mais de {t_est} h e o casco passaria de {Tmax_est} °C sob a chama.\n\nValidação: {n_ok} de {n} verificações contra a literatura (Suzuki, Faussone, Fulgencio-Medrano, dados da Beston e da UFRJ).\n\nCódigo aberto e reprodutível no Google Colab.\n\n#CFD #Pirólise #EconomiaCircular #LixoMarinho #BaíaDeGuanabara #EngenhariaQuímica #DEM #Combustão #Simulação",
 "rates": {
  "title": "A pirólise ao vivo: o que reage em cada momento",
  "subtitle": "Esquerda: os pedaços do DEM mudando de fase · direita: taxa de reação de cada componente (como uma DTG no tempo da batelada)"
 }
}

In [ ]:
#@title Carrega o pacote
import importlib, os, sys, json, pickle, math
os.makedirs("saida", exist_ok=True)
for m in [m for m in sys.modules if m == "tamborkit" or m.startswith("tamborkit.")]:
    del sys.modules[m]
sys.path.insert(0, ".")
import numpy as np, matplotlib.pyplot as plt, torch
from IPython.display import Image, Video, display
from tamborkit import feed as fd, validation as VAL, story as ST
from tamborkit.geometry import Drum
from tamborkit.batch import BatchDrum, BatchConfig, Control
from tamborkit.furnace import Furnace
from tamborkit.video import br

## Parâmetros

In [ ]:
#@title Parâmetros do caso
PRESET = "gpu"            #@param ["teste", "gpu"]
CARGA = "base"            #@param ["base", "plastico_triado", "rejeito_rio"]
MASSA_KG = 1000.0         #@param {type:"number"}
UMIDADE = 0.15            #@param {type:"number"}
RPM = 0.8                 #@param {type:"number"}
ALETAS = 6                #@param {type:"integer"}
T_PAREDE = 500.0          #@param {type:"slider", min:420, max:580, step:10}
AUTOR = "Gabriel Rozo · CAExperts"   #@param {type:"string"}
LINKEDIN = "linkedin.com/in/gabrielhrozo"   #@param {type:"string"}
import os
PRESET = os.environ.get("PRESET", PRESET)
PRESETS = {
    # DEM: rotação simulada, tamanho dos pedaços e fatia periódica; LES: malha e tempos; vídeo: segundos do time-lapse
    # o DEM grava um terço de volta (duas passagens de aleta): o laço do vídeo não tem emenda
    # DEM com mudança de fase: rotação simulada (regime de rolamento), fatia, voltas para a batelada inteira e
    # duração do contato; LES: malha e tempos; vídeo: segundos do time-lapse
    "teste": dict(dem_rpm=3.0, dem_slab=0.5, dem_rev=4.0, dem_tc=0.010, les="teste", les_t=17.0, les_rec=3.0,
                  timelapse_s=30.0),
    "gpu": dict(dem_rpm=2.0, dem_slab=0.5, dem_rev=6.0, dem_tc=0.006, les="gpu", les_t=24.0, les_rec=4.0,
                timelapse_s=40.0),
}
P = PRESETS[PRESET]
drum = Drum(R=0.70, L=4.9, rpm=RPM, n_lifters=ALETAS, lifter_h=0.12, lifter_t=0.012)
charge = fd.Charge(**{**fd.CHARGES[CARGA].__dict__, "mass": MASSA_KG, "moisture": UMIDADE, "fill": 0.65})
control = Control(setpoint_C=T_PAREDE)
print(charge.describe())
print(br(f"tambor Ø {drum.D:.2f} m × {drum.L:.1f} m, {ALETAS} aletas de {drum.lifter_h * 1000:.0f} mm, {RPM} rpm, Fr = {drum.froude():.1e}"))

## 1 · Cinética

In [ ]:
#@title 1 · Cinética e TGA (picos de DTG, rendimentos finais)
rows_kin = VAL.chemistry_checks() + VAL.furnace_checks()
print(VAL.summary(rows_kin))
fig, ax = plt.subplots(figsize=(10, 5))
for k in ["LDPE", "HDPE", "PP", "PS", "PET", "PVC", "CELL", "RUBc"]:
    T, m, d = fd.tga(k, 10)
    ax.plot(T - 273.15, d * 100, label=fd.COMPONENTS[k].name)
T, m, d = fd.tga(charge.masses(), 10)
ax.plot(T - 273.15, d * 100, "k", lw=3, label="carga (mistura)")
ax.set_xlim(100, 650); ax.set_xlabel("T (°C)"); ax.set_ylabel("−dm/dT (%/K) a 10 K/min"); ax.legend(fontsize=8)
fig.savefig("saida/fig_tga.png", dpi=120, bbox_inches="tight"); plt.show()
y = fd.ultimate_yields(charge)
print(br("rendimentos finais (isoterma, sem craqueamento secundário): " +
         ", ".join(f"{k} {100 * v:.1f}%" for k, v in y.items() if v > 0.001)))

## 2 · Batelada

In [ ]:
#@title 2 · Batelada completa: girando × parado, sem aletas e outras rotações
def roda(mode="rotativo", rpm=None, n_lifters=None, snap=120.0, t_max=14.0):
    d = Drum(R=drum.R, L=drum.L, rpm=rpm or drum.rpm, n_lifters=drum.n_lifters if n_lifters is None else n_lifters,
             lifter_h=drum.lifter_h, lifter_t=drum.lifter_t)
    b = BatchDrum(d, charge, cfg=BatchConfig(mode=mode, t_max_h=t_max, snap_every=snap, control=control))
    s = b.run(verbose=False)
    s["h_mean"] = float(np.mean([r["h_wb"] for r in b.log]))
    s["Tmax"] = max(r["T_casco_max"] for r in b.log)
    return b, s

b_rot, s_rot = roda("rotativo")
print(br(f"girando: {s_rot['duracao_h']:.1f} h, óleo {100 * s_rot['oleo']:.0f} %, casco máx {s_rot['Tmax']:.0f} °C"))
b_est, s_est = roda("estatico", t_max=16.0)
print(br(f"parado:  {s_est['duracao_h']:.1f} h, óleo {100 * s_est['oleo']:.0f} %, não convertido {100 * s_est['nao_convertido']:.1f} %, casco máx {s_est['Tmax']:.0f} °C"))
sweep = {}
for nome, kw in (("sem_aletas", dict(n_lifters=0)), ("rpm0.4", dict(rpm=0.4)), ("rpm2", dict(rpm=2.0))):
    _, sweep[nome] = roda(snap=1800.0, **kw)
    print(br(f"{nome:10s}: {sweep[nome]['duracao_h']:.2f} h, h parede-carga {sweep[nome]['h_mean']:.0f} W/m²K"))
ST.curves_figure({f"girando {RPM} rpm": b_rot.log, "parado": b_est.log}, "saida/fig_curvas.png",
                 "Girando × parado: mesma carga, mesma fornalha")
display(Image("saida/fig_curvas.png", width=1100))
for k in ("oleo", "aquoso", "gas", "carvao", "cinzas"):
    print(br(f"  {k:8s} {100 * s_rot[k]:5.1f} % da carga úmida"))
print(br(f"combustível auxiliar: {s_rot['aux_kg']:.0f} kg de GLP ({s_rot['aux_GJ_por_t']:.2f} GJ/t); gás para o flare: {s_rot['Q_flare_GJ']:.2f} GJ"))

## 3 · DEM da carga

In [ ]:
#@title 3 · DEM da carga com mudança de fase: da partida ao carvão
from tamborkit.phasedem import PhaseDEM, PhaseConfig, NAMES
pdem = PhaseDEM(drum, charge, b_rot.log, PhaseConfig(slab=P["dem_slab"], rpm=P["dem_rpm"], n_rev=P["dem_rev"],
                                                    contact_time=P["dem_tc"]))
print(pdem.describe())
recs = pdem.run_batch()
st_pre = pdem.pre_stats.as_dict()     # medidas no trecho frio, com a carga solta
fase = dict(recs=recs, label=pdem.label.cpu().numpy(), names=NAMES)
# trecho frio (antes do relógio da batelada): lixo solto girando com as aletas, para o segmento "dentro do tambor"
pre = pdem.pre_records
dem_solto = dict(frames=pre, r=pre[0]["r"], label=pdem.label.cpu().numpy(), names=NAMES, rpm=P["dem_rpm"],
                 fill=charge.fill, stats=st_pre, lacey=float("nan"))
rows_dem = VAL.dem_checks(st_pre, charge.fill, Drum(R=drum.R, L=drum.L, rpm=P["dem_rpm"]))
rows_fase = VAL.phase_checks(fase, b_rot.log, drum, P["dem_slab"])
print(VAL.summary(rows_dem + rows_fase))

## 4 · LES da fornalha

In [ ]:
#@title 4 · LES 3D da fornalha (chamas de GLP na partida, com o casco a 500 °C)
from tamborkit.furnace_les import FurnaceLES, FurnaceLESConfig
from tamborkit.flames import FlameRecorder, FlameRenderer
Q_LES = 400e3
les = FurnaceLES(FurnaceLESConfig.preset(P["les"], Q_fire=Q_LES, T_shell=T_PAREDE + 273.15, rpm=RPM))
print(br(f"LES: {les.n_cells / 1e3:.0f} mil células de {les.h * 100:.0f} cm; {les.tab['source']}"))
gravador = FlameRecorder(les, t_start=P["les_t"] - P["les_rec"])
les.run(P["les_t"], t_avg=P["les_rec"], frame_dt=1 / 30, on_frame=gravador)
eb = les.energy_balance()
flux = les.drum_flux_map()
chamas = gravador.data()
dev = "cuda" if torch.cuda.is_available() else "cpu"
fl_corte = FlameRenderer(chamas, view="corte", device=dev)
fl_lateral = FlameRenderer(chamas, view="lateral", device=dev)
les_frames = dict(Q=Q_LES, Tmean=les.mean("T"), xc=les.xc, yc=les.yc, zc=les.zc)
ST.flux_figure(flux, les_frames, "saida/fig_fornalha.png")
display(Image("saida/fig_fornalha.png", width=1100))
print(br(f"calor para o tambor: {eb['Q_paredes'] / 1e3:.0f} kW de {Q_LES / 1e3:.0f} kW ({100 * eb['Q_paredes'] / Q_LES:.0f} %); "
         f"T máx {eb['T_max'] - 273:.0f} °C; massa: saída/entrada = {eb['m_out'] / eb['m_in']:.2f}"))

## 5 · Validação

In [ ]:
#@title 5 · Validação
rows_batch = VAL.batch_checks(s_rot, s_est, sweep, log=b_rot.log)
rows_les = [VAL._row("N1 LES", "conservação de massa na fornalha (saída/entrada)", eb["m_out"] / eb["m_in"], "1 ± 0,1",
                     0.9, 1.1, "", "projeção na continuidade discreta")]
resumo = rows_kin + rows_dem + rows_fase + rows_les + rows_batch
print(VAL.summary(resumo))
ST.validation_figure(resumo, "saida/fig_validacao.png")
display(Image("saida/fig_validacao.png", width=1100))
json.dump(dict(rotativo=s_rot, estatico=s_est, sweep=sweep, validacao=[{k: (bool(v) if k == "ok" else v) for k, v in r.items()} for r in resumo]),
          open("saida/resultados.json", "w"), ensure_ascii=False, indent=1, default=float)

In [ ]:
#@title Roteiro do vídeo (textos editáveis)
ROTEIRO = json.load(open("video_roteiro.json", encoding="utf-8"))

## 6 · Vídeo

In [ ]:
#@title 6 · Vídeo em HD: time-lapse com gráfico ao vivo, parado × girando, DEM e chamas
from tamborkit import video as hd
from tamborkit.timelapse import BatchScene
from tamborkit.compare import render_compare
FPS = 30                  #@param {type:"integer"}
RESOLUCAO = "1080p"       #@param ["720p", "1080p"]
size = {"720p": (1280, 720), "1080p": (1920, 1080)}[RESOLUCAO]
nf = lambda x, dd=0: br(f"{x:.{dd}f}")  # noqa: E731
V = dict(massa=nf(MASSA_KG / 1000, 1), D=nf(drum.D, 1), L=nf(drum.L, 1), rpm=nf(RPM, 1), Tw=nf(T_PAREDE), autor=AUTOR,
         t_rot=nf(s_rot["duracao_h"], 1), t_est=nf(s_est["duracao_h"], 0), Tmax_est=nf(s_est["Tmax"]),
         oleo=nf(s_rot["oleo_kg"]), rpm_dem=nf(P["dem_rpm"], 1), Q_les=nf(Q_LES / 1e3),
         T_chama=nf(eb["T_max"] - 273.15), Q_tambor=nf(eb["Q_paredes"] / 1e3), ef=nf(100 * eb["Q_paredes"] / Q_LES),
         g_rpm=nf(100 * (s_rot["duracao_h"] - sweep["rpm2"]["duracao_h"]) / s_rot["duracao_h"]),
         g_aleta=nf(100 * (sweep["sem_aletas"]["duracao_h"] - s_rot["duracao_h"]) / sweep["sem_aletas"]["duracao_h"]),
         n_ok=sum(r["ok"] for r in resumo), n=len(resumo))
F = lambda t: t.format(**V) if isinstance(t, str) else [F(x) for x in t]  # noqa: E731
RT = ROTEIRO
seg, curto = [], []
tl = RT["timelapse"]
scene = BatchScene(b_rot, {"fase": fase}, None, size=size, fps=FPS, seconds=P["timelapse_s"], title=F(tl["title"]),
                   subtitle=F(tl["subtitle"]), flames=fl_corte, flames_Q=Q_LES)
seg.append(scene.render("saida/s00_timelapse.mp4")); curto.append(seg[-1])
tc = RT["title_card"]
fundo = hd.Backdrop("saida/s00_timelapse.mp4", size)
seg.append(hd.render_title_card("saida/s01_titulo.mp4", F(tc["title"]), F(tc["subtitle"]), F(tc["credit"]),
                                F(tc["disclaimer"]), size=size, fps=FPS, seconds=4.5, backdrop=fundo))
cards = [{k: F(v) for k, v in c.items()} for c in RT["context_cards"]]
seg.append(hd.render_text_cards("saida/s02_contexto.mp4", cards, size=size, fps=FPS, seconds=5.5, backdrop=fundo))
rt = RT["rates"]
seg.append(ST.render_rates_segment(drum, b_rot.log, fase, "saida/s025_taxas.mp4", seconds=20.0, fps=FPS, size=size,
                                   title=F(rt["title"]), subtitle=F(rt["subtitle"])))
seg.append(render_compare(b_rot, b_est, "saida/s03_comparacao.mp4", seconds=16.0, fps=FPS, size=size,
                          flames=fl_corte, flames_Q=Q_LES))
curto.append(seg[-1])
cp = RT["compare"]
seg.append(hd.caption_segment("saida/fig_curvas.png", "saida/s04_curvas.mp4", F(cp["captions"]), F(cp["chapter"]),
                              size=size, fps=FPS, seconds=8.0, handle=LINKEDIN))
dm = RT["dem"]
seg.append(ST.render_dem_segment(drum, dem_solto, "saida/s05_dem.mp4", seconds=9.0, fps=FPS, size=size,
                                 title=F(dm["title"]), captions=F(dm["captions"]), speed_label=F(dm["speed"])))
fl = RT["flame"]
seg.append(ST.render_flame_views(drum, fl_corte, fl_lateral, chamas["meta"], "saida/s06_chamas.mp4", Ts_C=T_PAREDE,
                                 seconds=9.0, fps=FPS, size=size, title=F(fl["title"]), captions=F(fl["captions"]),
                                 info=F(fl["info"])))
seg.append(hd.caption_segment("saida/fig_fornalha.png", "saida/s07_fluxo.mp4", F(fl["captions"])[1:], "Fornalha",
                              size=size, fps=FPS, seconds=6.0, handle=LINKEDIN))
cv = RT["curves"]
va = RT["validation"]
seg.append(hd.caption_segment("saida/fig_validacao.png", "saida/s08_validacao.mp4", F(va["captions"]), F(va["chapter"]),
                              size=size, fps=FPS, seconds=7.0, handle=LINKEDIN))
kv = [dict(value=s_rot["duracao_h"], fmt="{:.1f}", unit="h"), dict(value=s_rot["oleo_kg"], fmt="{:.0f}", unit="kg"),
      dict(value=s_rot["carvao_kg"] + s_rot["cinzas_kg"], fmt="{:.0f}", unit="kg"),
      dict(value=s_rot["aux_kg"] / (MASSA_KG / 1000), fmt="{:.0f}", unit="kg")]
for d_, lab in zip(kv, RT["kpis"]["labels"]):
    d_["label"] = lab
seg.append(hd.render_kpi_card("saida/s09_numeros.mp4", F(RT["kpis"]["title"]), kv, size=size, fps=FPS, seconds=6.0, backdrop=fundo))
curto.append(seg[-1])
seg.append(hd.render_text_cards("saida/s10_gestao.mp4", [{k: F(v) for k, v in RT["closing"].items()}], size=size,
                                fps=FPS, seconds=7.0, backdrop=fundo))
ec = RT["end_card"]
seg.append(hd.render_title_card("saida/s11_final.mp4", F(ec["line"]), "", F(ec["author"]), "", size=size, fps=FPS,
                                seconds=4.0, backdrop=fundo, contact=LINKEDIN))
curto.append(seg[-1])
hd.assemble(seg, "saida/tambor_pirolise.mp4", fps=FPS, size=size)
hd.assemble(curto, "saida/tambor_pirolise_curto.mp4", fps=FPS, size=size)
hd.still_of("saida/s00_timelapse.mp4", "saida/capa.png", P["timelapse_s"] * 0.55)
for p in ("saida/tambor_pirolise.mp4", "saida/tambor_pirolise_curto.mp4"):
    print(br(f"{p}: {os.path.getsize(p) / 1e6:.1f} MB"))
display(Video("saida/tambor_pirolise_curto.mp4", embed=True, width=960))
print("\nTexto sugerido para o post:\n")
print(br(F(RT["caption"])))

## Limitações e próximos passos

* **Carga**: em cada fatia o leito girando é bem misturado na seção (o DEM mostra que a mistura leva poucas voltas, minutos contra horas de batelada) e troca calor pelo modelo de penetração; o tambor parado é uma pilha que só conduz (camadas). A fusão é tratada como colapso do volume aparente, sem escoamento do fundido entre fatias.
* **DEM com mudança de fase**: esferas macias (rigidez reduzida) numa fatia periódica, com o relógio comprimido (algumas voltas para a batelada inteira) e acoplamento em uma via (a batelada dita a temperatura e a conversão). A poça é um inventário com superfície plana inclinada, não um fluido resolvido; faixas de fusão, retração e coesão são ordens de grandeza. No preset `teste` a rotação simulada é 3× a real (mesmo regime de rolamento).
* **Chamas**: renderizadas da LES pela emissão da fuligem e pela quimiluminescência na linha de visada; o detalhe abaixo da malha é só visual.
* **Fornalha**: o modelo zonal (Hottel + WSGG) conduz a batelada; a LES 3D mostra as chamas e a distribuição do calor no casco para um estado de queima (o refratário é tratado como rerradiante).
* **Química**: lumps de produtos por componente com rendimentos da literatura; craqueamento secundário simples no espaço livre; sem cloro no óleo (8–30 % do Cl do PVC iria para o óleo, Fulgencio-Medrano 2022).
* **Próximos passos**: medir no equipamento da UFRJ a temperatura da carga e da parede ao longo da batelada; calibrar a composição real do lixo; espaço livre 3D com o refluxo da cera.